In [ ]:
%matplotlib widget
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
singlecol = 3.5
plt.rcParams.update({'font.size': 8})
#plt.rcParams.update({'font.sans-serif': 'Helvetica'})
plt.rcParams.update({'lines.linewidth':1})
plt.rcParams.update({'axes.linewidth':1})
plt.rcParams.update({'lines.markersize':5})
plt.rcParams.update({'savefig.dpi':200})
plt.rcParams.update({'legend.fontsize': 8})

In [ ]:
def convert_dump_file(input_file, output_file, element_map):
    """
    Convert atom types to element names in a LAMMPS dump file.
    
    Parameters:
    input_file (str): Path to the original LAMMPS dump file
    output_file (str): Path to save the modified file
    element_map (dict): Dictionary mapping atom types to element names
    """
    with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
        in_atoms_section = False
        
        for line in infile:
            line = line.strip()
            
            # Check if we're starting the atoms section
            if line.startswith("ITEM: ATOMS"):
                in_atoms_section = True
                outfile.write(line + '\n')
                continue
            
            # Process atom lines when we're in the atoms section
            if in_atoms_section and not line.startswith("ITEM:"):
                parts = line.split()
                
                if len(parts) >= 2:  # Ensure we have at least atom ID and type
                    # Get the atom type (2nd column in your file)
                    atom_type = int(parts[1])
                    
                    # Replace type number with element name if in our map
                    if atom_type in element_map:
                        parts[1] = element_map[atom_type]
                    
                    # Write the modified line
                    outfile.write(' '.join(parts) + '\n')
                else:
                    # Just write the line unchanged if it doesn't match expected format
                    outfile.write(line + '\n')
            else:
                # If we encounter a new "ITEM:" line, we're leaving the atoms section
                if line.startswith("ITEM:") and not line.startswith("ITEM: ATOMS"):
                    in_atoms_section = False
                
                # Write all non-atom lines unchanged
                outfile.write(line + '\n')

In [ ]:
import re

# Function to read and parse LAMMPS log
def parse_lammps_log_diff(file_path):
    data = {'Step': [], 'Temp': [], 'TotEng': [], 'Press': [], 'Volume': [], 'v_nNa': [], 'MSD(PB)' : [], 'MSD(Na)' : []}

    with open(file_path, 'r') as file:
        for line in file:
            # Find thermo output section
            if re.match(r"^\s*Step\s+Temp\s+TotEng", line):
                # Skip header lines
                next(file)
                # Parse the actual data lines
                for data_line in file:
                    if re.match(r"^\s*\d+", data_line) and len(data_line.split())==8:
                        # Extract values (Assuming specific columns order here)
                        values = data_line.split()
                        data['Step'].append(int(values[0]))
                        data['Temp'].append(float(values[1]))
                        data['TotEng'].append(float(values[2]))
                        data['Press'].append(float(values[3]))
                        data['Volume'].append(float(values[4]))
                        data['v_nNa'].append(float(values[5]))
                        data['MSD(PB)'].append(float(values[6]))
                        data['MSD(Na)'].append(float(values[7]))                    
                    else:
                        break
    return data

### parse_lammps_free

In [ ]:
import re

# Function to read and parse LAMMPS log
def parse_lammps_free(file_path):
    data = {'Step': [], 'Temp': [], 'TotEng': [], 'Press': [], 'Volume': [], 'v_nNa': [], 'atoms': [], 'Mn_PE': [], 'Fe_PE': [] }

    with open(file_path, 'r', encoding='utf-8', errors='ignore') as file:
        for line in file:
            # Find thermo output section
            if re.match(r"^\s*Step\s+Temp\s+TotEng", line):
                # Skip header lines
                next(file)
                # Parse the actual data lines
                for data_line in file:
                    if re.match(r"^\s*\d+", data_line) and len(data_line.split())==6:
                        # Extract values (Assuming specific columns order here)
                        values = data_line.split()
                        data['Step'].append(int(values[0]))
                        data['Temp'].append(float(values[1]))
                        data['TotEng'].append(float(values[2]))
                        data['Press'].append(float(values[3]))
                        data['Volume'].append(float(values[4]))
                        data['v_nNa'].append(float(values[5]))
                    elif re.match(r"^\s*\d+", data_line) and len(data_line.split())==7:
                        # Extract values (Assuming specific columns order here)
                        values = data_line.split()
                        data['Step'].append(int(values[0]))
                        data['Temp'].append(float(values[1]))
                        data['TotEng'].append(float(values[2]))
                        data['Press'].append(float(values[3]))
                        data['Volume'].append(float(values[4]))
                        data['v_nNa'].append(float(values[5]))
                        data['atoms'].append(float(values[6]))
                    elif re.match(r"^\s*\d+", data_line) and len(data_line.split())==8:
                        # Extract values (Assuming specific columns order here)
                        values = data_line.split()
                        data['Step'].append(int(values[0]))
                        data['Temp'].append(float(values[1]))
                        data['TotEng'].append(float(values[2]))
                        data['Press'].append(float(values[3]))
                        data['Volume'].append(float(values[4]))
                        data['v_nNa'].append(float(values[5]))
                        data['atoms'].append(float(values[6]))
                        data['Mn_PE'].append(float(values[7]))
                    elif re.match(r"^\s*\d+", data_line) and len(data_line.split())==9:
                        # Extract values (Assuming specific columns order here)
                        values = data_line.split()
                        data['Step'].append(int(values[0]))
                        data['Temp'].append(float(values[1]))
                        data['TotEng'].append(float(values[2]))
                        data['Press'].append(float(values[3]))
                        data['Volume'].append(float(values[4]))
                        data['v_nNa'].append(float(values[5]))
                        data['atoms'].append(float(values[6]))
                        data['Mn_PE'].append(float(values[7]))
                        data['Fe_PE'].append(float(values[8]))
                    else:
                        break
                        
    return data

In [ ]:
import re

# Function to read and parse LAMMPS log
def parse_lammps_log(file_path):
    data = {'Step': [], 'Temp': [], 'TotEng': [], 'Press': [], 'Volume': [], 'v_nNa': []}

    with open(file_path, 'r') as file:
        for line in file:
            # Find thermo output section
            if re.match(r"^\s*Step\s+Temp\s+TotEng", line):
                # Skip header lines
                next(file)
                # Parse the actual data lines
                for data_line in file:
                    if re.match(r"^\s*\d+", data_line) and len(data_line.split())==6:
                        # Extract values (Assuming specific columns order here)
                        values = data_line.split()
                        data['Step'].append(int(values[0]))
                        data['Temp'].append(float(values[1]))
                        data['TotEng'].append(float(values[2]))
                        data['Press'].append(float(values[3]))
                        data['Volume'].append(float(values[4]))
                        data['v_nNa'].append(float(values[5]))
                        data['atoms'].append(float(values[6]))
                    else:
                        break
    return data

In [ ]:
import re

# Function to read and parse LAMMPS log
def parse_lammps_bias(file_path):
    data = {'Step': [], 'Temp': [], 'TotEng': [], 'Press': [], 'Volume': [], 'v_nNa': [], 'Atoms': [], 'v_Utotal': []} 

    with open(file_path, 'r') as file:
        for line in file:
            # Find thermo output section
            if re.match(r"^\s*Step\s+Temp\s+TotEng", line):
                # Skip header lines
                next(file)
                # Parse the actual data lines
                for data_line in file:
                    if re.match(r"^\s*\d+", data_line) and len(data_line.split())==8:
                        # Extract values (Assuming specific columns order here)
                        values = data_line.split()
                        data['Step'].append(int(values[0]))
                        data['Temp'].append(float(values[1]))
                        data['TotEng'].append(float(values[2]))
                        data['Press'].append(float(values[3]))
                        data['Volume'].append(float(values[4]))
                        data['Volume'].append(float(values[4]))
                        data['v_nNa'].append(float(values[5]))
                    else:
                        break
    return data

In [ ]:
def plot_log(log_file_path):
    log_data = parse_lammps_free(log_file_path)
    
    #E_Na = -.13023100E+01
    E_Na = -.14023100E+01
    
    my_list = log_data['TotEng'][20:40]
    E_i = sum(my_list) / len(my_list)

    my_list = log_data['TotEng'][-20:]
    E_f = sum(my_list) / len(my_list)

    my_list = np.array(log_data['atoms'][20:40])-28672
    n_Na_i = sum(my_list) / len(my_list)

    my_list = np.array(log_data['atoms'][-20:])-28672
    n_Na_f = sum(my_list) / len(my_list)
    
    n_Na = n_Na_i-n_Na_f
    
    V_ave = -(E_i-E_f-n_Na*E_Na)/n_Na
    #-(log_data['TotEng'][20]-log_data['TotEng'][-1]-(log_data['v_nNa'][20]-log_data['v_nNa'][-1])*E_Na)/(log_data['v_nNa'][20]-log_data['v_nNa'][-1])
    # -(Ei-Ef-n_Na*E_Na)/n_Na
    fig, ax1 = plt.subplots(figsize=(10, 6))
    
    # Plot RMS(E/atom) on the first y-axis
    color = 'tab:blue'
    ax1.set_xlabel('Timestep (0.4 fs/step)')
    ax1.set_ylabel('Total E (eV)', color=color)
    ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
    ax1.tick_params(axis='y', labelcolor=color)
    
    # Create the second y-axis and plot RMS(f)
    ax2 = ax1.twinx()
    color = 'tab:orange'
    ax2.set_ylabel('Volume (A^3)', color=color)
    ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
    ax2.tick_params(axis='y', labelcolor=color)
    
    ax3 = ax1.twinx()
    color = 'tab:green'
    ax3.spines['right'].set_position(('outward', 80)) 
    ax3.set_ylabel('Number of Na+', color=color)
    ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
    ax3.tick_params(axis='y', labelcolor=color)
    plt.grid()
    plt.title(log_file_path + f'    Average V = {round(V_ave,3)}')
    plt.tight_layout()
    plt.savefig(log_file_path+'.jpg', bbox_inches='tight')

In [ ]:
def plot_log_no_V(log_file_path):
    log_data = parse_lammps_free(log_file_path)
    
    fig, ax1 = plt.subplots(figsize=(singlecol*2, singlecol*1), dpi=200)
    
    # Plot RMS(E/atom) on the first y-axis
    color = 'tab:blue'
    ax1.set_xlabel('Timestep (0.4 fs/step)')
    ax1.set_ylabel('Total E (eV)', color=color)
    ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
    ax1.tick_params(axis='y', labelcolor=color)
    
    # Create the second y-axis and plot RMS(f)
    ax2 = ax1.twinx()
    color = 'tab:orange'
    ax2.set_ylabel('Volume (A^3)', color=color)
    ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
    ax2.tick_params(axis='y', labelcolor=color)
    
    ax3 = ax1.twinx()
    color = 'tab:green'
    ax3.spines['right'].set_position(('outward', 80)) 
    ax3.set_ylabel('Number of Na+', color=color)
    ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
    ax3.tick_params(axis='y', labelcolor=color)
    plt.grid()
    plt.title(log_file_path)
    plt.tight_layout()
    plt.savefig(log_file_path+'.jpg', bbox_inches='tight')

### plot_log_free

In [ ]:
def plot_log_free(log_file_path):
    Nt_ls = range(0,505,5)
    log_data = parse_lammps_free(log_file_path)
    
    fig, ax1 = plt.subplots(figsize=(10, 6))
    
    # Plot RMS(E/atom) on the first y-axis
    color = 'tab:blue'
    ax1.set_xlabel('Timestep (0.4 fs/step)')
    ax1.set_ylabel('Total E (eV)', color=color)
    ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
    ax1.tick_params(axis='y', labelcolor=color)
    
    # Create the second y-axis and plot RMS(f)
    ax2 = ax1.twinx()
    color = 'tab:orange'
    ax2.set_ylabel('Volume (A^3)', color=color)
    ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
    ax2.tick_params(axis='y', labelcolor=color)
    
    ax3 = ax1.twinx()
    color = 'tab:green'
    ax3.spines['right'].set_position(('outward', 80)) 
    ax3.set_ylabel('Number of Na+', color=color)
    atoms = np.array(log_data['atoms'])-28672
    ax3.plot(log_data['Step'], atoms, color=color, marker='*')
    ax3.tick_params(axis='y', labelcolor=color)

    nNa = log_data['v_nNa'][-1]
    plt.grid()
    plt.title(log_file_path+f'  final nNa = {nNa}')
    plt.savefig(log_file_path+'.jpg', bbox_inches='tight')
    plt.tight_layout()

In [ ]:
def plot_log_diff(log_file_path):
    log_data = parse_lammps_log_diff(log_file_path)
    #num_Na = int(log_data_2['v_nNa'][0])
    
    fig, ax1 = plt.subplots(figsize=(10, 6))
    
    # Plot RMS(E/atom) on the first y-axis
    color = 'tab:blue'
    ax1.set_xlabel('Timestep (1 fs/step)')
    ax1.set_ylabel('Total E (eV)', color=color)
    ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
    #ax1.plot(log_data_2['Step'], log_data_2['TotEng'], color=color, marker='o', markersize=3)
    ax1.tick_params(axis='y', labelcolor=color)
    
    # Create the second y-axis and plot RMS(f)
    ax2 = ax1.twinx()
    color = 'tab:orange'
    ax2.set_ylabel('Volume (A^3)', color=color)
    ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
    #ax2.plot(log_data_2['Step'][10:], log_data_2['Volume'][10:], color=color, marker='s', markersize=3)
    ax2.tick_params(axis='y', labelcolor=color)
    
    # MSD(PB) (Third y-axis, second right)
    ax3 = ax1.twinx()
    color3 = 'tab:green'
    ax3.spines['right'].set_position(('outward', 80))
    ax3.set_ylabel('MSD A^2', color=color3)
    ax3.plot(log_data['Step'], log_data['MSD(PB)'], color=color3, marker='*', markersize=3)
    ax3.tick_params(axis='y')
    
    # MSD(Na) (Fourth y-axis, third right)
    #data["c_msd_Na[1]"] = [value * num_Na for value in data["c_msd_Na[1]"]]
    color4 = 'tab:red'
    ax3.plot(log_data['Step'], log_data['MSD(Na)'], color=color4, marker='*', markersize=3)

    plt.title(log_file_path)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(log_file_path+'.jpg', dpi=300, bbox_inches='tight')

In [ ]:
def fit_chemical_potential(x, y, degree):
    import numpy as np
    import matplotlib.pyplot as plt
    from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
    
    # Your data fitting
    x = x[sort_indices]
    y = y[sort_indices]
    
    coefficients = np.polyfit(x, y, degree)
    poly_func = np.poly1d(coefficients)
    
    # Generate fitted values
    x_fit = np.linspace(0, int(x.max()), 100)
    y_fit = poly_func(x_fit)
    
    # Calculate fitted values for original data points to compute metrics
    y_pred = poly_func(x)
    
    # Calculate fitting quality metrics
    r2 = r2_score(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    mae = mean_absolute_error(y, y_pred)
    residual_std = np.std(y - y_pred)
    
    # Generate the equation string (fixed version)
    equation = "y = "
    for i, c in enumerate(coefficients):
        power = degree - i
        if i > 0:  # Add + or - for subsequent terms
            if c >= 0:
                equation += " + "
            else:
                equation += " - "
                c = abs(c)  # Use absolute value after adding minus sign
        
        if power == 0:
            equation += f"{c}"
        elif power == 1:
            equation += f"{c}x"
        else:
            equation += f"{c}x^{power}"
    
    # Create fitting quality text
    quality_text = f"R² = {r2:.4f}\nRMSE = {rmse:.4f}\nMAE = {mae:.4f}\nResidual σ = {residual_std:.4f}"
    
    plt.figure(figsize=(10, 10))
    
    # Main plot
    plt.subplot(2, 2, (1, 2))
    plt.scatter(x, y, label='Data', alpha=0.7)
    plt.plot(x, y, 'b-', alpha=0.5, label='Data trend')
    plt.plot(x_fit, y_fit, 'r-', label=f'Polynomial Fit (degree {degree})', linewidth=2)
    #plt.text(0.05, 0.95, equation, transform=plt.gca().transAxes, fontsize=9, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    #plt.text(0.05, 0.75, quality_text, transform=plt.gca().transAxes,
    #         fontsize=9, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
    plt.legend()
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title(f'Polynomial Fit (degree {degree}) with Quality Metrics')
    plt.grid(True, alpha=0.3)
    
    # Residuals plot
    plt.subplot(2, 2, 3)
    residuals = y - y_pred
    plt.scatter(x, residuals, alpha=0.7)
    plt.axhline(y=0, color='r', linestyle='--', alpha=0.5)
    plt.xlabel('x')
    plt.ylabel('Residuals')
    plt.title('Residuals vs Fitted')
    plt.grid(True, alpha=0.3)
    
    # Residuals histogram
    plt.subplot(2, 2, 4)
    plt.hist(residuals, bins=15, alpha=0.7, edgecolor='black')
    plt.xlabel('Residuals')
    plt.ylabel('Frequency')
    plt.title('Residuals Distribution')
    plt.axvline(x=0, color='r', linestyle='--', alpha=0.5)
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print detailed fitting statistics
    print(f"Polynomial Fitting Results (degree {degree}):")
    print(f"{'='*50}")
    print(f"Equation: {equation}")
    print(f"R-squared (R²): {r2:.6f}")
    print(f"Root Mean Square Error (RMSE): {rmse:.6f}")
    print(f"Mean Absolute Error (MAE): {mae:.6f}")
    print(f"Standard deviation of residuals: {residual_std:.6f}")
    print(f"Number of data points: {len(x)}")
    print(f"Degrees of freedom: {len(x) - degree - 1}")
    
    # Coefficients
    print(f"\nCoefficients:")
    for i, c in enumerate(coefficients):
        power = degree - i
        if power == 0:
            print(f"  Constant term: {c:.8f}")
        else:
            print(f"  x^{power} coefficient: {c:.8f}")
    return coefficients

In [ ]:
def parse_print_file(file_path):
    data = {'Step': [], 'atoms': []}
    try:
        with open(file_path, 'r') as file:
            # Skip lines until we find the header
            for line in file:
                if re.match(r"^\s*Step\s+Atoms", line):
                    # Try to skip the next line if it's a separator
                    next_line = next(file, '').strip()
                    if not re.match(r"^\s*\d+", next_line) and len(next_line.split()) != 2:
                        continue
                    else:
                        # If it's a data line, process it
                        values = next_line.split()
                        data['Step'].append(int(values[0]))
                        data['atoms'].append(float(values[1]))
                    
                    # Process remaining data lines
                    for data_line in file:
                        data_line = data_line.strip()
                        if re.match(r"^\s*\d+", data_line) and len(data_line.split()) == 2:
                            values = data_line.split()
                            data['Step'].append(int(values[0]))
                            data['atoms'].append(float(values[1]))
                        else:
                            break
    except Exception as e:
        print(f"Error parsing file {file_path}: {e}")
    
    # Convert lists to numpy arrays for easier manipulation
    data['Step'] = np.array(data['Step'])
    data['atoms'] = np.array(data['atoms'])
    
    return data

In [ ]:
def plot_loop(temp, Nt0, mu_b, fi, serie, curr_N):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,5))
    plt.subplots_adjust(wspace=0.35)
    temp = temp
    fi = fi
    Nf_ls = []
    mu_ls = []
    
    for i in range(1, curr_N):
        Nt = Nt0 + (i-1) * 50
        
        # Load atoms log file
        file = f'./production_run/free_E/test/Nt_{temp}/{serie}/atoms_log_{i}.txt'
        try:
            log_data = parse_print_file(file)
            step = np.array(log_data['Step'])
            n_Na = np.array(log_data['atoms']) - 28672  # Adjust baseline as needed
            ax1.plot(step, n_Na, label=f'Nf_{i}={n_Na[-1]}, Nt={Nt}')
        except Exception as e:
            print(f"Warning: Could not load atoms log file {file}: {e}")
            continue
        
        # Load convergence file with error handling
        file = f'./production_run/free_E/test/Nt_{temp}/{serie}/{temp}_converge_{i}.txt'
        try:
            data = np.loadtxt(file, skiprows=2)
            
            # Check if data is empty
            if data.size == 0:
                print(f"Warning: Empty data in file {file}, skipping iteration {i}")
                continue
            
            # Handle different data shapes
            if data.ndim == 1:
                # 1D array - single row of data
                if len(data) >= 2:  # Make sure we have at least 2 elements
                    ts = data[0]
                    Nf = data[-1] - 28672
                    Nf = data[:].mean - 28672
                else:
                    print(f"Warning: Insufficient data in file {file}, skipping iteration {i}")
                    continue
            else:
                # 2D array - multiple rows
                if data.shape[0] > 0 and data.shape[1] >= 2:  # Check dimensions
                    ts = data[-1, 0] 
                    Nf = data[-1, -1] - 28672
                    Nf = data[:].mean - 28672
                else:
                    print(f"Warning: Invalid data shape in file {file}, skipping iteration {i}")
                    continue
            
            # Calculate chemical potential
            mu = mu_b - 2 * fi * (Nf - Nt)
            
            # Store results
            Nf_ls.append(Nf)
            mu_ls.append(mu)
            
            # Add scatter points and text
            ax1.scatter(ts, Nf)
            ax1.text(ts, Nf-5, s=f'{Nf}', fontsize=8)
            ax2.scatter(Nf, mu)
            
        except FileNotFoundError:
            print(f"Warning: File {file} not found, skipping iteration {i}")
            continue
        except Exception as e:
            print(f"Warning: Error loading file {file}: {e}, skipping iteration {i}")
            continue
    
    # Only plot if we have data
    if len(Nf_ls) > 0:
        ax2.plot(Nf_ls, mu_ls)
    else:
        print("Warning: No valid data found for plotting")
        
    # Set plot properties
    plt.suptitle(f'{temp} K mu_bias = {mu_b}')
    ax1.grid()
    ax1.set_xlabel('Timestep (0.4 fs/step)')
    ax1.set_ylabel('Number of Na+')
    ax2.set_xlabel('Number of Na+')
    ax2.set_ylabel('Chemical potential (eV)')
    plt.tight_layout()
    
    # Save figure
    plt.savefig(f'./production_run/free_E/test/loop_{temp}.png')
    
    return Nf_ls, mu_ls

In [ ]:
import argparse
import sys
import re

def parse_args():
    p = argparse.ArgumentParser(
        description="Convert a single‐frame LAMMPS data file (atom_style atomic) "
                    "into one MLIP‐MTP .cfg file."
    )
    p.add_argument(
        "--in", "-i", dest="infile", required=True,
        help="Path to input LAMMPS data file (atom_style atomic)."
    )
    p.add_argument(
        "--out", "-o", dest="outfile", required=True,
        help="Path to output MLIP .cfg file."
    )
    p.add_argument(
        "--map", "-m", dest="type_map", required=True,
        help="Comma‐separated mapping of LAMMPS type→element, e.g. 1=Fe,2=Mn,3=C"
    )
    return p.parse_args()

def read_type_map(s):
    """
    Parse a string like "1=Fe,2=Mn,3=C,4=N,5=Na" into a dict {1:"Fe",2:"Mn",...}.
    """
    mp = {}
    for pair in s.strip().split(","):
        if '=' not in pair:
            continue
        t, sym = pair.split("=")
        try:
            t_int = int(t)
        except ValueError:
            raise ValueError(f"Cannot parse type ID '{t}' as integer in mapping '{s}'")
        mp[t_int] = sym
    return mp

def parse_lammps_data(filename):
    """
    Very minimal LAMMPS-data parser (atom_style atomic).
    Returns:
      • natoms        : int
      • box_bounds    : ((xlo,xhi),(ylo,yhi),(zlo,zhi))
      • atoms         : list of (id, type, x, y, z) tuples (floats for coords)
    """
    natoms = None
    box_bounds = {"x":None, "y":None, "z":None}
    atoms_section = False
    atoms = []
    # We'll record how many atoms we saw, for sanity
    num_atoms_parsed = 0

    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            # 1) Look for “<number> atoms”
            m = re.match(r"^(\d+)\s+atoms\s*$", line)
            if m:
                natoms = int(m.group(1))
                continue

            # 2) Box bounds: “xlo xhi”
            m = re.match(r"^([+-]?\d*\.?\d+(?:[eE][+-]?\d+)?)\s+([+-]?\d*\.?\d+(?:[eE][+-]?\d+)?)\s+xlo\s+xhi\s*$", line)
            if m:
                box_bounds["x"] = (float(m.group(1)), float(m.group(2)))
                continue
            m = re.match(r"^([+-]?\d*\.?\d+(?:[eE][+-]?\d+)?)\s+([+-]?\d*\.?\d+(?:[eE][+-]?\d+)?)\s+ylo\s+yhi\s*$", line)
            if m:
                box_bounds["y"] = (float(m.group(1)), float(m.group(2)))
                continue
            m = re.match(r"^([+-]?\d*\.?\d+(?:[eE][+-]?\d+)?)\s+([+-]?\d*\.?\d+(?:[eE][+-]?\d+)?)\s+zlo\s+zhi\s*$", line)
            if m:
                box_bounds["z"] = (float(m.group(1)), float(m.group(2)))
                continue

            # 3) Detect start of “Atoms” section
            if line.startswith("Atoms"):
                # Next non-blank line is the header of columns (we ignore it),
                # then successive lines are “id  type  x  y  z ...”
                atoms_section = True
                # Skip the next line (usually the column header)
                next(f)
                continue

            # 4) If we are in Atoms section, parse until we have read natoms lines
            if atoms_section:
                parts = line.split()
                if len(parts) < 5:
                    # If <5 columns, probably blank or end
                    atoms_section = False
                    continue
                # LAMMPS “atomic” style: id, type, x, y, z
                try:
                    at_id   = int(parts[0])
                    at_type = int(parts[1])
                    x = float(parts[2])
                    y = float(parts[3])
                    z = float(parts[4])
                except ValueError:
                    # Malformed line → stop parsing Atoms
                    atoms_section = False
                    continue
                atoms.append((at_id, at_type, x, y, z))
                num_atoms_parsed += 1
                if natoms is not None and num_atoms_parsed >= natoms:
                    # We have read all atoms → exit Atoms section
                    atoms_section = False

    if natoms is None:
        raise RuntimeError(f"Could not find number of atoms in {filename}")
    if any(box_bounds[dim] is None for dim in ("x","y","z")):
        raise RuntimeError(f"Failed to parse box bounds from {filename}")
    if len(atoms) != natoms:
        raise RuntimeError(f"Expected {natoms} atoms, but parsed {len(atoms)} in {filename}")
    return natoms, (box_bounds["x"], box_bounds["y"], box_bounds["z"]), atoms

def write_mlip_cfg(outfile, natoms, box, atoms):
    (xlo, xhi), (ylo, yhi), (zlo, zhi) = box
    # Build lattice vectors (orthorhombic):
    ax = (xhi - xlo, 0.0,       0.0)
    by = (0.0,       yhi - ylo, 0.0)
    cz = (0.0,       0.0,       zhi - zlo)

    with open(outfile, 'w') as out:
        out.write("BEGIN_CFG\n")
        out.write(" Size\n")
        out.write(f"    {natoms}\n")
        out.write(" Supercell\n")
        out.write(f"    {ax[0]:15.8f} {ax[1]:15.8f} {ax[2]:15.8f}\n")
        out.write(f"    {by[0]:15.8f} {by[1]:15.8f} {by[2]:15.8f}\n")
        out.write(f"    {cz[0]:15.8f} {cz[1]:15.8f} {cz[2]:15.8f}\n")
        out.write(" AtomData:  id  type  cartes_x  cartes_y  cartes_z\n")
        # Sort atoms by ID (optional, but often nicer)
        atoms_sorted = sorted(atoms, key=lambda x: x[0])
        for (at_id, at_type, x, y, z) in atoms_sorted:
            # Map numeric type → element symbol:
            out.write(f"    {at_id:4d}  {at_type-1:4d}   {x:10.6f}   {y:10.6f}   {z:10.6f}\n")
        out.write("END_CFG\n")

## Start from here

## Magnetization

In [ ]:
"""
VASP OUTCAR Parser
==================
Extracts from the LAST ionic step:
  - Per-atom charge       (s, p, d, tot)
  - Per-atom magnetization (x) (s, p, d, tot)
  - Atomic positions      (Angstrom)
  - Atom types            (element symbol for every ion)

Usage
-----
    python parse_outcar.py                  # reads OUTCAR in current dir
    python parse_outcar.py path/to/OUTCAR   # explicit path
    python parse_outcar.py OUTCAR --json    # also write results.json
"""

import re
import sys
import json
from pathlib import Path


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def read_outcar(path: str) -> list[str]:
    with open(path, "r", errors="replace") as fh:
        return fh.readlines()


def find_last(lines: list[str], pattern: str) -> int:
    """Return the line-index of the LAST occurrence of a regex pattern."""
    idx = -1
    rx = re.compile(pattern)
    for i, line in enumerate(lines):
        if rx.search(line):
            idx = i
    if idx == -1:
        raise ValueError(f"Pattern not found in OUTCAR: {pattern!r}")
    return idx


def parse_spd_block(lines: list[str], header_idx: int, n_ions: int) -> list[dict]:
    """
    Parse a block that looks like:

        # of ion       s       p       d       tot
        ------------------------------------------
            1        0.535   1.000   0.000   1.535
            ...

    Returns a list of dicts, one per ion (0-indexed internally, but stored
    as 1-indexed 'ion' key for readability).
    """
    results = []
    # skip "# of ion …" header line and the dashes line
    data_start = header_idx + 2
    for i in range(n_ions):
        parts = lines[data_start + i].split()
        results.append({
            "ion": int(parts[0]),
            "s":   float(parts[1]),
            "p":   float(parts[2]),
            "d":   float(parts[3]),
            "tot": float(parts[4]),
        })
    return results


# ---------------------------------------------------------------------------
# Atom-type map
# ---------------------------------------------------------------------------

def get_atom_types(outcar_path: str) -> list[int]:
    """
    Build a list of integer atom types (1, 2, 3...), one per ion.
    Reads the ion counts from the adjacent POSCAR file.
    """
    poscar_path = Path(outcar_path).parent / "POSCAR"
    if not poscar_path.exists():
        raise FileNotFoundError(
            f"Could not find POSCAR at '{poscar_path}'. "
            "POSCAR is required to determine the number of ions per type."
        )

    with open(poscar_path, "r", errors="replace") as fh:
        lines = fh.readlines()

    # In a standard POSCAR file:
    # - VASP 4: Line 6 (index 5) contains the atom counts
    # - VASP 5+: Line 6 (index 5) has element names, and Line 7 (index 6) has the counts
    # We check lines 5 through 7 for the first line that consists purely of numbers.
    counts = None
    for line in lines[5:8]:
        parts = line.split()
        if not parts:
            continue
        # If every part of the line is a digit, we've found our counts
        if all(p.isdigit() for p in parts):
            counts = list(map(int, parts))
            break

    if not counts:
        raise ValueError(f"Could not parse ion counts from POSCAR ({poscar_path}).")

    atom_types = []
    # Enumerate starting at 1, so the first group is Type 1, second is Type 2, etc.
    for type_idx, count in enumerate(counts, start=1):
        atom_types.extend([type_idx] * count)
        
    return atom_types

# ---------------------------------------------------------------------------
# Positions
# ---------------------------------------------------------------------------

def get_positions(lines: list[str]) -> list[dict]:
    """
    Parse the LAST 'POSITION  TOTAL-FORCE' block.
    Returns list of {'x', 'y', 'z'} dicts.
    """
    header_idx = find_last(lines, r"POSITION\s+TOTAL-FORCE")
    # skip header + dashes line
    data_start = header_idx + 2
    positions = []
    for line in lines[data_start:]:
        parts = line.split()
        if len(parts) < 3:
            break
        try:
            x, y, z = float(parts[0]), float(parts[1]), float(parts[2])
        except ValueError:
            break
        positions.append({"x": x, "y": y, "z": z})
    return positions


# ---------------------------------------------------------------------------
# Charge & magnetization
# ---------------------------------------------------------------------------

def get_charge(lines: list[str], n_ions: int) -> list[dict]:
    """Parse the LAST 'total charge' block."""
    header_idx = find_last(lines, r"^\s+total charge\s*$")
    # structure:  "total charge"  →  blank  →  "# of ion …"  →  dashes  →  data
    # find the "# of ion" line after header_idx
    for i in range(header_idx + 1, header_idx + 6):
        if re.search(r"#\s+of\s+ion", lines[i]):
            return parse_spd_block(lines, i, n_ions)
    raise ValueError("Could not find '# of ion' line after 'total charge' header.")


def get_magnetization(lines: list[str], n_ions: int) -> list[dict]:
    """Parse the LAST 'magnetization (x)' block."""
    header_idx = find_last(lines, r"magnetization \(x\)")
    for i in range(header_idx + 1, header_idx + 6):
        if re.search(r"#\s+of\s+ion", lines[i]):
            return parse_spd_block(lines, i, n_ions)
    raise ValueError("Could not find '# of ion' line after 'magnetization (x)' header.")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def parse_outcar(path: str) -> dict:
    lines = read_outcar(path)
    atom_types = get_atom_types(path)  # <--- Changed: pass 'path' instead of 'lines'
    n_ions = len(atom_types)

    positions     = get_positions(lines)
    charge        = get_charge(lines, n_ions)
    magnetization = get_magnetization(lines, n_ions)

    if not (len(positions) == len(charge) == len(magnetization) == n_ions):
        raise ValueError(
            f"Length mismatch: positions={len(positions)}, "
            f"charge={len(charge)}, mag={len(magnetization)}, "
            f"n_ions={n_ions}"
        )

    atoms = []
    for i in range(n_ions):
        atoms.append({
            "ion":           i + 1,
            "type":          atom_types[i], 
            "position":      positions[i],
            "charge":        charge[i],
            "magnetization": magnetization[i],
        })

    return {"n_ions": n_ions, "atoms": atoms}


def print_results(data: dict) -> None:
    n = data["n_ions"]
    print(f"\n{'='*70}")
    print(f"  OUTCAR summary  –  {n} ions")
    print(f"{'='*70}")
    print(
        f"{'Ion':>4} {'Type':>4}  "  # Changed header
        f"{'x (Å)':>10} {'y (Å)':>10} {'z (Å)':>10}  "
        f"{'Q_tot':>7}  {'Mag_tot':>8}"
    )
    print("-" * 70)
    for a in data["atoms"]:
        p = a["position"]
        print(
            f"{a['ion']:>4} {a['type']:>4}  "  # Uses a['type']
            f"{p['x']:>10.5f} {p['y']:>10.5f} {p['z']:>10.5f}  "
            f"{a['charge']['tot']:>7.3f}  "
            f"{a['magnetization']['tot']:>8.4f}"
        )
    print(f"{'='*70}\n")


# ---------------------------------------------------------------------------

if __name__ == "__main__":
    args = sys.argv[1:]
    outcar_path = "./validation/VASP/0/OUTCAR"
    write_json  = False

    for arg in args:
        if arg == "--json":
            write_json = True
        else:
            outcar_path = arg

    if not Path(outcar_path).exists():
        print(f"ERROR: file not found: {outcar_path}")
        sys.exit(1)

    print(f"Parsing {outcar_path} …")
    data = parse_outcar(outcar_path)
    print_results(data)

    if write_json:
        out_file = "results.json"
        with open(out_file, "w") as fh:
            json.dump(data, fh, indent=2)
        print(f"Results written to {out_file}")

In [ ]:
def extract_outcar_data(filename):
    mag_data = []
    with open(filename, 'r') as f:
        lines = f.readlines()
        
    # 1. Find the start of the LAST magnetization block 
    # (If you have multiple ionic steps, you usually want the final converged step)
    mag_start = -1
    for i, line in enumerate(lines):
        if "magnetization (x)" in line:
            mag_start = i
            
    if mag_start == -1:
        print("Warning: No magnetization data found in OUTCAR.")
        return mag_data
        
    # 2. Parse the block
    # VASP table format:
    # magnetization (x)
    # # of ion       s       p       d       tot
    # --------------------------------------------------
    #    1        0.001   0.008   0.000   0.009
    #   ...
    
    # Start iterating 3 lines after the header to skip the text and dashes
    for line in lines[mag_start + 4:]:
        # A line of dashes marks the end of the atomic values
        if "--------" in line: 
            break 
            
        parts = line.split()
        if len(parts) >= 2:
            try:
                # Using parts[-1] gracefully handles cases where you have 'f' orbitals 
                # (which would make the line 6 parts long instead of 5)
                mag_data.append(float(parts[-1]))
            except ValueError:
                continue # Skip if it still can't be converted to a float
                
    return mag_data

In [ ]:
def extract_charge_data(filename):
    charge_data = []
    with open(filename, 'r') as f:
        lines = f.readlines()
        
    # 1. Find the start of the LAST "total charge" block 
    charge_start = -1
    for i, line in enumerate(lines):
        # strip() ensures we match the header exactly
        if line.strip() == "total charge":
            charge_start = i
            
    if charge_start == -1:
        print("Warning: No total charge data found in OUTCAR.")
        return charge_data
        
    # 2. Parse the block
    # Start iterating 3 lines after the header to skip the blank line, "# of ion", and dashes
    for line in lines[charge_start + 4:]:
        # A line of dashes marks the end of the atomic values
        if "--------" in line: 
            break 
            
        parts = line.split()
        if len(parts) >= 2:
            try:
                # parts[-1] safely grabs the final 'tot' column
                charge_data.append(float(parts[-2]))
            except ValueError:
                continue 
                
    return charge_data

In [ ]:
M_C = []
M_Mn = []
M_Fe = []
M_N = []
M_Na = []

Q_C  = []
Q_Mn = []
Q_Fe = []
Q_N  = []
Q_Na = []

for i in range(0,64,2):
    file = f"./validation/VASP/{i}/OUTCAR"
    magnetizations = extract_outcar_data(file)
    print('----------------------------------------------')
    print(f"Extracted {len(magnetizations)} atomic values.")
    M_C.append(magnetizations[:192])
    M_Mn.append(magnetizations[192:192+32])
    M_Fe.append(magnetizations[192+32:192+64])
    M_N.append(magnetizations[192+64:192+64+192])
    M_Na.append(magnetizations[192+64+192:])
    
    charges = extract_charge_data(file)
    print(f"Extracted {len(charges)} atomic charge values.")
    Q_C.append(charges[:192])
    Q_Mn.append(charges[192:192+32])
    Q_Fe.append(charges[192+32:192+64])
    Q_N.append(charges[192+64:192+64+192])
    Q_Na.append(charges[192+64+192:])

    print(f"In dir = {i}. There're {len(Q_Na)} Na.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Initialize lists to store the arrays for each directory
M_C, M_Mn, M_Fe, M_N, M_Na = [], [], [], [], []
Q_C, Q_Mn, Q_Fe, Q_N, Q_Na = [], [], [], [], []

# Lists specifically for our plot (Means and Standard Deviations)
num_Na_list = []

avg_M_Fe_list, std_M_Fe_list = [], []
avg_M_Mn_list, std_M_Mn_list = [], []

avg_Q_Fe_list, std_Q_Fe_list = [], []
avg_Q_Mn_list, std_Q_Mn_list = [], []

for i in range(0, 64, 2):
    file = f"./validation/VASP/{i}/OUTCAR"
    
    # Extract data (Assuming extract_outcar_data and extract_charge_data are defined)
    magnetizations = extract_outcar_data(file)
    charges = extract_charge_data(file)
    
    mag_arr = np.array(magnetizations)
    chg_arr = np.array(charges)
    
    print('----------------------------------------------')
    print(f"Directory {i}: Extracted {len(mag_arr)} mag values, {len(chg_arr)} charge values.")
    
    # --- Slice Magnetizations ---
    m_c  = mag_arr[:192]
    m_mn = abs(mag_arr[192:224])       
    m_fe = abs(mag_arr[224:256])
    
    #m_mn = mag_arr[192:224]       
    #m_fe = mag_arr[224:256]   
    
    m_n  = mag_arr[256:448]       
    m_na = mag_arr[448:]
    m_mn_2 = mag_arr[192:224]       
    m_fe_2 = mag_arr[224:256]
    
    M_C.append(m_c)
    M_Mn.append(m_mn)
    M_Fe.append(m_fe)
    M_N.append(m_n)
    M_Na.append(m_na)
    
    # --- Slice Charges ---
    q_c  = chg_arr[:192]
    q_mn = chg_arr[192:224]
    q_fe = chg_arr[224:256]
    q_n  = chg_arr[256:448]
    q_na = chg_arr[448:]
    
    Q_C.append(q_c)
    Q_Mn.append(q_mn)
    Q_Fe.append(q_fe)
    Q_N.append(q_n)
    Q_Na.append(q_na)

    # --- Data for Plotting ---
    num_na = len(m_na)
    num_Na_list.append(num_na)
    
    # Calculate Mean and Std Dev for Magnetization
    avg_M_Fe_list.append(np.mean(m_fe))
    std_M_Fe_list.append(np.std(m_fe))
    avg_M_Mn_list.append(np.mean(m_mn))
    std_M_Mn_list.append(np.std(m_mn))

    # Calculate Mean and Std Dev for Charge
    avg_Q_Fe_list.append(np.mean(q_fe))
    std_Q_Fe_list.append(np.std(q_fe))
    avg_Q_Mn_list.append(np.mean(q_mn))
    std_Q_Mn_list.append(np.std(q_mn))

    print(f"In dir = {i}. There are {num_na} Na atoms.")

# ==========================================
# Convert Plotting Lists to Numpy Arrays
# ==========================================
num_Na_arr = np.array(num_Na_list)

avg_M_Fe_arr, std_M_Fe_arr = np.array(avg_M_Fe_list), np.array(std_M_Fe_list)
avg_M_Mn_arr, std_M_Mn_arr = np.array(avg_M_Mn_list), np.array(std_M_Mn_list)

avg_Q_Fe_arr, std_Q_Fe_arr = np.array(avg_Q_Fe_list), np.array(std_Q_Fe_list)
avg_Q_Mn_arr, std_Q_Mn_arr = np.array(avg_Q_Mn_list), np.array(std_Q_Mn_list)

# ==========================================
# Dual-Axis Plotting
# ==========================================

In [ ]:
fig, ax1 = plt.subplots(figsize=(singlecol*2, singlecol))
plt.rcParams.update({'font.size': 12})
# --- AXIS 1: Magnetization (Left) ---
color_M_fe = 'darkred'
color_M_mn = 'darkblue'

ax1.set_xlabel(r"$x$ in Na$_x$Mn[Fe(CN)$_6$]")
ax1.set_ylabel('|Ave. absolute magnetization| (|$\mu_B$|)', color='black')

# Use errorbar to show standard deviation
ax1.errorbar(num_Na_arr/32, avg_M_Fe_arr, yerr=std_M_Fe_arr, marker='o', linestyle='-', 
             color=color_M_fe, capsize=4, label='Mag Fe')
ax1.errorbar(num_Na_arr/32, avg_M_Mn_arr, yerr=std_M_Mn_arr, marker='s', linestyle='-', 
             color=color_M_mn, capsize=4, label='Mag Mn')

ax1.tick_params(axis='y', labelcolor='black')
#ax1.grid(True, linestyle='--', alpha=0.5)

# --- AXIS 2: Charge (Right) ---
#ax2 = ax1.twinx()  # Instantiate a second axes that shares the same x-axis

color_Q_fe = 'lightcoral'
color_Q_mn = 'dodgerblue'

#ax2.set_ylabel('Average Charge',color='black')

# Using dashed lines to visually separate Charge vs Magnetization
#ax2.errorbar(num_Na_arr, avg_Q_Fe_arr, yerr=std_Q_Fe_arr, marker='^', linestyle='--', 
#             color=color_Q_fe, capsize=4, label='Charge Fe')
#ax2.errorbar(num_Na_arr, avg_Q_Mn_arr, yerr=std_Q_Mn_arr, marker='v', linestyle='--', 
#             color=color_Q_mn, capsize=4, label='Charge Mn')

#ax2.tick_params(axis='y', labelcolor='black')

# --- Combine Legends from both axes ---
lines_1, labels_1 = ax1.get_legend_handles_labels()
#lines_2, labels_2 = ax2.get_legend_handles_labels()
#ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='best', fontsize=10)
ax1.legend(loc='best', frameon=False, fontsize=12)

#plt.title('Average Magnetization of Fe and Mn vs. Number of Na Atoms', fontsize=14)

fig.tight_layout()  # Ensure nothing is clipped
plt.show()

In [ ]:
abs(M_Fe[-2])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define your bins explicitly
# This creates 20 bins between -5 and 5 (Adjust these limits to fit your actual data!)
my_bins = np.linspace(-5, 5, 21) 
plt.rcParams.update({'font.size': 12})
# Create a figure with 1 row and 2 columns
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(singlecol*2, singlecol))

# --- Plot 1: Manganese (Mn) ---
# Pass 'bins=my_bins' instead of a number
data_Mn = [M_Mn[0], M_Mn[15], M_Mn[-2]]
colors = ['blue', 'orange', 'green']
labels = [f'x = {num_Na_arr[0]/32}', f'x = {num_Na_arr[15]/32}', f'x = {num_Na_arr[-2]/32}']

# 2. Pass the lists directly to a SINGLE hist call
ax1.hist(data_Mn, bins=my_bins, color=colors, label=labels, edgecolor='black')

ax1.set_title('Mn')
ax1.set_xlabel('Magnetization ($\mu_B$)')
ax1.set_ylabel('Count')
ax1.legend(frameon=False)
#ax1.grid(True, linestyle='--', alpha=0.5)

my_bins = np.linspace(-1.5, 1.5, 21)
# --- Plot 2: Iron (Fe) ---
data_Fe = [M_Fe[0], M_Fe[15], M_Fe[-2]]
colors = ['blue', 'orange', 'green']
labels = [f'x = {num_Na_arr[0]/32}', f'x = {num_Na_arr[15]/32}', f'x = {num_Na_arr[-2]/32}']

ax2.hist(data_Fe, bins=my_bins, color=colors, label=labels, edgecolor='black')

ax2.set_title('Fe')
ax2.set_xlabel('Magnetization ($\mu_B$)')
ax2.set_ylabel('Count')
ax2.legend(frameon=False)
#ax2.grid(True, linestyle='--', alpha=0.5)

# Adjust layout to prevent overlap and display
plt.tight_layout()
plt.show()

## POSCAR to CFG

In [ ]:
def read_POSCAR(filename):
    """
    Return all the relevant information contained in a POSCAR file.
    WARNING: in this function lattvec is in Angstroms
    """
    nruter = dict()
    nruter["lattvec"] = np.empty((3, 3))
    f = open(filename, "r")
    firstline = next(f)
    factor = 1. * float(next(f).strip())
    for i in range(3):
        nruter["lattvec"][:, i] = [float(j) for j in next(f).split()]
    nruter["lattvec"] *= factor
    line = next(f)
    fields = next(f).split()
    old = False
    try:
        int(fields[0])
    except ValueError:
        old = True
    if old:
        nruter["elements"] = firstline.split()
        nruter["numbers"] = np.array([int(i) for i in line.split()])
        typeline = "".join(fields)
    else:
        nruter["elements"] = line.split()
        nruter["numbers"] = np.array([int(i) for i in fields], dtype=np.intc)
        typeline = next(f)
    natoms = nruter["numbers"].sum()
    nruter["positions"] = np.empty((3, natoms))
    for i in range(natoms):
        nruter["positions"][:, i] = [float(j) for j in next(f).split()]
    f.close()
    nruter["types"] = []
    for i in range(len(nruter["numbers"])):
        nruter["types"] += [i] * nruter["numbers"][i]
    if typeline[0] == "C" or typeline[0] == "c":
        nruter["positions"] = np.linalg.solve(nruter["lattvec"],
                                              nruter["positions"] * factor)
    return nruter
    
def poscar2cfg(poscar,cfgfile):
    """
    Convert one poscar into mlip format and append to cfg file.
    """
    with open(cfgfile,'w') as f:
        f.write("BEGIN_CFG\n")
        f.write(" Size\n")
        f.write("    "+str(poscar["numbers"].sum())+"\n")
        f.write(" Supercell\n")
        for i in range(3):
            f.write("    {0[0]:>13.6f} {0[1]:>13.6f} {0[2]:>13.6f}\n".format(
                (poscar["lattvec"][:, i]).tolist()))
        f.write(" AtomData:  id type       cartes_x      cartes_y      cartes_z\n")
        for i in range(poscar["positions"].shape[1]):
            nelement = -1
            for element in range(len(poscar["numbers"])):
                if i < poscar["numbers"][:element+1].sum():
                    nelement=element
                    break
            f.write("    "+"{0[0]:10d} {0[1]:4d}".format([i+1,nelement]))
            f.write("  {0[0]:>13.6f} {0[1]:>13.6f} {0[2]:>13.6f}\n".format(np.dot(poscar["lattvec"], poscar["positions"])[:, i].tolist()))
        f.write("END_CFG\n")
        f.write("\n")

In [ ]:
for i in range(4):
    POSCAR = read_POSCAR(f'./validation/relax/{i}/CONTCAR')
    poscar2cfg(POSCAR, f'./validation/relax/relaxed_{i}.cfg')

In [ ]:
import numpy as np

def generate_na_sites(input_file, grid_res=(4, 4, 4)):
    with open(input_file, 'r') as f:
        lines = f.readlines()

    # 1. Extract Supercell and Atoms
    supercell = []
    atom_lines = []
    for i, line in enumerate(lines):
        if "Supercell" in line:
            supercell = [lines[i+1].split(), lines[i+2].split(), lines[i+3].split()]
            supercell = np.array(supercell, dtype=float)
        # Identify lines that contain atom data (starting with an integer ID)
        parts = line.split()
        if len(parts) >= 5 and parts[0].isdigit() and parts[1].isdigit():
            atom_lines.append(line.strip())

    # 2. Define the targeted box (Half of the supercell dimensions)
    max_x = 10.332026 / 4
    max_y = 10.332026 / 4
    max_z = 9.942532 / 4

    # Create coordinate grid
    x_coords = np.linspace(0, max_x, grid_res[0])
    y_coords = np.linspace(0, max_y, grid_res[1])
    z_coords = np.linspace(0, max_z, grid_res[2])

    count = 0
    for x in x_coords:
        for y in y_coords:
            for z in z_coords:
                new_size = len(atom_lines) + 1
                filename = f"MnFeC6N6_Na_{count}.cfg"
                
                with open(filename, 'w') as f:
                    f.write("BEGIN_CFG\n Size\n")
                    f.write(f"    {new_size}\n")
                    f.write(" Supercell\n")
                    for row in supercell:
                        f.write(f"    {row[0]:14.6f} {row[1]:14.6f} {row[2]:14.6f}\n")
                    f.write(" AtomData:  id type       cartes_x      cartes_y      cartes_z\n")
                    
                    # Write original 56 atoms
                    for line in atom_lines:
                        f.write(f"    {line}\n")
                    
                    # Add the single Na atom (Type 4)
                    f.write(f"           {new_size}    4       {x:14.6f} {y:14.6f} {z:14.6f}\n")
                    f.write("END_CFG\n")
                
                count += 1
    print(f"Generated {count} files in the sub-box.")

# Usage:
generate_na_sites('MnFeC6N6.cfg', grid_res=(5, 5, 5))

## P2D

### 01C

In [ ]:
import pandas as pd
import numpy as np
import io

file_path = './p2d/OCV_01C.txt'

SKIP_LINES = 27

try:
    df = pd.read_csv(
        file_path,
        skiprows=SKIP_LINES,
        header=None,
        sep='\s+',
        engine='python'
    )
    
    df.rename(columns={0: 'x'}, inplace=True)

    print("Data loaded successfully.")
    print("\nFirst 5 rows of the DataFrame:")
    print(df.head())
    
    print("\nDataFrame Information:")
    df.info()

except FileNotFoundError:
    print(f"Error: File not found at {file_path}. Make sure the file is in the correct location.")
except pd.errors.EmptyDataError:
    print("Error: The file is empty or only contains comments after skipping lines.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [ ]:
df.head()

In [ ]:
voltage_data = df.drop(columns=['x'])
column_means = voltage_data.mean(axis=0)
OCV_01C = column_means.values

In [ ]:
file_path = './p2d/01C.csv'

SKIP_LINES = 9

try:
    df = pd.read_csv(
        file_path,
        skiprows=SKIP_LINES,
        header=None,
        sep=',',
        engine='python'
    )
    
    #df.rename(columns={0: 'x'}, inplace=True)

    print("Data loaded successfully.")
    print("\nFirst 5 rows of the DataFrame:")
    print(df.head())
    
    print("\nDataFrame Information:")
    df.info()
    
except FileNotFoundError:
    print(f"Error: File not found at {file_path}. Make sure the file is in the correct location.")
except pd.errors.EmptyDataError:
    print("Error: The file is empty or only contains comments after skipping lines.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [ ]:
df.head()

In [ ]:
df[0].min(), df[0].max(), df[1].min(), df[1].max()

In [ ]:
X = np.linspace(1.85, 0, len(df[0]))
Y = np.linspace(61, 0, len(df[1]))

In [ ]:
X.shape, Y.shape, df[2].shape

In [ ]:
SOC = df.drop(columns=[0,1])
column_means = SOC.mean(axis=0)
SOC_all = column_means.values

lx_1, lx_2 = 1.25, 1.3
cx_1, cx_2 = 1.75, 1.8
rx_1, rx_2 = 2.2, 2.25

uy_1, uy_2 = 0.95, 1
cy_1, cy_2 = 0.5, 0.55
ly_1, ly_2 = 0, 0.05

##### Upper #####
Y_min, Y_max = uy_1, uy_2

X_min, X_max = lx_1, lx_2
condition_X = (df[0] >= X_min) & (df[0] <= X_max)
condition_Y = (df[1] >= Y_min) & (df[1] <= Y_max)
combined_mask = condition_X & condition_Y

upper_left = df[combined_mask].index.tolist()
####
X_min, X_max = cx_1, cx_2
condition_X = (df[0] >= X_min) & (df[0] <= X_max)
condition_Y = (df[1] >= Y_min) & (df[1] <= Y_max)
combined_mask = condition_X & condition_Y

upper_center = df[combined_mask].index.tolist()
####
X_min, X_max = rx_1, rx_2
condition_X = (df[0] >= X_min) & (df[0] <= X_max)
condition_Y = (df[1] >= Y_min) & (df[1] <= Y_max)
combined_mask = condition_X & condition_Y

upper_right = df[combined_mask].index.tolist()

##### Center #####
Y_min, Y_max = cy_1, cy_2

X_min, X_max = lx_1, lx_2
condition_X = (df[0] >= X_min) & (df[0] <= X_max)
condition_Y = (df[1] >= Y_min) & (df[1] <= Y_max)
combined_mask = condition_X & condition_Y

center_left = df[combined_mask].index.tolist()
####
X_min, X_max = cx_1, cx_2
condition_X = (df[0] >= X_min) & (df[0] <= X_max)
condition_Y = (df[1] >= Y_min) & (df[1] <= Y_max)
combined_mask = condition_X & condition_Y

center_center = df[combined_mask].index.tolist()
####
X_min, X_max = rx_1, rx_2
condition_X = (df[0] >= X_min) & (df[0] <= X_max)
condition_Y = (df[1] >= Y_min) & (df[1] <= Y_max)
combined_mask = condition_X & condition_Y

center_right = df[combined_mask].index.tolist()

##### Lower #####
Y_min, Y_max = ly_1, ly_2

X_min, X_max = lx_1, lx_2
condition_X = (df[0] >= X_min) & (df[0] <= X_max)
condition_Y = (df[1] >= Y_min) & (df[1] <= Y_max)
combined_mask = condition_X & condition_Y

lower_left = df[combined_mask].index.tolist()
####
X_min, X_max = cx_1, cx_2
condition_X = (df[0] >= X_min) & (df[0] <= X_max)
condition_Y = (df[1] >= Y_min) & (df[1] <= Y_max)
combined_mask = condition_X & condition_Y

lower_center = df[combined_mask].index.tolist()
####
X_min, X_max = rx_1, rx_2
condition_X = (df[0] >= X_min) & (df[0] <= X_max)
condition_Y = (df[1] >= Y_min) & (df[1] <= Y_max)
combined_mask = condition_X & condition_Y

lower_right = df[combined_mask].index.tolist()

In [ ]:
start_col_index = 2
end_col_index = len(df.columns) # Exclusive end index, covers up to the last column
columns_to_mean = list(range(start_col_index, end_col_index))

##### Upper #####
filtered_data = df.loc[upper_left, columns_to_mean]
SOC_upper_left = filtered_data.mean().tolist()

filtered_data = df.loc[upper_center, columns_to_mean]
SOC_upper_center = filtered_data.mean().tolist()

filtered_data = df.loc[upper_right, columns_to_mean]
SOC_upper_right = filtered_data.mean().tolist()

##### Center #####
filtered_data = df.loc[center_left, columns_to_mean]
SOC_center_left = filtered_data.mean().tolist()

filtered_data = df.loc[center_center, columns_to_mean]
SOC_center_center = filtered_data.mean().tolist()

filtered_data = df.loc[center_right, columns_to_mean]
SOC_center_right = filtered_data.mean().tolist()

##### Lower #####
filtered_data = df.loc[lower_left, columns_to_mean]
SOC_lower_left = filtered_data.mean().tolist()

filtered_data = df.loc[lower_center, columns_to_mean]
SOC_lower_center = filtered_data.mean().tolist()

filtered_data = df.loc[lower_right, columns_to_mean]
SOC_lower_right = filtered_data.mean().tolist()

In [ ]:
soc_data = [
    (SOC_upper_left, 'Upper Left', 'C0--'),
    (SOC_upper_center, 'Upper Center', 'C1--'),
    (SOC_upper_right, 'Upper Right', 'C2--'),
    (SOC_center_left, 'Center Left', 'C0-'),
    (SOC_center_center, 'Center Center', 'C1-'),
    (SOC_center_right, 'Center Right', 'C2-'),
    (SOC_lower_left, 'Lower Left', 'C0:'),
    (SOC_lower_center, 'Lower Center', 'C1:'),
    (SOC_lower_right, 'Lower Right', 'C2:')
]

fig, ax = plt.subplots(figsize=(10, 6))

for data, label, style in soc_data:
    # Plot each region's mean SOC over the column indices
    # We use different linestyles/colors to distinguish the regions:
    # C0 = Blue, C1 = Orange, C2 = Green
    # -- = Dashed (Upper), - = Solid (Center), : = Dotted (Lower)
    ax.plot(columns_to_mean, data, style, label=label, linewidth=2)
ax.plot(SOC_all, label='all', color='black')
ax.legend()#loc='upper left', bbox_to_anchor=(1.05, 1))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

# Create figure with wider width
fig, ax = plt.subplots(figsize=(singlecol, 1.2), dpi=200)  # Increased width from 1.5 to 2.5

# Create a ScalarMappable with your colormap and normalization
sm = ScalarMappable(cmap=cm.jet, norm=Normalize(vmin=0, vmax=2))
sm.set_array([])

# Create colorbar
cbar = plt.colorbar(sm, cax=ax, orientation='horizontal')
cbar.set_label(r"$x$ in Na$_x$Mn[Fe(CN)$_6$]")
cbar.set_ticks(np.linspace(0, 2, 5))
cbar.set_ticklabels(['0', '0.5', '1', '1.5', '2'])
ax.xaxis.set_ticks_position('top')
ax.xaxis.set_label_position('top')

plt.tight_layout()
plt.savefig('colorbar_only.svg', bbox_inches='tight')
plt.show()

In [ ]:
# ... (Previous code including plt.rcParams updates and imports) ...

# 1. Get both the Figure (fig) and Axes (ax) objects
fig, ax = plt.subplots(figsize = (singlecol, singlecol), dpi=200)

ax.set_aspect('equal') # You can try 'auto' or a specific number if 'equal' distorts it too much

# 2. Call the plotting function on the Axes object (ax)
data = df[400+2]*2
im = ax.tricontourf(df[0], df[1], data, levels=100, vmax=2, vmin=0, cmap=cm.jet)

divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = plt.colorbar(im, cax=cax)
cbar.set_label(r"$x$ in Na$_x$Mn[Fe(CN)$_6$]")
cbar.set_ticks(np.linspace(0, 2, 5))  # Remove the extra brackets
cbar.set_ticklabels(['0', '0.5', '1', '1.5', '2'])


# --- X-axis modification: 0 to 61 ---
num_x_ticks = len(x_tick_locations)
new_x_tick_labels = np.linspace(0, 61, num=num_x_ticks).astype(int)
ax.set_xticks(x_tick_locations) # Set the tick locations first
ax.set_xticklabels(new_x_tick_labels)

# --- Y-axis modification: 1.85 to 0 ---
num_y_ticks = len(y_tick_locations)
new_y_tick_labels = np.linspace(1.85, 0, num=num_y_ticks)
new_y_tick_labels_str = [f'{label:.2f}' for label in new_y_tick_labels]
ax.set_yticklabels(new_y_tick_labels_str)

# ... (All ax.axvspan calls) ...

ax.set_xlabel('Electrode depth (μm)')
ax.set_ylabel('Particle depth (μm)')

# 5. Display the plot
plt.tight_layout()
#plt.savefig('p2D'+'.svg', bbox_inches='tight')

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec

fig = plt.figure(figsize = (singlecol*3, singlecol*1.5), dpi=100, tight_layout = True)
grid = plt.GridSpec(3,7, hspace=0.25, wspace=0.05)

ax1 = fig.add_subplot(grid[0, :], frameon=False) #HC_SAXS_big
#ax2 = fig.add_subplot(grid[1, :], frameon=False) #PBA_WAXS_big
ax3 = fig.add_subplot(grid[1:, :]) #Echem

nested_grid_0 = GridSpecFromSubplotSpec(1, 7, subplot_spec=grid[0, :], wspace=0.08)
nested_grid_1 = GridSpecFromSubplotSpec(1, 7, subplot_spec=grid[1, :], wspace=0.08)

time_ls = [13, 200, 380, 1000, 1600, 2290, 2445]
#[13,100,200,360,400,743,1000, 1200,1400,1600,1800,2000,2280,2440]

for i in range(7):
    ax = fig.add_subplot(nested_grid_0[i])
    ax.tricontourf(df[0], df[1], df[time_ls[i]+2], levels=300, vmax=1, vmin=0, cmap=cm.jet)
    ax.set_title(f"{round(OCV_01C[time_ls[i]],2)} V", fontsize=12, color='black')
    ax.set_xticks([])
    ax.set_yticks([])
ax1.set_xticks([])
ax1.set_yticks([])

#for i in range(7):
#    ax = fig.add_subplot(nested_grid_1[i])
#    ax.tricontourf(df[0], df[1], df[time_ls[i+7]+2], levels=100, vmax=1, vmin=0, cmap=cm.jet)
#    ax.set_title(f"{round(OCV_01C[time_ls[i+7]],2)} V", fontsize=12, color='purple')
#    ax.set_xticks([])
#    ax.set_yticks([])
#ax2.set_xticks([])
#ax2.set_yticks([])

ax3.plot(np.arange(OCV_01C.shape[0])/360, OCV_01C, color='black', lw=2)

for i in range(7):
    x = np.arange(OCV_01C.shape[0])[time_ls[i]]
    ax3.scatter(x/360, OCV_01C[time_ls[i]], color='r', s=30)

#for i in range(7):
#    x = np.arange(OCV_01C.shape[0])[time_ls[i+7]]
#    ax3.scatter(x/360, OCV_01C[time_ls[i+7]], color='purple', s=30)


plt2 = ax3.twinx()
soc_data = [
    (SOC_upper_left, 'Left', 'C0-'),
    (SOC_upper_center, 'Center', 'C0--'),
    (SOC_upper_right, 'Right', 'C0:'),
    (SOC_center_left,'', 'C1-'),
    (SOC_center_center, '', 'C1--'),
    (SOC_center_right, '', 'C1:'),
    (SOC_lower_left, '', 'C2-'),
    (SOC_lower_center, '', 'C2--'),
    (SOC_lower_right, '', 'C2:')
]

for data, label, style in soc_data:
    # Plot each region's mean SOC over the column indices
    # We use different linestyles/colors to distinguish the regions:
    # C0 = Blue, C1 = Orange, C2 = Green
    # -- = Dashed (Upper), - = Solid (Center), : = Dotted (Lower)
    plt2.plot(np.arange(OCV_01C.shape[0])/360, np.array(data)*2, style, label=label, linewidth=2)

Y_start_data = 1.58
Y_end_data = 2# The top limit of your modified Y-axis labels

# Use axhspan to shade the region
# We use xmin=0 and xmax=1 to cover the entire horizontal extent of the plot
plt2.axhspan(ymin=Y_start_data,
           ymax=Y_end_data,
           xmin=0,
           xmax=7,
           facecolor='orange',
           alpha=0.15,
           label='Solid solution')

plt2.axhspan(ymin=0,
           ymax=0.02,
           xmin=0,
           xmax=7,
           facecolor='orange',
           alpha=0.15)
    
#plt2.plot(np.arange(OCV_01C.shape[0])/360, np.array(SOC_all)*2, label='average', color='grey', linewidth=2)

#plt2.legend(frameon=False,bbox_to_anchor=(0.43,0.5))
plt2.set_ylabel(r"$x$ in Na$_x$Mn[Fe(CN)$_6$]")
plt2.set_ylim(0,0.05)
ax3.set_ylim(2.5, 3.8)

ax3.set_xlim(0,7)#(np.arange(OCV_01C.shape[0]).max()+10)/360)
ax3.set_xlabel('Time (h)')
ax3.set_ylabel('Cell voltage (V)')
plt.tight_layout()
plt.savefig('p2D_01C_2'+'.svg', bbox_inches='tight')

### 05C

In [ ]:
import pandas as pd
import numpy as np
import io

file_path = './p2d/05C_OCV.txt'

SKIP_LINES = 27

try:
    df = pd.read_csv(
        file_path,
        skiprows=SKIP_LINES,
        header=None,
        sep='\s+',
        engine='python'
    )
    
    df.rename(columns={0: 'x'}, inplace=True)

    print("Data loaded successfully.")
    print("\nFirst 5 rows of the DataFrame:")
    print(df.head())
    
    print("\nDataFrame Information:")
    df.info()

except FileNotFoundError:
    print(f"Error: File not found at {file_path}. Make sure the file is in the correct location.")
except pd.errors.EmptyDataError:
    print("Error: The file is empty or only contains comments after skipping lines.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [ ]:
voltage_data = df.drop(columns=['x'])
column_means = voltage_data.mean(axis=0)
OCV_05C = column_means.values

In [ ]:
plt.figure()
plt.plot(np.arange(OCV_05C.shape[0]), OCV_05C)

In [ ]:
file_path = './p2d/05C.csv'

SKIP_LINES = 9

try:
    df = pd.read_csv(
        file_path,
        skiprows=SKIP_LINES,
        header=None,
        sep=',',
        engine='python'
    )
    
    #df.rename(columns={0: 'x'}, inplace=True)

    print("Data loaded successfully.")
    print("\nFirst 5 rows of the DataFrame:")
    print(df.head())
    
    print("\nDataFrame Information:")
    df.info()
    
except FileNotFoundError:
    print(f"Error: File not found at {file_path}. Make sure the file is in the correct location.")
except pd.errors.EmptyDataError:
    print("Error: The file is empty or only contains comments after skipping lines.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [ ]:
SOC = df.drop(columns=[0,1])
column_means = SOC.mean(axis=0)
SOC_all = column_means.values

lx_1, lx_2 = 1.25, 1.3
cx_1, cx_2 = 1.75, 1.8
rx_1, rx_2 = 2.2, 2.25

uy_1, uy_2 = 0.95, 1
cy_1, cy_2 = 0.5, 0.55
ly_1, ly_2 = 0, 0.05

##### Upper #####
Y_min, Y_max = uy_1, uy_2

X_min, X_max = lx_1, lx_2
condition_X = (df[0] >= X_min) & (df[0] <= X_max)
condition_Y = (df[1] >= Y_min) & (df[1] <= Y_max)
combined_mask = condition_X & condition_Y

upper_left = df[combined_mask].index.tolist()
####
X_min, X_max = cx_1, cx_2
condition_X = (df[0] >= X_min) & (df[0] <= X_max)
condition_Y = (df[1] >= Y_min) & (df[1] <= Y_max)
combined_mask = condition_X & condition_Y

upper_center = df[combined_mask].index.tolist()
####
X_min, X_max = rx_1, rx_2
condition_X = (df[0] >= X_min) & (df[0] <= X_max)
condition_Y = (df[1] >= Y_min) & (df[1] <= Y_max)
combined_mask = condition_X & condition_Y

upper_right = df[combined_mask].index.tolist()

##### Center #####
Y_min, Y_max = cy_1, cy_2

X_min, X_max = lx_1, lx_2
condition_X = (df[0] >= X_min) & (df[0] <= X_max)
condition_Y = (df[1] >= Y_min) & (df[1] <= Y_max)
combined_mask = condition_X & condition_Y

center_left = df[combined_mask].index.tolist()
####
X_min, X_max = cx_1, cx_2
condition_X = (df[0] >= X_min) & (df[0] <= X_max)
condition_Y = (df[1] >= Y_min) & (df[1] <= Y_max)
combined_mask = condition_X & condition_Y

center_center = df[combined_mask].index.tolist()
####
X_min, X_max = rx_1, rx_2
condition_X = (df[0] >= X_min) & (df[0] <= X_max)
condition_Y = (df[1] >= Y_min) & (df[1] <= Y_max)
combined_mask = condition_X & condition_Y

center_right = df[combined_mask].index.tolist()

##### Lower #####
Y_min, Y_max = ly_1, ly_2

X_min, X_max = lx_1, lx_2
condition_X = (df[0] >= X_min) & (df[0] <= X_max)
condition_Y = (df[1] >= Y_min) & (df[1] <= Y_max)
combined_mask = condition_X & condition_Y

lower_left = df[combined_mask].index.tolist()
####
X_min, X_max = cx_1, cx_2
condition_X = (df[0] >= X_min) & (df[0] <= X_max)
condition_Y = (df[1] >= Y_min) & (df[1] <= Y_max)
combined_mask = condition_X & condition_Y

lower_center = df[combined_mask].index.tolist()
####
X_min, X_max = rx_1, rx_2
condition_X = (df[0] >= X_min) & (df[0] <= X_max)
condition_Y = (df[1] >= Y_min) & (df[1] <= Y_max)
combined_mask = condition_X & condition_Y

lower_right = df[combined_mask].index.tolist()

In [ ]:
start_col_index = 2
end_col_index = len(df.columns) # Exclusive end index, covers up to the last column
columns_to_mean = list(range(start_col_index, end_col_index))

##### Upper #####
filtered_data = df.loc[upper_left, columns_to_mean]
SOC_upper_left = filtered_data.mean().tolist()

filtered_data = df.loc[upper_center, columns_to_mean]
SOC_upper_center = filtered_data.mean().tolist()

filtered_data = df.loc[upper_right, columns_to_mean]
SOC_upper_right = filtered_data.mean().tolist()

##### Center #####
filtered_data = df.loc[center_left, columns_to_mean]
SOC_center_left = filtered_data.mean().tolist()

filtered_data = df.loc[center_center, columns_to_mean]
SOC_center_center = filtered_data.mean().tolist()

filtered_data = df.loc[center_right, columns_to_mean]
SOC_center_right = filtered_data.mean().tolist()

##### Lower #####
filtered_data = df.loc[lower_left, columns_to_mean]
SOC_lower_left = filtered_data.mean().tolist()

filtered_data = df.loc[lower_center, columns_to_mean]
SOC_lower_center = filtered_data.mean().tolist()

filtered_data = df.loc[lower_right, columns_to_mean]
SOC_lower_right = filtered_data.mean().tolist()

In [ ]:
lower_right

In [ ]:
soc_data = [
    (SOC_upper_left, 'Upper Left', 'C0--'),
    (SOC_upper_center, 'Upper Center', 'C1--'),
    (SOC_upper_right, 'Upper Right', 'C2--'),
    (SOC_center_left, 'Center Left', 'C0-'),
    (SOC_center_center, 'Center Center', 'C1-'),
    (SOC_center_right, 'Center Right', 'C2-'),
    (SOC_lower_left, 'Lower Left', 'C0:'),
    (SOC_lower_center, 'Lower Center', 'C1:'),
    (SOC_lower_right, 'Lower Right', 'C2:')
]

fig, ax = plt.subplots(figsize=(10, 6))

for data, label, style in soc_data:
    # Plot each region's mean SOC over the column indices
    # We use different linestyles/colors to distinguish the regions:
    # C0 = Blue, C1 = Orange, C2 = Green
    # -- = Dashed (Upper), - = Solid (Center), : = Dotted (Lower)
    ax.plot(columns_to_mean, data, style, label=label, linewidth=2)
ax.plot(SOC_all, label='all', color='black')
ax.legend()#loc='upper left', bbox_to_anchor=(1.05, 1))

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec

fig = plt.figure(figsize = (singlecol*3, singlecol*1.5), dpi=300, tight_layout = True)
grid = plt.GridSpec(3,7, hspace=0.25, wspace=0.05)

ax1 = fig.add_subplot(grid[0, :], frameon=False) #HC_SAXS_big
#ax2 = fig.add_subplot(grid[1, :], frameon=False) #PBA_WAXS_big
ax3 = fig.add_subplot(grid[1:, :]) #Echem

nested_grid_0 = GridSpecFromSubplotSpec(1, 7, subplot_spec=grid[0, :], wspace=0.08)
nested_grid_1 = GridSpecFromSubplotSpec(1, 7, subplot_spec=grid[1, :], wspace=0.08)

time_ls=[3,90,170,250,330,450,490]
#[2,50,90,130,170,210,250,
#         290,330,370,410,450,490,493]

for i in range(7):
    ax = fig.add_subplot(nested_grid_0[i])
    ax.tricontourf(df[0], df[1], df[time_ls[i]+2], levels=100, vmax=1, vmin=0, cmap=cm.jet)
    ax.set_title(f"{round(OCV_05C[time_ls[i]],2)} V", fontsize=12, color='black')
    ax.set_xticks([])
    ax.set_yticks([])
ax1.set_xticks([])
ax1.set_yticks([])
'''
for i in range(7):
    ax = fig.add_subplot(nested_grid_1[i])
    ax.tricontourf(df[0], df[1], df[time_ls[i+7]+2], levels=100, vmax=1, vmin=0, cmap=cm.jet)
    ax.set_title(f"{round(OCV_05C[time_ls[i+7]],2)} V", fontsize=12, color='purple')
    ax.set_xticks([])
    ax.set_yticks([])
ax2.set_xticks([])
ax2.set_yticks([])
'''
ax3.plot(np.arange(OCV_05C.shape[0])/360, OCV_05C, color='black', lw=2)

for i in range(7):
    x = np.arange(OCV_05C.shape[0])[time_ls[i]]
    ax3.scatter(x/360, OCV_05C[time_ls[i]], color='r', s=30)
'''
for i in range(7):
    x = np.arange(OCV_05C.shape[0])[time_ls[i+7]]
    ax3.scatter(x/360, OCV_05C[time_ls[i+7]], color='purple', s=30)
'''
plt2 = ax3.twinx()
soc_data = [
    (SOC_upper_left, 'Left', 'C0-'),
    (SOC_upper_center, 'Center', 'C0--'),
    (SOC_upper_right, 'Right', 'C0:'),
    (SOC_center_left,'', 'C1-'),
    (SOC_center_center, '', 'C1--'),
    (SOC_center_right, '', 'C1:'),
    (SOC_lower_left, '', 'C2-'),
    (SOC_lower_center, '', 'C2--'),
    (SOC_lower_right, '', 'C2:')
]
for data, label, style in soc_data:
    # Plot each region's mean SOC over the column indices
    # We use different linestyles/colors to distinguish the regions:
    # C0 = Blue, C1 = Orange, C2 = Green
    # -- = Dashed (Upper), - = Solid (Center), : = Dotted (Lower)
    plt2.plot(np.arange(OCV_05C.shape[0])/360, np.array(data)*2, style, label=label, linewidth=2)
    
plt2.axhspan(ymin=Y_start_data,
           ymax=Y_end_data,
           xmin=0,
           xmax=7,
           facecolor='orange',
           alpha=0.15,
           label='Solid solution')

plt2.axhspan(ymin=0,
           ymax=0.02,
           xmin=0,
           xmax=7,
           facecolor='orange',
           alpha=0.15)#

plt2.legend(frameon=False,bbox_to_anchor=(0.4,0.5))
plt2.set_ylabel(r"$x$ in Na$_x$Mn[Fe(CN)$_6$]")
plt2.set_ylim(0,2)
ax3.set_ylim(2.5, 3.8)

ax3.set_xlim(-0.01,np.arange(OCV_05C.shape[0]).max()/360)
ax3.set_xlabel('Time (h)')
ax3.set_ylabel('Cell voltage (V)')
plt.savefig('p2D_05C'+'.svg', bbox_inches='tight')

In [ ]:
plt.figure()
plt.plot(np.linspace(0,1,OCV_05C.shape[0]), np.array(OCV_05C))
plt.plot(np.linspace(0,1,OCV_01C.shape[0]), np.array(OCV_01C))


In [ ]:
E_f = -235032.368493900140 
E_i = -254039.211738945014

#E_Na = -.14023100E+01
E_Na = -.13023100E+01
n_Na = 4095

V_ave = -(E_i-E_f-n_Na*E_Na)/n_Na
V_ave

In [ ]:
plot_log('./gcmc_9.log')

In [ ]:
%matplotlib inline
plot_log_no_V('./interface/300K_interface_free_9.log')
plot_log_no_V('./interface/300K_interface_free_10.log')

In [ ]:
plot_log_no_V('./interface/300K_interface_equil_9.log')
plot_log_no_V('./interface/300K_interface_equil_10.log')

In [ ]:
plot_log_no_V('./interface/interface_1.log')
plot_log_no_V('./interface/interface_2.log')
plot_log_no_V('./interface/interface_3.log')
plot_log_no_V('./interface/interface_4.log')

In [ ]:
element_map = {
    1: "C",
    2: "Mn", 
    3: "Fe",
    4: "N",
    5: "Na"
}
for i in [5,6,7,8]:#,3,4]:
    input_file = f'./interface/dump.interface_sub_{i}.trj'
    output_file = f'./interface/new_dump.interface_sub_{i}.trj'
# Convert the file
    convert_dump_file(input_file, output_file, element_map)
    print(f"Conversion complete. Modified file saved as {output_file}")

In [ ]:
%matplotlib inline
plot_log_no_V('./interface/interface_post_1.log')
plot_log_no_V('./interface/interface_post_2.log')
plot_log_no_V('./interface/interface_post_3.log')
plot_log_no_V('./interface/interface_post_4.log')

In [ ]:
plot_log_no_V('./interface/interface_sub2_1.log')
plot_log_no_V('./interface/interface_sub2_2.log')
plot_log_no_V('./interface/interface_sub2_3.log')
plot_log_no_V('./interface/interface_sub2_4.log')

In [ ]:
#plot_log_no_V('./interface/interface_sub2_1.log')
plot_log_no_V('./interface/400K_interface_sub2_2.log')
plot_log_no_V('./interface/400K_interface_sub2_3.log')
plot_log_no_V('./interface/400K_interface_sub2_4.log')

In [ ]:
plot_log_no_V('./interface/interface_sub2_1.log')

In [ ]:
plot_log_no_V('./interface/interface_sub_5.log')
plot_log_no_V('./interface/interface_sub_6.log')
plot_log_no_V('./interface/interface_sub_7.log')
plot_log_no_V('./interface/interface_sub_8.log')

In [ ]:
0.12262375 - 0.8402002 

In [ ]:
def bias_analysis(j, n1, n2, points):
    plt.figure(figsize=(8, 6))
    
    N_t_ls = np.linspace(n1, n2, points)
    fi_ls = [0.2, 1, 2,5]
    
    j=j
    for i in range(len(N_t_ls)):
        G_bias = -x**2 * (1 - 0.25*x**2) - 0.25*x - 1 + fi_ls[j]*(x-N_t_ls[i])**2
        f = sp.lambdify(x, G_bias, 'numpy')
        x_vals = np.linspace(-2, 2, 1000)
        y_vals = f(x_vals)
        plt.plot(x_vals, y_vals, label=r'$N_t = $' +f'{round(N_t_ls[i],2)}, N_min={round(x_vals[y_vals.argmin()],3)}', color=cm.winter(i/len(N_t_ls)))
        plt.scatter(x_vals[y_vals.argmin()], y_vals.min(), color='r')
        
        bias = fi_ls[j]*(x-N_t_ls[i])**2
        f = sp.lambdify(x, bias, 'numpy')
        x_vals = np.linspace(-2, 2, 1000)
        y_vals = f(x_vals)
        plt.plot(x_vals, y_vals, '--', color=cm.winter(i/len(N_t_ls)))#, label=r'$N_t = $' +f'{round(N_t_ls[i],2)}, N_min={round(x_vals[y_vals.argmin()],3)}', color=cm.winter(i/len(N_t_ls)))
        #plt.scatter(x_vals[y_vals.argmin()], y_vals.min(), color='r')
    
    # Define the function
    f_expr = -x**2 * (1 - 0.25*x**2) - 0.25*x - 1
    
    # Compute the derivative symbolically
    f_derivative_expr = sp.diff(f_expr, x)
    
    # Convert symbolic expressions to numerical functions
    f = sp.lambdify(x, f_expr, 'numpy')
    f_derivative = sp.lambdify(x, f_derivative_expr, 'numpy')
    
    # Generate x values
    x_vals = np.linspace(-2, 2, 1000)
    y_vals = f(x_vals)
    y_derivative_vals = f_derivative(x_vals)
    
    # Plot function and its derivative
    plt.plot(x_vals, y_vals, label=f'G(x) N_min={round(x_vals[y_vals.argmin()],3)}', color='black')
    plt.scatter(x_vals[y_vals.argmin()], y_vals.min(), color='black')
    plt.plot(x_vals, y_derivative_vals, label=r"$mu(x)$", color='r', linestyle='--')
    
    # Axes, grid, and labels
    plt.axhline(0, color='black', linewidth=0.5)
    plt.axvline(0, color='black', linewidth=0.5)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(bbox_to_anchor=(1, 1))
    plt.xlabel("x")
    plt.ylabel("Function Value")
    plt.ylim(-2.5,2.5)
    plt.title(f"Fi = {fi_ls[j]}")


In [ ]:
for i in range(4):
    bias_analysis(i,1.5,2,6)

In [ ]:
for i in range(4):
    bias_analysis(i,0,1,6)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

plt.figure(figsize=(8, 6))

N_t_ls = np.linspace(0,1,6)
fi_ls = [0.2, 1, 2]

j=0
for i in range(len(N_t_ls)):
    G_bias = -x**2 * (1 - 0.25*x**2) - 0.25*x - 1 + fi_ls[j]*(x-N_t_ls[i])**2
    f = sp.lambdify(x, G_bias, 'numpy')
    x_vals = np.linspace(-2, 2, 1000)
    y_vals = f(x_vals)
    plt.plot(x_vals, y_vals, label=r'$N_t = $' +f'{round(N_t_ls[i],2)}, N_min={round(x_vals[y_vals.argmin()],3)}', color=cm.winter(i/len(N_t_ls)))
    plt.scatter(x_vals[y_vals.argmin()], y_vals.min(), color='r')

# Define the function
f_expr = -x**2 * (1 - 0.25*x**2) - 0.25*x - 1

# Compute the derivative symbolically
f_derivative_expr = sp.diff(f_expr, x)

# Convert symbolic expressions to numerical functions
f = sp.lambdify(x, f_expr, 'numpy')
f_derivative = sp.lambdify(x, f_derivative_expr, 'numpy')

# Generate x values
x_vals = np.linspace(-2, 2, 1000)
y_vals = f(x_vals)
y_derivative_vals = f_derivative(x_vals)

# Plot function and its derivative
plt.plot(x_vals, y_vals, label=r'$G(x) = -x^2 (1 - x^2) - 0.25x - 0.5$', color='black')
plt.plot(x_vals, y_derivative_vals, label=r"$mu(x)$", color='r', linestyle='--')

# Axes, grid, and labels
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.5)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(bbox_to_anchor=(1, 1))
plt.xlabel("x")
plt.ylabel("Function Value")
plt.ylim(-2.5,2.5)
plt.title(f"Fi = {fi_ls[j]}")


In [ ]:
y_vals.argmin()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

plt.figure(figsize=(8, 6))

N_t_ls = np.linspace(1.5,2,9)
fi_ls = [0.2, 1, 2]

j=1
for i in range(len(N_t_ls)):
    G_bias = -x**2 * (1 - 0.25*x**2) - 0.25*x - 1 + fi_ls[j]*(x-N_t_ls[i])**2
    f = sp.lambdify(x, G_bias, 'numpy')
    x_vals = np.linspace(-2, 2, 1000)
    y_vals = f(x_vals)
    plt.plot(x_vals, y_vals, label=r'$N_t = $' +f'{N_t_ls[i]}', color=cm.winter(i/len(N_t_ls)))

# Define the function
f_expr = -x**2 * (1 - 0.25*x**2) - 0.25*x - 1

# Compute the derivative symbolically
f_derivative_expr = sp.diff(f_expr, x)

# Convert symbolic expressions to numerical functions
f = sp.lambdify(x, f_expr, 'numpy')
f_derivative = sp.lambdify(x, f_derivative_expr, 'numpy')

# Generate x values
x_vals = np.linspace(-2, 2, 1000)
y_vals = f(x_vals)
y_derivative_vals = f_derivative(x_vals)

# Plot function and its derivative
plt.plot(x_vals, y_vals, label=r'$G(x) = -x^2 (1 - x^2) - 0.25x - 0.5$', color='black')
plt.plot(x_vals, y_derivative_vals, label=r"$mu(x)$", color='r', linestyle='--')

# Axes, grid, and labels
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.5)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(bbox_to_anchor=(1, 1))
plt.xlabel("x")
plt.ylabel("Function Value")
plt.ylim(-2.5,2.5)
plt.title(f"Fi = {fi_ls[j]}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

plt.figure(figsize=(8, 6))

N_t_ls = np.linspace(0,1,6)
fi_ls = [0.2, 1, 2]

j=1
for i in range(len(N_t_ls)):
    G_bias = -x**2 * (1 - 0.25*x**2) - 0.25*x - 1 + fi_ls[j]*(x-N_t_ls[i])**2
    f = sp.lambdify(x, G_bias, 'numpy')
    x_vals = np.linspace(-2, 2, 1000)
    y_vals = f(x_vals)
    plt.plot(x_vals, y_vals, label=r'$N_t = $' +f'{N_t_ls[i]}', color=cm.winter(i/len(N_t_ls)))

# Define the function
f_expr = -x**2 * (1 - 0.25*x**2) - 0.25*x - 1

# Compute the derivative symbolically
f_derivative_expr = sp.diff(f_expr, x)

# Convert symbolic expressions to numerical functions
f = sp.lambdify(x, f_expr, 'numpy')
f_derivative = sp.lambdify(x, f_derivative_expr, 'numpy')

# Generate x values
x_vals = np.linspace(-2, 2, 1000)
y_vals = f(x_vals)
y_derivative_vals = f_derivative(x_vals)

# Plot function and its derivative
plt.plot(x_vals, y_vals, label=r'$G(x) = -x^2 (1 - x^2) - 0.25x - 0.5$', color='black')
plt.plot(x_vals, y_derivative_vals, label=r"$mu(x)$", color='r', linestyle='--')

# Axes, grid, and labels
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.5)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(bbox_to_anchor=(1, 1))
plt.xlabel("x")
plt.ylabel("Function Value")
plt.ylim(-2.5,2.5)
plt.title(f"Fi = {fi_ls[j]}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

# Define the symbolic variable
x = sp.symbols('x')

# Define the function
f_expr = -x**2 * (1 - 0.25*x**2) - 0.25*x - 1

# Compute the derivative symbolically
f_derivative_expr = sp.diff(f_expr, x)

# Convert symbolic expressions to numerical functions
f = sp.lambdify(x, f_expr, 'numpy')
f_derivative = sp.lambdify(x, f_derivative_expr, 'numpy')

# Generate x values
x_vals = np.linspace(-2, 2, 1000)
y_vals = f(x_vals)
y_derivative_vals = f_derivative(x_vals)

# Plot function and its derivative
plt.figure(figsize=(8, 6))
plt.plot(x_vals, y_derivative_vals, label=r"$mu(x)$", color='r', linestyle='--')

N_t_ls = np.linspace(-1.75,1.75,8)
fi_ls = [0.2, 1, 2]

j=1
for i in range(len(N_t_ls)):
    G_bias = -x**2 * (1 - 0.25*x**2) - 0.25*x - 1 + fi_ls[j]*(x-N_t_ls[i])**2
    f = sp.lambdify(x, G_bias, 'numpy')
    x_vals = np.linspace(-2, 2, 1000)
    y_vals = f(x_vals)
    plt.plot(x_vals, y_vals, label=r'$N_t = $' +f'{N_t_ls[i]}', color=cm.jet(i/len(N_t_ls)))
plt.plot(x_vals, y_vals, label=r'$G(x) = -x^2 (1 - x^2) - 0.25x - 0.5$', color='black')
# Axes, grid, and labels
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.5)
plt.grid(True, linestyle='--', alpha=0.6)
#plt.legend()
plt.xlabel("x")
plt.ylabel("Function Value")
#plt.ylim(-2.5,0)
plt.title(f"Fi = {fi_ls[j]}")


In [ ]:
np.linspace(0,1,11)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

# Define the symbolic variable
x = sp.symbols('x')

# Define the function
f_expr = -x**2 * (1 - 0.25*x**2) - 0.25*x - 1

# Compute the derivative symbolically
f_derivative_expr = sp.diff(f_expr, x)

# Convert symbolic expressions to numerical functions
f = sp.lambdify(x, f_expr, 'numpy')
f_derivative = sp.lambdify(x, f_derivative_expr, 'numpy')

# Generate x values
x_vals = np.linspace(-2, 2, 1000)
y_vals = f(x_vals)
y_derivative_vals = f_derivative(x_vals)

# Plot function and its derivative
plt.figure(figsize=(8, 6))
plt.plot(x_vals, y_vals, label=r'$G(x) = -x^2 (1 - x^2) - 0.25x - 0.5$', color='black')
plt.plot(x_vals, y_derivative_vals, label=r"$mu(x)$", color='r', linestyle='--')

N_t_ls = [0.2, 0.1, 0, -0.1, -0.2]
fi_ls = [0.2, 1, 2]

j=1
for i in range(len(N_t_ls)):
    G_bias = -x**2 * (1 - 0.25*x**2) - 0.25*x - 1 + fi_ls[j]*(x-N_t_ls[i])**2
    f = sp.lambdify(x, G_bias, 'numpy')
    x_vals = np.linspace(-2, 2, 1000)
    y_vals = f(x_vals)
    plt.plot(x_vals, y_vals, label=r'$N_t = $' +f'{N_t_ls[i]}', color=cm.winter(i/len(N_t_ls)))

# Axes, grid, and labels
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.5)
plt.grid(True, linestyle='--', alpha=0.6)
#plt.legend()
plt.xlabel("x")
plt.ylabel("Function Value")
plt.ylim(-2.5,0)
plt.title(f"Fi = {fi_ls[j]}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

# Define symbolic variable and function
x = sp.symbols('x')
f_expr = -x**2 * (1 - x**2) - 0.25*x - 0.5

# Compute derivative
f_derivative_expr = sp.diff(f_expr, x)

# Solve f'(x) = 0
critical_points = sp.solve(f_derivative_expr, x)
critical_points = [p.evalf() for p in critical_points]  # Convert to numerical values

# Convert symbolic functions to numerical functions
f = sp.lambdify(x, f_expr, 'numpy')

# Generate x values for plotting
x_vals = np.linspace(-2, 2, 400)
y_vals = f(x_vals)

# Compute y-values of critical points
y_critical = [f(xc) for xc in critical_points]

# Plot function
plt.figure(figsize=(8, 6))
plt.plot(x_vals, y_vals, label=r'$f(x)$', color='b')

# Mark critical points
plt.scatter(critical_points, y_critical, color='r', zorder=5, label="Critical Points")
for xc, yc in zip(critical_points, y_critical):
    plt.text(xc, yc, f"  x={xc:.2f}", verticalalignment='bottom', fontsize=10, color='r')

# Axes, grid, and labels
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.5)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.xlabel("x")
plt.ylabel("y")
plt.title("Function and Critical Points")

# Show the plot
plt.show()

# Print the critical points
print("Critical points (where f'(x) = 0):", critical_points)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

# Define symbolic variable and function
x = sp.symbols('x')
f_expr = -x**2 * (1 - x**2) - 0.25*x - 0.5

# Compute derivative
f_derivative_expr = sp.diff(f_expr, x)

# Convert symbolic expressions to numerical functions
f = sp.lambdify(x, f_expr, 'numpy')
f_derivative = sp.lambdify(x, f_derivative_expr, 'numpy')

# Choose a point x0
x0 = 1  # Change this as needed

# Compute y0 and the slopes
y0 = f(x0)
m_tangent = f_derivative(x0)
m_cotangent = -1 / m_tangent if m_tangent != 0 else np.inf  # Avoid division by zero

# Define tangent and cotangent lines
def tangent_line(x_vals):
    return m_tangent * (x_vals - x0) + y0

def cotangent_line(x_vals):
    return m_cotangent * (x_vals - x0) + y0 if m_cotangent != np.inf else np.full_like(x_vals, y0)

# Generate x values
x_vals = np.linspace(-2, 2, 400)
y_vals = f(x_vals)
y_tangent = tangent_line(x_vals)
y_cotangent = cotangent_line(x_vals)

# Plot function, tangent, and cotangent lines
plt.figure(figsize=(8, 6))
plt.plot(x_vals, y_vals, label=r'$f(x)$', color='b')
plt.plot(x_vals, y_tangent, label="Tangent line", color='g', linestyle='--')
if m_cotangent != np.inf:
    plt.plot(x_vals, y_cotangent, label="Cotangent line", color='r', linestyle=':')

# Mark the point (x0, y0)
plt.scatter([x0], [y0], color='black', zorder=5, label=f'Point ({x0}, {y0:.2f})')

# Axes, grid, and labels
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.5)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.xlabel("x")
plt.ylabel("y")
plt.title("Function, Tangent, and Cotangent Lines")

# Show the plot
plt.show()


## LV12 scale by force

In [ ]:
for i in range(2):
    file = f'./validation/gcmc_{i+1}.log'
    plot_log_no_V(file)

In [ ]:
for i in range(1,9):
    file = f'gcmc_{i}.log'
    plot_log_no_V(file)

In [ ]:
plot_log('gcmc_4.log')

In [ ]:
plot_log('gcmc_6.log')

# Raise Temp

In [ ]:
plot_log_diff('./production_run/free_E/test/temp_1.log')

In [ ]:
plot_log_diff('./production_run/free_E/test/temp_2.log')

### Prepare data 

In [ ]:
a=range(400,4000,400)
a[1]

In [ ]:
for i in range(400,4000,400):
    file = f'./production_run/free_E/test/{i}_temp.log'
    plot_log_diff(file)

In [ ]:
for i in [400]:
    file = f'./production_run/free_E/test/diffusion_{i}/diffusion.log'
    plot_log_free(file)

In [ ]:
plot_log_free('./production_run/free_E/test/diffusion_800/diffusion.log')

## Free energy

In [ ]:
Nt = 200
mu_ls = [-2.5, -2.5, -2.5, -2.5, 1, 1, 1, 1]
fi_ls = [0.001, 0.002, 0.005, 0.01, 0.001, 0.002, 0.005, 0.01]
Nf_ls = [197, 197, 196, 199, 206, 203, 201, 198]
mu_e_ls_1 = []
mu_e_ls_2 = []

for i in range(8):
    mu = mu_ls[i]
    fi = fi_ls[i]
    Nf = Nf_ls[i]
    mu_e_ls_2.append(-2*fi*(Nf-Nt))

In [ ]:
mu_e_ls_1

In [ ]:
mu_e_ls_2

In [ ]:
for i in range(15):
    file = f'./production_run/free_E/free_{i+1}.log'
    plot_log_free(file)

In [ ]:
for i in range(4,8):
    file = f'./production_run/free_E/gcmc_{i+1}.log'
    plot_log_free(file)

In [ ]:
file = f'./test_lammps/gcmc_8.log'
plot_log(file)

## dump convertor

In [ ]:
element_map = {
    1: "C",
    2: "Mn", 
    3: "Fe",
    4: "N",
    5: "Na",
    6: "O",
    7: "H"
}
temp=800
i=3600
input_file = f'./production_run/free_E/test/diffusion_400/dump.400_diffuse_3600.trj'
output_file = f'./production_run/free_E/test/diffusion_400/new_dump.400_diffuse_3600.trj'
# Convert the file
convert_dump_file(input_file, output_file, element_map)
print(f"Conversion complete. Modified file saved as {output_file}")

### plot_Na_profile_bias

In [ ]:
def plot_Na_profile_bias(temp, n_1, n_2, mu_1, step, fi, Nt, serie=0):
    plt.rcParams.update({'legend.fontsize': 12})
    fig, ax1 = plt.subplots(figsize=(10, 6))
    temp, n_1, n_2, mu_1,  step = temp, n_1, n_2, mu_1,  step
    
    mu_ls = np.linspace(mu_1, (n_2-n_1-1)*step+mu_1, (n_2-n_1))
    nNa_ls = []
    
    for i in range(n_1, n_2):
        if serie==0:
            file = f'./production_run/free_E/test/bias_{temp}/bias_{i+1}.log'
        elif serie!=0:
            file = f'./production_run/free_E/test/bias_{temp}/bias_{i+1}_{serie}.log'
        log_data = parse_lammps_free(file)

        if len(log_data['atoms']) > 0:
            natom = np.array(log_data['atoms'])-28672
            step = np.array(log_data['Step'])
            nNa=natom[-1]
            ax1.plot(step[:-1], natom[:-1], color=cm.jet((i-n_1)/(n_2-n_1)),
                     label=f'mu_{i+1} = {round(mu_ls[i-n_1],4)}  Na_f = {nNa}')#  V = {round(V_ave,2)}')#, marker='*')
    
    ax1.set_xlabel('Timestep (0.4 fs/step)')
    ax1.set_ylabel('Number of Na+')
    plt.title(f'{temp} K  mu = {mu_1} ~ {mu_ls[-1]}, fi={fi}, Nt={Nt}')
    ax1.legend(bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.grid()

In [ ]:
def plot_Na_profile_Nt(temp, n_1, n_2, Nt_1, step, fi, mu):
    fig, ax1 = plt.subplots(figsize=(10, 6))
    temp, n_1, n_2, mu_1,  step = temp, n_1, n_2, mu_1,  step
    
    mu_ls = np.linspace(mu_1, (n_2-n_1-1)*step+mu_1, (n_2-n_1))
    nNa_ls = []
    
    for i in range(n_1, n_2):
        file = f'./production_run/free_E/test/bias_{temp}/bias_{i+1}.log'
        log_data = parse_lammps_free(file)
        """
        E_Na = -.14023100E+01
        my_list = log_data['TotEng'][20:40]
        E_i = sum(my_list) / len(my_list)
        my_list = log_data['TotEng'][-20:]
        E_f = sum(my_list) / len(my_list)
        my_list = log_data['v_nNa'][20:40]
        n_Na_i = sum(my_list) / len(my_list)
        my_list = log_data['v_nNa'][-20:]
        n_Na_f = sum(my_list) / len(my_list)   
        n_Na = n_Na_i-n_Na_f
        V_ave = -(E_i-E_f-n_Na*E_Na)/n_Na
        """   
        natom = np.array(log_data['atoms'])-28672
        step = np.array(log_data['Step'])
        nNa=natom[-1]
        ax1.plot(step[:-1], natom[:-1], color=cm.jet((i-n_1)/(n_2-n_1)),
                 label=f'mu_{i+1} = {round(mu_ls[i-n_1],4)}  Na_f = {nNa}')#  V = {round(V_ave,2)}')#, marker='*')
    
    #ax1.hlines(512,0,3e6, linestyle='--', color='black')
    #ax1.hlines(0,0,3e6, linestyle='--', color='black')
    
    ax1.set_xlabel('Timestep (0.4 fs/step)')
    ax1.set_ylabel('Number of Na+')
    #ax1.set_xlim(0,3e6)
    plt.title(f'{temp} K  Nt = {mu_1} ~ {mu_ls[-1]}, fi={fi}, mu={Nt}')
    #ax1.tick_params(axis='y', labelcolor=color)
    #ax1.set_ylim(-5,515)
    ax1.legend(bbox_to_anchor=(1, 1))
    plt.grid()

### plot_Na_profile

In [ ]:
def plot_Na_profile(temp, n_1, n_2, mu_1, step):
    fig, ax1 = plt.subplots(figsize=(10, 6))
    temp, n_1, n_2, mu_1,  step = temp, n_1, n_2, mu_1,  step
    
    mu_ls = np.linspace(mu_1, (n_2-n_1-1)*step+mu_1, (n_2-n_1))
    nNa_ls = []
    
    for i in range(n_1, n_2):
        file = f'./production_run/free_E/test/1100/free_{i+1}.log'
        log_data = parse_lammps_free(file)
        """
        E_Na = -.14023100E+01
        my_list = log_data['TotEng'][20:40]
        E_i = sum(my_list) / len(my_list)
        my_list = log_data['TotEng'][-20:]
        E_f = sum(my_list) / len(my_list)
        my_list = log_data['v_nNa'][20:40]
        n_Na_i = sum(my_list) / len(my_list)
        my_list = log_data['v_nNa'][-20:]
        n_Na_f = sum(my_list) / len(my_list)   
        n_Na = n_Na_i-n_Na_f
        V_ave = -(E_i-E_f-n_Na*E_Na)/n_Na
        """   
        natom = np.array(log_data['atoms'])-28672
        step = np.array(log_data['Step'])
        nNa=natom[-1]
        ax1.plot(step[:-1], natom[:-1], color=cm.jet((i-n_1)/(n_2-n_1)),
                 label=f'mu_{i+1} = {round(mu_ls[i-n_1],4)}  Na_f = {nNa}')#  V = {round(V_ave,2)}')#, marker='*')
    
    #ax1.hlines(512,0,3e6, linestyle='--', color='black')
    #ax1.hlines(0,0,3e6, linestyle='--', color='black')
    
    ax1.set_xlabel('Timestep (0.4 fs/step)')
    ax1.set_ylabel('Number of Na+')
    #ax1.set_xlim(0,3e6)
    plt.title(f'{temp} K  mu = {mu_1} ~ {mu_ls[-1]}')
    #ax1.tick_params(axis='y', labelcolor=color)
    #ax1.set_ylim(-5,515)
    ax1.legend(bbox_to_anchor=(1, 1))
    plt.grid()

In [ ]:
def plot_Na_profile_from_dump(temp, n_1, n_2, mu_1,  step):
    temp, n_1, n_2, mu_1,  step = temp, n_1, n_2, mu_1,  step
    
    mu_ls = np.linspace(mu_1, (n_2-n_1-1)*step+mu_1, (n_2-n_1))
    timestep_ls = []
    num_atoms_ls = []
    for i in range(n_1, n_2):
        trj_filename = f'./production_run/free_E/test/{temp}/dump.gcmc_{i+1}.trj'  # Replace with your actual file name
        timestep, num_atoms = extract_timestep_and_atom_count(trj_filename)
        timestep_ls.append(timestep)
        num_atoms_ls.append(num_atoms)
        
    fig, ax1 = plt.subplots(figsize=(10, 6))
    for i in range(n_1, n_2):
        nNa=-28672+num_atoms_ls[i-n_1][-1]
        ax1.plot(timestep_ls[i-n_1], -28672+num_atoms_ls[i-n_1], color=cm.jet((i-n_1)/(n_2-n_1)),
                 label=f'mu_{i+1} = {round(mu_ls[i-n_1],4)}  Na_f = {nNa}')#  V = {round(V_ave,2)}')#, marker='*')
    plt.title(f'{temp} K  mu = {mu_1} ~ {mu_ls[-1]}')   
    plt.legend(bbox_to_anchor=(1, 1))
    plt.grid()
    ax1.set_xlabel('Timestep (0.4 fs/step)')
    ax1.set_ylabel('Number of Na+')

In [ ]:
def plot_Na_bias_from_dump(temp, n_1, n_2, mu_1, step, fi, Nt):
    temp, n_1, n_2, mu_1,  step = temp, n_1, n_2, mu_1,  step
    
    mu_ls = np.linspace(mu_1, (n_2-n_1-1)*step+mu_1, (n_2-n_1))
    timestep_ls = []
    num_atoms_ls = []
    for i in range(n_1, n_2):
        trj_filename = f'./production_run/free_E/test/bias_{temp}/dump.bias_{i+1}.trj'  # Replace with your actual file name
        timestep, num_atoms = extract_timestep_and_atom_count(trj_filename)
        timestep_ls.append(timestep)
        num_atoms_ls.append(num_atoms)
        
    fig, ax1 = plt.subplots(figsize=(10, 6))
    for i in range(n_1, n_2):
        nNa=-28672+num_atoms_ls[i-n_1][-1]
        ax1.plot(timestep_ls[i-n_1], -28672+num_atoms_ls[i-n_1], color=cm.jet((i-n_1)/(n_2-n_1)),
                 label=f'mu_{i+1} = {round(mu_ls[i-n_1],4)}  Na_f = {nNa}')#  V = {round(V_ave,2)}')#, marker='*')
    plt.title(f'{temp} K  mu = {mu_1} ~ {mu_ls[-1]}, fi={fi}, Nt={Nt}')   
    plt.legend(bbox_to_anchor=(1, 1))
    plt.grid()
    ax1.set_xlabel('Timestep (0.4 fs/step)')
    ax1.set_ylabel('Number of Na+')

In [ ]:
def extract_timestep_and_atom_count(trj_file):
    timestep = []
    num_atoms = []
    
    with open(trj_file, 'r') as file:
        for line in file:
            if "ITEM: TIMESTEP" in line:
                timestep.append(int(next(file).strip()))  # Read the next line for TIMESTEP value
            elif "ITEM: NUMBER OF ATOMS" in line:
                num_atoms.append(int(next(file).strip()))  # Read the next line for atom count
    return np.array(timestep), np.array(num_atoms)

In [ ]:
extract_to_data_file(f'./production_run/free_E/test/600/dump.gcmc_{i+1}.trj',f'./production_run/free_E/test/600/gcmc_{i+1}.data')

In [ ]:
# Example usage:
timestep_ls = []
num_atoms_ls = []
for i in range(21):
    trj_filename = f'./production_run/free_E/test/dump.gcmc_{i+1}.trj'  # Replace with your actual file name
    timestep, num_atoms = extract_timestep_and_atom_count(trj_filename)
    timestep_ls.append(timestep)
    num_atoms_ls.append(num_atoms)

In [ ]:
mu_ls = np.linspace(mu_1, (n_2-n_1-1)*step+mu_1, (n_2-n_1))
mu_ls

## 1100K

In [ ]:
plot_Na_profile(1100, 0, 11, -2.5, 0.05)

In [ ]:
for i in range(40,43):
    file = f'./production_run/free_E/test/1100/free_{i+1}.log'
    plot_log_free(file)

In [ ]:
plot_Na_profile(1100, 33, 42, -2.19, 0.01)

In [ ]:
plot_Na_profile(1100, 11, 22, -1.9, 0.05)

### plot: run_RDF

In [ ]:
for i in [12]:
    file = f'./production_run/free_E/test/1100/free_{i}.log'
    plot_log_free(file)

In [ ]:
for i in [9,10]:
    file = f'./interface/300K_interface_free_{i}.log'
    plot_log_no_V(file)

In [ ]:
singlecol = 3.5
plt.rcParams.update({'font.size': 8})
#plt.rcParams.update({'font.sans-serif': 'Helvetica'})
plt.rcParams.update({'lines.linewidth':1})
plt.rcParams.update({'axes.linewidth':0.5})
plt.rcParams.update({'lines.markersize':3})
plt.rcParams.update({'savefig.dpi':300})
plt.rcParams.update({'legend.fontsize': 8})

file = f'./interface_1.log'
log_data = parse_lammps_free(file)
T = np.array(log_data['Step'])
E = np.array(log_data['TotEng'])
V = np.array(log_data['Volume'])
atoms = np.array(log_data['atoms'])-28672

fig, ax1 = plt.subplots(figsize=(singlecol*2,singlecol*0.75), dpi=200)

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Simulation time (ps)')
ax1.set_ylabel('Total E (keV)', color=color)
ax1.plot(T*0.0004, E/1000, color=color, marker='o', markersize=3)
#ax1.vlines(
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel(r'Volume (nm$^3$)', color=color)
ax2.plot(T[10:]*0.0004, V[10:]/1000, color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 40)) 
ax3.set_ylabel(r"$x$ in Na$_x$Mn[Fe(CN)$_6$]", color=color)
ax3.plot(T*0.0004, atoms/2048, color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)

nNa = log_data['v_nNa'][-1]

file = f'./interface_2.log'
log_data = parse_lammps_free(file)
T = np.array(log_data['Step'])
E = np.array(log_data['TotEng'])
V = np.array(log_data['Volume'])
atoms = np.array(log_data['atoms'])-28672


# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Simulation time (ps)')
ax1.set_ylabel('Total E (keV)', color=color)
ax1.plot(T*0.0004, E/1000, color=color, marker='o', markersize=3)
#ax1.vlines(
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel(r'Volume (nm$^3$)', color=color)
ax2.plot(T[10:]*0.0004, V[10:]/1000, color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 40)) 
ax3.set_ylabel(r"$x$ in Na$_x$Mn[Fe(CN)$_6$]", color=color)
ax3.plot(T*0.0004, atoms/2048, color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)

nNa = log_data['v_nNa'][-1]

file = f'./interface_3.log'
log_data = parse_lammps_free(file)
T = np.array(log_data['Step'])
E = np.array(log_data['TotEng'])
V = np.array(log_data['Volume'])
atoms = np.array(log_data['atoms'])-28672


# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Simulation time (ps)')
ax1.set_ylabel('Total E (keV)', color=color)
ax1.plot(T*0.0004, E/1000, color=color, marker='o', markersize=3)
#ax1.vlines(
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel(r'Volume (nm$^3$)', color=color)
ax2.plot(T[10:]*0.0004, V[10:]/1000, color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 40)) 
ax3.set_ylabel(r"$x$ in Na$_x$Mn[Fe(CN)$_6$]", color=color)
ax3.plot(T*0.0004, atoms/2048, color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)

nNa = log_data['v_nNa'][-1]
#plt.grid()
#plt.title(log_file_path+f'  final nNa = {nNa}')
#plt.savefig('./production_run/free_E/test/sodiation'+'.svg', bbox_inches='tight')
plt.tight_layout()

In [ ]:
nNa, E[-10:].mean()

In [ ]:
E_full_Na = -252768.54499999998
E_No_Na = -233927.607

In [ ]:
x=2146/4096
(-243318.90300000002-(x*E_full_Na+(1-x)*E_No_Na))/(78.26*77.49)/2

In [ ]:
x=2078/4096
(-243108.454-(x*E_full_Na+(1-x)*E_No_Na))/(39.02*77.743)/2

In [ ]:
x=2102/4096
(-243165.21100000004-(x*E_full_Na+(1-x)*E_No_Na))/(39.2756*39.0552)/2

In [ ]:
E_full_Na = E[10:20].mean()
E_full_Na

In [ ]:
E_No_Na = E[10:20].mean()
E_No_Na

In [ ]:
singlecol = 3.5
plt.rcParams.update({'font.size': 8})
#plt.rcParams.update({'font.sans-serif': 'Helvetica'})
plt.rcParams.update({'lines.linewidth':1})
plt.rcParams.update({'axes.linewidth':0.5})
plt.rcParams.update({'lines.markersize':3})
plt.rcParams.update({'savefig.dpi':300})
plt.rcParams.update({'legend.fontsize': 8})

#file = f'./production_run/free_E/test/bias_1100/bias_105.log'

file = './gcmc_9.log'
log_data = parse_lammps_free(file)
T = np.array(log_data['Step'])
E = np.array(log_data['TotEng'])
V = np.array(log_data['Volume'])
atoms = np.array(log_data['atoms'])-28672

fig, axs = plt.subplots(2,1 ,figsize=(singlecol*2,singlecol*1.5), dpi=300, sharex=True)

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
#axs[0].set_xlabel('Simulation time (ps)')
axs[0].set_ylabel('Total E (keV)', color=color)
axs[0].plot(T, E/1000, color=color, marker='o', markersize=3)
axs[0].tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = axs[0].twinx()
color = 'tab:orange'
ax2.set_ylabel(r'Volume (nm$^3$)', color=color)
ax2.plot(T, V/1000, color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = axs[0].twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 40)) 
ax3.set_ylabel(r"$x$ in Na$_x$Mn[Fe(CN)$_6$]", color=color)
ax3.plot(T, atoms/2048, color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)

nNa = log_data['v_nNa'][-1]
axs[0].vlines(40000,-254,-233, linestyle='--', color='r')
axs[0].vlines(260000,-254,-233, linestyle='--', color='r')
axs[0].vlines(290000,-254,-233, linestyle='--', color='r')
axs[0].vlines(320000,-254,-233, linestyle='--', color='r')

axs[0].set_xlim(20000,340000)
axs[0].set_ylim(-254,-233)


color = 'tab:purple'
valid_data = pd.read_csv("./gcmc_9_CN.csv")
axs[1].plot(valid_data['Timestep'][11:], valid_data['Avg_Coordination'][11:], '-o', color=color,
            markersize=3, linewidth=1, alpha=0.8)
axs[1].fill_between(valid_data['Timestep'][11:], valid_data['Avg_Coordination'][11:] - valid_data['Std_Coordination'][11:],
                    valid_data['Avg_Coordination'][11:] + valid_data['Std_Coordination'][11:],
                    alpha=0.2, color=color)
axs[1].set_ylabel('Coordination number of sodium', color=color)
axs[1].tick_params(axis='y', labelcolor=color)
ax4 = axs[1].twinx()
ax4.legend(frameon=False)

df = pd.read_csv("./bond_gcmc_9.csv")
ax4.plot(df['Timestep'][df['Pair']=='N-Mn'],df['Bond_Length'][df['Pair']=='N-Mn'], label='d(N-Mn)', color='blue', marker='s', markersize=3)
ax4.plot(df['Timestep'][df['Pair']=='N-Na'],df['Bond_Length'][df['Pair']=='N-Na'], label='d(N-Na)', color='green', marker='s', markersize=3)
ax4.legend(frameon=False, loc="lower left" )
ax4.set_ylabel('Bond distance (Å)')
axs[1].set_xlabel('Simulation timesteps (0.4 fs / step)')
#plt.grid()
#plt.title(log_file_path+f'  final nNa = {nNa}')
#plt.savefig('./production_run/free_E/test/desodiation'+'.svg', bbox_inches='tight')
plt.tight_layout()
plt.savefig('./desodiation_gcmc_9'+'.svg', bbox_inches='tight')

In [ ]:
singlecol = 3.5
plt.rcParams.update({'font.size': 8})
#plt.rcParams.update({'font.sans-serif': 'Helvetica'})
plt.rcParams.update({'lines.linewidth':1})
plt.rcParams.update({'axes.linewidth':0.5})
plt.rcParams.update({'lines.markersize':3})
plt.rcParams.update({'savefig.dpi':300})
plt.rcParams.update({'legend.fontsize': 8})

#file = f'./production_run/free_E/test/bias_1100/bias_105.log'

file = './gcmc_5.log'
log_data = parse_lammps_free(file)
T = np.array(log_data['Step'])
E = np.array(log_data['TotEng'])
V = np.array(log_data['Volume'])
atoms = np.array(log_data['atoms'])-28672

fig, axs = plt.subplots(2,1 ,figsize=(singlecol*2,singlecol*1.5), dpi=300, sharex=True)

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
#axs[0].set_xlabel('Simulation time (ps)')
axs[0].set_ylabel('Total E (keV)', color=color)
axs[0].plot(T*0.0004, E/1000, color=color, marker='o', markersize=3)
axs[0].tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = axs[0].twinx()
color = 'tab:orange'
ax2.set_ylabel(r'Volume (nm$^3$)', color=color)
ax2.plot(T*0.0004, V/1000, color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = axs[0].twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 40)) 
ax3.set_ylabel(r"$x$ in Na$_x$Mn[Fe(CN)$_6$]", color=color)
ax3.plot(T*0.0004, atoms/2048, color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)

nNa = log_data['v_nNa'][-1]
axs[0].vlines(80,-254,-233, linestyle='--', color='r')
axs[0].vlines(120,-254,-233, linestyle='--', color='r')
axs[0].vlines(140,-254,-233, linestyle='--', color='r')
axs[0].vlines(8,-254,-233, linestyle='--', color='r')

axs[0].set_xlim(0,160)
axs[0].set_ylim(-254,-233)


color = 'tab:purple'
valid_data = pd.read_csv("./production_run/free_E/test/gcmc_5_CN.csv")
axs[1].plot(valid_data['Timestep']*0.0004, valid_data['Avg_Coordination'], '-o', color=color,
            markersize=3, linewidth=1, alpha=0.8)
axs[1].fill_between(valid_data['Timestep']*0.0004, valid_data['Avg_Coordination'] - valid_data['Std_Coordination'],
                    valid_data['Avg_Coordination'] + valid_data['Std_Coordination'],
                    alpha=0.2, color=color)
axs[1].set_ylabel('Coordination number of sodium', color=color)
axs[1].tick_params(axis='y', labelcolor=color)
ax4 = axs[1].twinx()
ax4.legend(frameon=False)

df = pd.read_csv("./production_run/free_E/test/bond_gcmc_5.csv")
ax4.plot(df['Timestep'][df['Pair']=='N-Mn']*0.0004,df['Bond_Length'][df['Pair']=='N-Mn'], label='d(N-Mn)', color='blue', marker='s', markersize=3)
ax4.plot(df['Timestep'][df['Pair']=='N-Na']*0.0004,df['Bond_Length'][df['Pair']=='N-Na'], label='d(N-Na)', color='green', marker='s', markersize=3)
ax4.legend(frameon=False)
ax4.set_ylabel('Bond distance (Å)')
axs[1].set_xlabel('Simulation time (ps)')
#plt.grid()
#plt.title(log_file_path+f'  final nNa = {nNa}')
#plt.savefig('./production_run/free_E/test/desodiation'+'.svg', bbox_inches='tight')
plt.tight_layout()

In [ ]:
active_pairs = ['C-Fe','C-N','N-Mn','N-Na','Fe-Mn']
df = pd.read_csv("./production_run/free_E/test/bond_gcmc_4.csv")
df

In [ ]:
df['Bond_Length'][df['Pair']=='C-Fe']

In [ ]:
bonds = ['C-Fe', 'C-N', 'N-Mn', 'N-Na', 'Fe-Mn']

plt.figure(figsize=(10, 6))

for bond in bonds:
    plt.plot(df['Timestep'][, df[bond], label=bond)  # Replace df.index with df['Time'] or df['Step'] if present


In [ ]:
#E_Na = -.13023100E+01
E_Na = -.14023100E+01

my_list = log_data['TotEng'][30:40]
E_i = sum(my_list) / len(my_list)

my_list = log_data['TotEng'][520:540]
E_f = sum(my_list) / len(my_list)

my_list = np.array(log_data['atoms'][30:40])-28672
n_Na_i = sum(my_list) / len(my_list)

my_list = np.array(log_data['atoms'][520:540])-28672
n_Na_f = sum(my_list) / len(my_list)

n_Na = n_Na_i-n_Na_f

V_ave = -(E_i-E_f-n_Na*E_Na)/n_Na

In [ ]:
T

In [ ]:
t_ls = [40000, 260000, 290000, 320000]
for i in range(4):
    step = int(t_ls[i]/500)
    print(atoms[step]/2048)

In [ ]:
V_ave

In [ ]:
V_ls = []
k = 8
for i in range(41, int(len(log_data['Step'])/k)):
    my_list = log_data['TotEng'][i-k:i]
    E_i = sum(my_list) / len(my_list)
    
    my_list = log_data['TotEng'][i:i+k]
    E_f = sum(my_list) / len(my_list)
    
    my_list = np.array(log_data['atoms'][i-k:i])-28672
    n_Na_i = sum(my_list) / len(my_list)
    
    my_list = np.array(log_data['atoms'][i:i+k])-28672
    n_Na_f = sum(my_list) / len(my_list)
    
    n_Na = n_Na_i-n_Na_f
    
    V_ls.append(-(E_i-E_f-n_Na*E_Na)/n_Na)

In [ ]:
mean(V_ls)

In [ ]:
plt.figure()
plt.plot(V_ls)

In [ ]:
E_Na = -.14023100E+01

my_list = log_data['TotEng'][10:40]
E_i = sum(my_list) / len(my_list)

my_list = log_data['TotEng'][-200:]
E_f = sum(my_list) / len(my_list)

my_list = np.array(log_data['atoms'][10:40])-28672
n_Na_i = sum(my_list) / len(my_list)

my_list = np.array(log_data['atoms'][-200:])-28672
n_Na_f = sum(my_list) / len(my_list)

n_Na = n_Na_i-n_Na_f

V_ave = -(E_i-E_f-n_Na*E_Na)/n_Na

V_ave

In [ ]:
plot_Na_profile(1100, 22, 33, -2.2, 0.02)

In [ ]:
plot_Na_profile(1100, 42, 53, -2.5, 0.02)

In [ ]:
temp, n_1, n_2, mu_1,  step = 1100, 42, 53, -2.5, 0.02

mu_ls = np.linspace(mu_1, (n_2-n_1-1)*step+mu_1, (n_2-n_1))
timestep_ls = []
num_atoms_ls = []
for i in range(n_1, n_2):
    trj_filename = f'./production_run/free_E/test/{temp}/dump.gcmc_{i+1}.trj'  # Replace with your actual file name
    timestep, num_atoms = extract_timestep_and_atom_count(trj_filename)
    timestep_ls.append(timestep)
    num_atoms_ls.append(num_atoms)
    
fig, ax1 = plt.subplots(figsize=(10, 6))
for i in range(n_1, n_2):
    nNa=-28672+num_atoms_ls[i-n_1][-1]
    ax1.plot(timestep_ls[i-n_1], -28672+num_atoms_ls[i-n_1], color=cm.jet((i-n_1)/(n_2-n_1)),
             label=f'mu_{i+1} = {round(mu_ls[i-n_1],4)}  Na_f = {nNa}')#  V = {round(V_ave,2)}')#, marker='*')
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_ls[-1]}')   
plt.legend(bbox_to_anchor=(1, 1))
plt.grid()
ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')

temp, n_1, n_2, mu_1,  step = 1100, 53, 66, -2.6, 0.02

mu_ls = np.linspace(mu_1, (n_2-n_1-1)*step+mu_1, (n_2-n_1))
timestep_ls = []
num_atoms_ls = []
for i in range(n_1, n_2):
    trj_filename = f'./production_run/free_E/test/{temp}/dump.gcmc_{i+1}.trj'  # Replace with your actual file name
    timestep, num_atoms = extract_timestep_and_atom_count(trj_filename)
    timestep_ls.append(timestep)
    num_atoms_ls.append(num_atoms)
for i in range(n_1, n_2):
    nNa=-28672+num_atoms_ls[i-n_1][-1]
    ax1.plot(timestep_ls[i-n_1], -28672+num_atoms_ls[i-n_1], color=cm.jet((i-n_1)/(n_2-n_1)),
             label=f'mu_{i+1} = {round(mu_ls[i-n_1],4)}  Na_f = {nNa}')#  V = {round(V_ave,2)}')#, marker='*')
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_ls[-1]}')   
plt.legend(bbox_to_anchor=(1, 1))

In [ ]:
for i in range(42,45):
    file = f'./production_run/free_E/test/1100/free_{i+1}.log'
    plot_log_free(file)

In [ ]:
plot_Na_profile_from_dump(1100, 53, 66, -2.6, 0.02)

In [ ]:
plot_Na_profile(1100, 67, 75, -3, 0.03)

In [ ]:
plot_Na_bias_from_dump(1100, 6, 14, -3.2, 0.2, 0.00001, 4096)

In [ ]:
plot_Na_bias_from_dump(1100, 14, 21, -10, 1, 0.00001, 4096)

In [ ]:
plot_Na_bias_from_dump(1100, 21, 26, -20, 2, 0.00001, 4096)

In [ ]:
plot_Na_bias_from_dump(1100, 26, 31, -30, 2, 0.00001, 4096)

In [ ]:
plot_Na_bias_from_dump(1100, 31, 36, -40, 2, 0.00001, 4096)

In [ ]:
plot_Na_bias_from_dump(1100, 46,48, -78, 2, 0.00001, 4096)

In [ ]:
plot_Na_bias_from_dump(1100, 37, 56, -78, 2, 0.00001, 4096)

In [ ]:
plot_Na_bias_from_dump(1100, 56, 61, -62, 2, 0.00005, 4096)

In [ ]:
plot_Na_bias_from_dump(1100, 63, 65, -62, 2, 0.00005, 4096)

In [ ]:
plot_Na_bias_from_dump(1100, 67, 7, 0, 400, 0.0002, 4096)

### mu=0.000001

In [ ]:
plot_Na_bias_from_dump(1100, 78, 89, -5, 0.3, 0.000001, 4096)

### mu=0.000001

In [ ]:
plot_Na_bias_from_dump(1100, 89, 100, -7, 0.5, 0.000005, 4096)

In [ ]:
plot_Na_bias_from_dump(1100, 100, 111, -7, 0.5, 0.0, 4096)

In [ ]:
plot_Na_profile(1100, 100, 111, -7, 0.5, 0.0, 4096)

In [ ]:
plot_Na_bias_from_dump(1100, 111, 122, -7, 0.5, 0.0000001, 4096)

In [ ]:
plot_Na_profile(1100, 111, 122, -7, 0.5, 0.0000001, 4096)

In [ ]:
plot_Na_bias_from_dump(1100, 122, 132, -7, 0.5, 0.0000005, 4096)

In [ ]:
#%matplotlib inline
plot_Na_profile_bias(1100, 122, 132, -7, 0.5, 0.0000005, 4096)

In [ ]:
#%matplotlib inline
plot_Na_profile_bias(1100, 143, 146, -2.4, 0.1, 0.0000001, 4096)

In [ ]:
#%matplotlib inline
plot_Na_profile_bias(1100, 147, 153, -1.9, 0.1, 0.0000001, 4096)

In [ ]:
plot_Na_profile_bias(1100, 163, 173, -7, 0.5, 0.0001, 4096)

In [ ]:
plot_Na_profile_bias(1100, 183, 190, -6, 0.5, 0.000, 4096)

In [ ]:
plot_Na_profile_bias(1100, 200, 210, 400, 400, 0.001, -3)

In [ ]:
plot_Na_profile_bias(1100, 210, 219, 400, 400, 0.001, -2)

In [ ]:
plot_Na_profile_bias(1100, 220, 230, 400, 400, 0.001, -3)


### Scan Nt

In [ ]:
%matplotlib widget
plot_Na_profile_bias(1100, 220, 230, 400, 400, 0.001, -3)

In [ ]:
#%matplotlib widget
plot_Na_profile_bias(1100, 240, 250, 200, 400, 0.001, -3,0)

In [ ]:
#%matplotlib widget
plot_Na_profile_bias(1100, 260, 270, 200, 400, 0.001, -3, 0)

In [ ]:
#%matplotlib widget
plot_Na_profile_bias(1100, 270, 280, 200, 400, 0.0005, -3,0)

In [ ]:
temp=1100
for i in range(228,231):
    file = f'./production_run/free_E/test/bias_{temp}/bias_{i}.log'
    plot_log_free(file)

In [ ]:
temp=1100

file = f'./production_run/free_E/test/Nt_{temp}/loop_bias_1.log'
plot_log_free(file)

In [ ]:
temp=1100

file = f'./production_run/free_E/test/Nt_{temp}/loop_bias_1.log'
plot_log_free(file)

In [ ]:
temp=1100

file = f'./production_run/free_E/test/Nt_{temp}/1/loop_bias_1.log'
plot_log_free(file)

In [ ]:
plot_Na_profile_bias(1100, 230, 240, 400, 400, 0.001, -2)

In [ ]:
temp=1100
i=114
file = f'./production_run/free_E/test/bias_{temp}/bias_{i}.log'
log_data = parse_lammps_free(file)

i=-100
x0 = log_data['v_nNa'][0]
x1 = log_data['v_nNa'][i]
y0 = log_data['TotEng'][0]
y1 = log_data['TotEng'][i]

slope = (y1 - y0) / (x1 - x0)
intercept = y0 - slope * x0

x = np.array(log_data['v_nNa'][:i])
y = np.array(log_data['TotEng'][:i])

y_linear = slope * x + intercept
difference = y - y_linear

plt.plot(log_data['v_nNa'][:i],log_data['TotEng'][:i],label='biphasic')
plt.plot([log_data['v_nNa'][0],log_data['v_nNa'][i]], [log_data['TotEng'][0],log_data['TotEng'][i]])
plt.legend()
plt.grid()
plt.xlabel('Number of Na+')
plt.ylabel('Total Energy (eV)')
plt.title(f'{file}')
plt2=plt.twinx()

plt2.plot(log_data['v_nNa'][:i],difference)
plt2.set_ylabel('delta E (eV)')

In [ ]:
temp=1100
for i in [105,115,123]:
    file = f'./production_run/free_E/test/bias_{temp}/bias_{i}.log'
    plot_log_free(file)

In [ ]:
temp=1100
for i in range(101,111):
    file = f'./production_run/free_E/test/bias_{temp}/bias_{i}.log'
    plot_log_free(file)

In [ ]:
for i in [1,2,3]:
    file = f'./production_run/free_E/test/bias_{i}.log'
    plot_log_free(file)

In [ ]:
for i in [4,5,6,7,8,9]:
    file = f'./production_run/free_E/test/bias_{i}.log'
    plot_log_free(file)

In [ ]:
for i in [47,48]:
    file = f'./production_run/free_E/test/bias_1100/bias_{i}.log'
    plot_log_free(file)

In [ ]:
for i in [105]:
    file = f'./production_run/free_E/test/bias_1100/bias_{i}.log'
    plot_log_free(file)
log_data = parse_lammps_free(file)

#E_Na = -.13023100E+01
E_Na = -.14023100E+01

my_list = log_data['TotEng'][20:40]
E_i = sum(my_list) / len(my_list)

my_list = log_data['TotEng'][-20:]
E_f = sum(my_list) / len(my_list)

my_list = np.array(log_data['atoms'][20:40])-28672
n_Na_i = sum(my_list) / len(my_list)

my_list = np.array(log_data['atoms'][-20:])-28672
n_Na_f = sum(my_list) / len(my_list)

n_Na = n_Na_i-n_Na_f

V_ave = -(E_i-E_f-n_Na*E_Na)/n_Na

In [ ]:
V_ave,n_Na

### Convergence test

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

class ATATEquilibrationDetector:
    def __init__(self, trajectory):
        """
        Initialize ATAT-inspired equilibration time detector.
        
        Parameters:
        -----------
        trajectory : array-like
            Time series data of the property of interest
        """
        self.trajectory = np.array(trajectory)
        self.num_steps = len(trajectory)
    
    def calculate_block_averages(self, max_block_size=None):
        """
        Calculate block averages to estimate equilibration time.
        
        Parameters:
        -----------
        max_block_size : int, optional
            Maximum block size to consider. If None, defaults to 1/4 of trajectory length
        
        Returns:
        --------
        dict
            Block average analysis results
        """
        if max_block_size is None:
            max_block_size = self.num_steps // 4
        
        block_variances = []
        block_sizes = []
        
        for block_size in range(1, max_block_size + 1):
            # Number of non-overlapping blocks
            num_blocks = self.num_steps // block_size
            
            # Truncate trajectory to full blocks
            truncated_traj = self.trajectory[:num_blocks * block_size]
            
            # Reshape into blocks
            blocks = truncated_traj.reshape((num_blocks, block_size))
            
            # Calculate block means
            block_means = np.mean(blocks, axis=1)
            
            # Calculate variance of block means
            block_mean_variance = np.var(block_means)
            
            block_variances.append(block_mean_variance)
            block_sizes.append(block_size)
        
        return {
            'block_sizes': block_sizes,
            'block_variances': block_variances
        }
    
    def estimate_equilibration_time(self, method='variance_plateau', threshold=0.05):
        """
        Estimate equilibration time using different methods.
        
        Parameters:
        -----------
        method : str, optional
            Method for equilibration detection
            - 'variance_plateau': Detect when block variance stabilizes
            - 'relative_variance': Use relative variance change
        threshold : float, optional
            Threshold for detecting equilibration
        
        Returns:
        --------
        dict
            Equilibration time estimation results
        """
        # Calculate block averages
        block_analysis = self.calculate_block_averages()
        block_sizes = block_analysis['block_sizes']
        block_variances = block_analysis['block_variances']
        
        if method == 'variance_plateau':
            # Find where variance stabilizes (minimal change)
            variance_changes = np.diff(block_variances)
            
            # Find where variance changes become minimal
            equilibration_index = np.argmin(np.abs(variance_changes)) + 1
            equilibration_time = block_sizes[equilibration_index]
        
        elif method == 'relative_variance':
            # Normalize variances
            normalized_variances = np.array(block_variances) / block_variances[0]
            
            # Find where relative variance change is below threshold
            equilibration_indices = np.where(normalized_variances < (1 + threshold))[0]
            
            # Take the first point where variance stabilizes
            equilibration_index = equilibration_indices[0] if len(equilibration_indices) > 0 else 0
            equilibration_time = block_sizes[equilibration_index]
        
        else:
            raise ValueError("Invalid method. Choose 'variance_plateau' or 'relative_variance'")
        
        return {
            'equilibration_time': equilibration_time,
            'block_sizes': block_sizes,
            'block_variances': block_variances,
            'method': method
        }
    
    def plot_block_analysis(self, block_analysis):
        """
        Visualize block analysis results.
        
        Parameters:
        -----------
        block_analysis : dict
            Results from estimate_equilibration_time method
        """
        plt.figure(figsize=(12, 6))
        
        # Plot block variances
        plt.subplot(2, 1, 1)
        plt.plot(block_analysis['block_sizes'], block_analysis['block_variances'], 'b-')
        plt.title('Block Variance Analysis')
        plt.xlabel('Block Size')
        plt.ylabel('Block Variance')
        
        # Highlight equilibration time
        plt.axvline(x=block_analysis['equilibration_time'], color='r', linestyle='--', 
                    label=f"Equilibration Time: {block_analysis['equilibration_time']}")
        plt.legend()
        
        # Plot original trajectory
        plt.subplot(2, 1, 2)
        plt.plot(self.trajectory)
        plt.title('Original Trajectory')
        plt.xlabel('Step')
        plt.ylabel('Value')
        plt.axvline(x=block_analysis['equilibration_time'], color='r', linestyle='--')
        
        plt.tight_layout()
        plt.show()

def analyze_equilibration(trajectory, methods=None):
    """
    Comprehensive equilibration analysis using ATAT-inspired methods.
    
    Parameters:
    -----------
    trajectory : array-like
        Time series data to analyze
    methods : list, optional
        Methods to use for equilibration detection
    
    Returns:
    --------
    dict
        Equilibration analysis results
    """
    # Create detector
    detector = ATATEquilibrationDetector(trajectory)
    
    # Default methods if not specified
    if methods is None:
        methods = ['variance_plateau', 'relative_variance']
    
    # Store results
    results = {}
    
    # Analyze using different methods
    for method in methods:
        try:
            block_analysis = detector.estimate_equilibration_time(method=method)
            results[method] = block_analysis
            
            # Visualize results
            detector.plot_block_analysis(block_analysis)
        except Exception as e:
            print(f"Error with method {method}: {e}")
    
    return results

# Example usage function
def main():
    # Example: Replace with your actual trajectory data
    import numpy as np
    
    # Simulate a trajectory with equilibration
    np.random.seed(42)
    initial_fluctuation = np.random.normal(10, 2, 200)
    equilibrated_region = np.random.normal(10, 0.5, 800)
    trajectory = np.concatenate([initial_fluctuation, equilibrated_region])
    
    # Perform equilibration analysis
    results = analyze_equilibration(trajectory)
    
    # Print detailed results
    for method, analysis in results.items():
        print(f"\nMethod: {method}")
        print(f"Estimated Equilibration Time: {analysis['equilibration_time']} steps")

# Uncomment to run in script
# if __name__ == '__main__':
#     main()

In [ ]:
temp, i = 1100, 8
file = f'./production_run/free_E/test/{temp}/free_{i}.log'

parsed_data = parse_lammps_free(file)
atoms_trajectory = parsed_data['atoms']

results = analyze_equilibration(atoms_trajectory)

# Print detailed results
for method, analysis in results.items():
    print(f"\nMethod: {method}")
    print(f"Estimated Equilibration Time: {analysis['equilibration_time']} steps")

### Convergence test 2

In [ ]:
step_2,window_size

In [ ]:
i=1
file = f'./production_run/free_E/test/Nt_1100/3/atoms_log_{i}.txt'
window_fraction=0.2
f=1

if f == 0:
    log_data = parse_lammps_free(file)
    precision=0.005*4096
    window_fraction=window_fraction
    step = np.array(log_data['Step'])
    if log_data['Step'][-1] > 5e5:
        window_size = int((step.shape[0]-30) * window_fraction)
    else:
        window_size = int((step.shape[0]-10) * window_fraction)
    n_Na = np.array(log_data['atoms'])-28672
    
elif f != 0:
    log_data = parse_print_file(file)
    precision=0.005*4096
    window_fraction=window_fraction
    step = np.array(log_data['Step'])
    window_size = int(int(step.shape[0]) * window_fraction)
    n_Na = np.array(log_data['atoms'])-28672
    
fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.plot(log_data['Step'],n_Na)    

Q_mean_ls=[]
Q_std_ls=[]
for i in range(int(1/window_fraction)):
    # Initial window
    if f == 0:
        step_1 = 9+window_size*(i+0)
        step_2 = 9+window_size*(i+1)
    elif f != 0:
        step_1 = window_size*(i+0)
        step_2 = window_size*(i+1)
    print(step_1,step_2)
    Q_mean = np.mean(n_Na[step_1:step_2])
    Q_mean_ls.append(Q_mean)
    Q_std = np.std(n_Na[step_1:step_2])
    Q_std_ls.append(Q_std)
    if step_2 >= len(log_data['Step']):
        ax1.hlines(Q_mean, log_data['Step'][step_1], log_data['Step'][-1], color='r', linestyles='--')
        ax1.vlines(log_data['Step'][-1], n_Na.min(), n_Na.max(), color='black', linestyles='--', linewidth=0.5)
    elif step_2 < len(log_data['Step']):
        ax1.hlines(Q_mean, log_data['Step'][step_1], log_data['Step'][step_2], color='r', linestyles='--')
        ax1.vlines(log_data['Step'][step_2], n_Na.min(), n_Na.max(), color='black', linestyles='--', linewidth=0.5)
    ax1.vlines(log_data['Step'][step_1], n_Na.min(), n_Na.max(), color='black', linestyles='--', linewidth=0.5)


ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')
Q_mean_ls=np.array(Q_mean_ls)
Q_std_ls=np.array(Q_std_ls)

Q_std_B1_B2_ls = []
for i in range(len(Q_std_ls)-1):
    Q_std_B1_B2_ls.append(Q_std_ls[i]+Q_std_ls[i+1])
Q_std_B1_B2_ls=np.array(Q_std_B1_B2_ls)
ax2=ax1.twinx()
color = 'tab:orange'
x=np.linspace(10000+window_size*500,log_data['Step'][-1]-window_size*500,len(Q_std_B1_B2_ls))
ax2.plot(x, Q_std_B1_B2_ls, color='orange')
ax2.scatter(x[Q_std_B1_B2_ls.argmin()], Q_std_B1_B2_ls.min(), color='orange')
ax2.set_ylabel('Std(n-1)+Std(n+1)', color=color)

Q_diff_ls = []
for i in range(len(Q_mean_ls)-1):
    Q_diff_ls.append(abs(Q_mean_ls[i+1]-Q_mean_ls[i]))

ax3 = ax1.twinx()
color = 'tab:purple'
ax3.spines['right'].set_position(('outward', 80)) 
ax3.set_ylabel('|nNa(n-1)-nNa(n+1)|', color=color)
Q_diff_ls=np.array(Q_diff_ls)
x=np.linspace(10000+window_size*500,log_data['Step'][-1] -window_size*500,len(Q_diff_ls))
ax3.plot(x, Q_diff_ls, color='purple')
ax3.scatter(x[Q_diff_ls.argmin()], Q_diff_ls.min(), color='purple')
plt.tight_layout()
plt.title(f'{file}')

In [ ]:
Q_std_B1_B2_ls

In [ ]:
Q_mean_ls

In [ ]:
np.std(n_Na)

In [ ]:
def plot_convergence(file, window_fraction=0.1, f=0):

    if f == 0:
        log_data = parse_lammps_free(file)
        precision=0.005*4096
        window_fraction=window_fraction
        step = np.array(log_data['Step'])
        if log_data['Step'][-1] > 5e5:
            window_size = int((step.shape[0]-30) * window_fraction)
        else:
            window_size = int((step.shape[0]-10) * window_fraction)
        n_Na = np.array(log_data['atoms'])-28672
        
    elif f != 0:
        log_data = parse_print_file(file)
        precision=0.005*4096
        window_fraction=window_fraction
        step = np.array(log_data['Step'])
        window_size = int(int(step.shape[0]) * window_fraction)
        n_Na = np.array(log_data['atoms'])-28672
        
    fig, ax1 = plt.subplots(figsize=(10, 4))
    ax1.plot(log_data['Step'],n_Na)    
    
    Q_mean_ls=[]
    Q_std_ls=[]
    for i in range(int(1/window_fraction)):
        # Initial window
        if f == 0:
            step_1 = 9+window_size*(i+0)
            step_2 = 9+window_size*(i+1)
        elif f != 0:
            step_1 = window_size*(i+0)
            step_2 = window_size*(i+1)
        Q_mean = np.mean(n_Na[step_1:step_2])
        Q_mean_ls.append(Q_mean)
        Q_std = np.std(n_Na[step_1:step_2])
        Q_std_ls.append(Q_std)
        if step_2 >= len(log_data['Step']):
            ax1.hlines(Q_mean, log_data['Step'][step_1], log_data['Step'][-1], color='r', linestyles='--')
            ax1.vlines(log_data['Step'][-1], n_Na.min(), n_Na.max(), color='black', linestyles='--', linewidth=0.5)
        elif step_2 < len(log_data['Step']):
            ax1.hlines(Q_mean, log_data['Step'][step_1], log_data['Step'][step_2], color='r', linestyles='--')
            ax1.vlines(log_data['Step'][step_2], n_Na.min(), n_Na.max(), color='black', linestyles='--', linewidth=0.5)
        ax1.vlines(log_data['Step'][step_1], n_Na.min(), n_Na.max(), color='black', linestyles='--', linewidth=0.5)
    
    
    ax1.set_xlabel('Timestep (0.4 fs/step)')
    ax1.set_ylabel('Number of Na+')
    Q_mean_ls=np.array(Q_mean_ls)
    Q_std_ls=np.array(Q_std_ls)
    
    Q_std_B1_B2_ls = []
    for i in range(len(Q_std_ls)-1):
        Q_std_B1_B2_ls.append(Q_std_ls[i]+Q_std_ls[i+1])
    Q_std_B1_B2_ls=np.array(Q_std_B1_B2_ls)
    ax2=ax1.twinx()
    color = 'tab:orange'
    x=np.linspace(log_data['Step'][0]+window_size*500,log_data['Step'][-1]-window_size*500,len(Q_std_B1_B2_ls))
    ax2.plot(x, Q_std_B1_B2_ls, color='orange')
    ax2.scatter(x[Q_std_B1_B2_ls.argmin()], Q_std_B1_B2_ls.min(), color='orange')
    ax2.set_ylabel('Std(n-1)+Std(n+1)', color=color)
    
    Q_diff_ls = []
    for i in range(len(Q_mean_ls)-1):
        Q_diff_ls.append(abs(Q_mean_ls[i+1]-Q_mean_ls[i]))
    
    ax3 = ax1.twinx()
    color = 'tab:purple'
    ax3.spines['right'].set_position(('outward', 80)) 
    ax3.set_ylabel('|nNa(n-1)-nNa(n+1)|', color=color)
    Q_diff_ls=np.array(Q_diff_ls)
    x=np.linspace(log_data['Step'][0]+window_size*500,log_data['Step'][-1] -window_size*500,len(Q_diff_ls))
    ax3.plot(x, Q_diff_ls, color='purple')
    ax3.scatter(x[Q_diff_ls.argmin()], Q_diff_ls.min(), color='purple')
    plt.tight_layout()
    plt.title(f'{file}')

In [ ]:
np.std(n_Na)

In [ ]:
i=18
for i in range(18):
    file = f'./production_run/free_E/test/Nt_1100/1/atoms_log_{i+1}.txt'
    plot_convergence(file, window_fraction=0.2, f=1)

In [ ]:
int(3.5)

In [ ]:
i=1
file = f'./production_run/free_E/test/atoms_log_{i}.txt'
for i in [0.1,0.2,0.25]:
    plot_convergence(file, window_fraction=i, f=1)

In [ ]:
def parse_print_file(file_path):
    data = {'Step': [], 'atoms': []}

    with open(file_path, 'r') as file:
        for line in file:
            if re.match(r"^\s*Step\s+Atoms", line):
                next(file)
                for data_line in file:
                    if re.match(r"^\s*\d+", data_line) and len(data_line.split()) == 2:
                        values = data_line.split()
                        data['Step'].append(int(values[0]))
                        data['atoms'].append(float(values[1]))
                    else:
                        break

In [ ]:
log_data

In [ ]:
fig = plt.figure()
i=1
file = f'./production_run/free_E/test/atoms_log_{i}.txt'
log_data = parse_print_file(file)
step = np.array(log_data['Step'])
n_Na = np.array(log_data['atoms']) - 3584 # Adjust baseline as needed
plt.plot(step, n_Na, label=f'Nf_{i}={n_Na[-1]}, Nt={Nt}')
#plt.plot([step[0],step[-1]], [Nt, Nt], color='black')

### LOOP

In [ ]:
import re
import numpy as np
from lammps import lammps

def parse_print_file(file_path):
    data = {'Step': [], 'atoms': []}

    with open(file_path, 'r') as file:
        for line in file:
            if re.match(r"^\s*Step\s+Atoms", line):
                next(file)
                for data_line in file:
                    if re.match(r"^\s*\d+", data_line) and len(data_line.split()) == 2:
                        values = data_line.split()
                        data['Step'].append(int(values[0]))
                        data['atoms'].append(float(values[1]))
                    else:
                        break

In [ ]:
def parse_print_file(file_path):
    """
    Parse a LAMMPS print file with Step and Atoms columns.
    
    Args:
        file_path (str): Path to the LAMMPS output file
        
    Returns:
        dict: Dictionary with 'Step' and 'atoms' arrays
    """
    data = {'Step': [], 'atoms': []}
    try:
        with open(file_path, 'r') as file:
            # Skip lines until we find the header
            for line in file:
                if re.match(r"^\s*Step\s+Atoms", line):
                    # Try to skip the next line if it's a separator
                    next_line = next(file, '').strip()
                    if not re.match(r"^\s*\d+", next_line) and len(next_line.split()) != 2:
                        continue
                    else:
                        # If it's a data line, process it
                        values = next_line.split()
                        data['Step'].append(int(values[0]))
                        data['atoms'].append(float(values[1]))
                    
                    # Process remaining data lines
                    for data_line in file:
                        data_line = data_line.strip()
                        if re.match(r"^\s*\d+", data_line) and len(data_line.split()) == 2:
                            values = data_line.split()
                            data['Step'].append(int(values[0]))
                            data['atoms'].append(float(values[1]))
                        else:
                            break
    except Exception as e:
        print(f"Error parsing file {file_path}: {e}")
    
    # Convert lists to numpy arrays for easier manipulation
    data['Step'] = np.array(data['Step'])
    data['atoms'] = np.array(data['atoms'])
    
    return data

### 300K test

In [ ]:
fig = plt.figure()
for i in range(1,11):
    Nt = 50+(i-1)*50
    file = f'./production_run/free_E/test/atoms_log_{i}.txt'
    log_data = parse_print_file(file)
    step = np.array(log_data['Step'])
    n_Na = np.array(log_data['atoms'])-3584  # Adjust baseline as needed
    plt.plot(step, n_Na, label=f'Nf_{i}={n_Na[-1]}, Nt={Nt}')
    plt.plot([step[0],step[-1]], [Nt, Nt], color='black')
    plt.vlines(step[-1], 0, 400, color='black', linestyles='--', linewidth=0.5)
plt.legend(bbox_to_anchor=(1, 1))
plt.title('300 K, mu_bias = -3, fi=0.01')
#plt.grid()

In [ ]:
temp =1100
i=1
file = f'./production_run/free_E/test/Nt_{temp}/2/{temp}_converge_{i}.txt'
data = np.loadtxt(file, skiprows=2) 
data[-1,-1]

### 1100K

In [ ]:
%matplotlib widget
fig = plt.figure()
for i in range(2,22):
    Nt = 100+(i-1)*50
    file = f'./production_run/free_E/test/Nt_1100/1/atoms_log_{i}.txt'
    log_data = parse_print_file(file)
    step = np.array(log_data['Step'])
    n_Na = np.array(log_data['atoms']) - 28672  # Adjust baseline as needed
    plt.plot(step, n_Na, label=f'Nf_{i}={n_Na[-1]}, Nt={Nt}')
    plt.plot([step[0],step[-1]], [Nt, Nt], color='black')
    plt.vlines(step[-1], 0, 1000, color='black', linestyles='--', linewidth=0.5)
plt.legend(bbox_to_anchor=(1, 1))
plt.title('1100 K mu_bias = -3')
plt.grid()

In [ ]:
fig = plt.figure()
for i in range(1,12):
    Nt = 150+(i-1)*50
    file = f'./production_run/free_E/test/Nt_1000/1/atoms_log_{i}.txt'
    log_data = parse_print_file(file)
    step = np.array(log_data['Step'])
    n_Na = np.array(log_data['atoms']) - 28672  # Adjust baseline as needed
    plt.plot(step, n_Na, label=f'Nf_{i}={n_Na[-1]}, Nt={Nt}')
    plt.plot([step[0],step[-1]], [Nt, Nt], color='black')
    #plt.vlines(step[-1], 0, 1000, color='black', linestyles='--', linewidth=0.5)
plt.legend(bbox_to_anchor=(1, 1))
plt.title('1000 K mu_bias = -3')
plt.grid()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,5))
plt.subplots_adjust(wspace=0.35)
temp = 1100
fi = 0.002
for i in range(1,15):
    Nt = 4000-(i-1)*50
    file = f'./production_run/free_E/test/Nt_{temp}/2/atoms_log_{i}.txt'
    log_data = parse_print_file(file)
    step = np.array(log_data['Step'])
    n_Na = np.array(log_data['atoms']) - 28672  # Adjust baseline as needed
    ax1.plot(step, n_Na, label=f'Nf_{i}={n_Na[-1]}, Nt={Nt}')
    #ax1.plot([step[0],step[-1]], [Nt, Nt], color='black')
    #plt.vlines(step[-1], 0, 1000, color='black', linestyles='--', linewidth=0.5)
    
    file = f'./production_run/free_E/test/Nt_{temp}/2/{temp}_converge_{i}.txt'
    data = np.loadtxt(file, skiprows=2) 
    ts = data[-1,0] 
    Nf = data[-1,-1] - 28672
    mu = -2-2*fi*(Nf-Nt)
    ax1.scatter(ts,Nf)
    ax1.text(ts,Nf-5,s=f'{Nf}',fontsize=8)
    ax2.scatter(Nf,mu)
    
ax1.legend(bbox_to_anchor=(0,1))
plt.title(f'{temp} K mu_bias = -3')
ax1.grid()
ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')
ax2.set_xlabel('Number of Na+')
ax2.set_ylabel('Chemical potential (eV)')


In [ ]:
fig = plt.figure()
for i in range(1,3):
    Nt = 4000-(i-1)*50
    file = f'./production_run/free_E/test/Nt_1000/2/atoms_log_{i}.txt'
    log_data = parse_print_file(file)
    step = np.array(log_data['Step'])
    n_Na = np.array(log_data['atoms']) - 28672  # Adjust baseline as needed
    plt.plot(step, n_Na, label=f'Nf_{i}={n_Na[-1]}, Nt={Nt}')
    #plt.plot([step[0],step[-1]], [Nt, Nt], color='black')
    #plt.vlines(step[-1], 0, 1000, color='black', linestyles='--', linewidth=0.5)
plt.legend(bbox_to_anchor=(1, 1))
plt.title('1000 K mu_bias = -3')
plt.grid()

In [ ]:
temp,serie,i=900,1,61
file = f'./production_run/free_E/test/Nt_{temp}/{serie}/{temp}_converge_{i}.txt'
data = np.loadtxt(file, skiprows=2) 
data.shape

In [ ]:
def plot_loop(temp, Nt0, mu_b, fi, serie, curr_N):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,5))
    plt.subplots_adjust(wspace=0.35)
    temp = temp
    fi = fi
    Nf_ls = []
    mu_ls = []
    for i in range(1,curr_N):
        Nt = Nt0+(i-1)*50
        file = f'./production_run/free_E/test/Nt_{temp}/{serie}/atoms_log_{i}.txt'
        log_data = parse_print_file(file)
        step = np.array(log_data['Step'])
        n_Na = np.array(log_data['atoms']) - 28672  # Adjust baseline as needed
        ax1.plot(step, n_Na)#, label=f'Nf_{i}={n_Na[-1]}, Nt={Nt}')
        #ax1.plot([step[0],step[-1]], [Nt, Nt], color='black')
        #plt.vlines(step[-1], 0, 1000, color='black', linestyles='--', linewidth=0.5)
        
        file = f'./production_run/free_E/test/Nt_{temp}/{serie}/{temp}_converge_{i}.txt'
        data = np.loadtxt(file, skiprows=2) 
        if data.ndim == 1:
            # Handle 1D case - adjust indexing as needed
            ts = data[0] if len(data) > 0 else 0
            Nf = data[-1] - 28672 if len(data) > 0 else 0
        else:
            # Handle 2D case as before
            ts = data[-1,0] 
            Nf = data[-1,-1] - 28672
        mu = mu_b-2*fi*(Nf-Nt)
        Nf_ls.append(Nf)
        mu_ls.append(mu)
        ax1.scatter(ts,Nf)
        ax1.text(ts,Nf-5,s=f'{Nf}',fontsize=8)
        ax2.scatter(Nf,mu)
    ax2.plot(Nf_ls,mu_ls)
        
    #ax1.legend(bbox_to_anchor=(0,1))
    plt.title(f'{temp} K mu_bias = -3')
    ax1.grid()
    ax1.set_xlabel('Timestep (0.4 fs/step)')
    ax1.set_ylabel('Number of Na+')
    ax2.set_xlabel('Number of Na+')
    ax2.set_ylabel('Chemical potential (eV)')
    plt.tight_layout()
    plt.savefig(f'./production_run/free_E/test/loop_{temp}.png')
    return Nf_ls, mu_ls

### 1100K

In [ ]:
%matplotlib widget
Nf_ls_1100, mu_ls_1100 = plot_loop(1100, 150, -3, 0.002, 1, 22)
Nf_ls_1100_2, mu_ls_1100_2 = plot_loop(1100, 150, -3, 0.002, 1, 13)
Nf_ls_1100_3, mu_ls_1100_3 = plot_loop(1100, 150, -3, 0.001, 3, 46)

In [ ]:
plt.figure()
plt.plot(Nf_ls_1100[1:], mu_ls_1100[1:])
plt.plot(Nf_ls_1100_2[1:], mu_ls_1100_2[1:])
plt.plot(Nf_ls_1100_3[1:], mu_ls_1100_3[1:])

### 1000K

In [ ]:
%matplotlib inline
Nf_ls_1000, mu_ls_1000 = plot_loop(1000, 150,-3, 0.002, 1, 79)
Nf_ls_1000_2, mu_ls_1000_2 = plot_loop(1000, 150,-3, 0.002, 2, 23)
Nf_ls_1000_3, mu_ls_1000_3 = plot_loop(1000, 150,-3, 0.002, 3, 51)

In [ ]:
plt.figure()
plt.plot(Nf_ls_1000, mu_ls_1000)
plt.plot(Nf_ls_1000_2, mu_ls_1000_2)
plt.plot(Nf_ls_1000_3, mu_ls_1000_3)

### 900K

In [ ]:
Nf_ls_900, mu_ls_900 = plot_loop(900, 150,-3, 0.002, 1, 80)
Nf_ls_900_2, mu_ls_900_2 = plot_loop(900, 150,-3, 0.002, 2, 23)
Nf_ls_900_3, mu_ls_900_3 = plot_loop(900, 150,-3, 0.002, 3, 37)

In [ ]:
plt.figure()
plt.plot(Nf_ls_900[1:], mu_ls_900[1:])
plt.plot(Nf_ls_900_2[1:], mu_ls_900_2[1:])
plt.plot(Nf_ls_900_3[1:], mu_ls_900_3[1:])

### 800K

In [ ]:
Nf_ls_800, mu_ls_800 = plot_loop(800, 200, -3, 0.002, 1, 82)
Nf_ls_800_2, mu_ls_800_2 = plot_loop(800, 200, -3, 0.002, 2, 58)
Nf_ls_800_3, mu_ls_800_3 = plot_loop(800, 200, -2.5, 0.002, 3, 64)
Nf_ls_800_4, mu_ls_800_4 = plot_loop(800, 200, -3, 0.002, 4, 60)

In [ ]:
plt.figure()
plt.plot(Nf_ls_800, mu_ls_800)
Nf_ls_800 = np.array(Nf_ls_800)
mu_ls_800 = np.array(mu_ls_800)
plt.plot(Nf_ls_800_2[:20], mu_ls_800_2[:20])
plt.plot(Nf_ls_800_2[20:], np.array(mu_ls_800_2[20:])+0.5)
plt.plot(Nf_ls_800_3, mu_ls_800_3)
plt.plot(Nf_ls_800_4, mu_ls_800_4)

### 700K

In [ ]:
#Nf_ls_700, mu_ls_700 = plot_loop(700, 150, -2.5, 0.002, 1, 80)
#Nf_ls_700_2, mu_ls_700_2 = plot_loop(700, 200, -2.5, 0.002, 2, 52)
#Nf_ls_700_3, mu_ls_700_3 = plot_loop(700, 200, -2, 0.002, 3, 39)
#Nf_ls_700_4, mu_ls_700_4 = plot_loop(700, 200, -2, 0.002, 4, 82)
Nf_ls_700_5, mu_ls_700_5 = plot_loop(700, 200, -2, 0.002, 50, 5, 82)
Nf_ls_700_6, mu_ls_700_6 = plot_loop(700, 200, -2, 0.002, 50, 6, 55)
Nf_ls_700_7, mu_ls_700_7 = plot_loop(700, 175, -2, 0.002, 50, 7, 82)
#(temp, Nt0, mu_b, fi, step_size, serie, curr_N, vac=False):

In [ ]:
plt.figure()
plt.plot(Nf_ls_700_5, mu_ls_700_5)
plt.plot(Nf_ls_700_6, mu_ls_700_6)
plt.plot(Nf_ls_700_7, mu_ls_700_7)

### 600K

In [ ]:
#Nf_ls_600, mu_ls_600 = plot_loop(600, 200, -2, 0.002, 1, 82)
#Nf_ls_600_2, mu_ls_600_2 = plot_loop(600, 200, -2, 0.002, 2, 82)
#Nf_ls_600_3, mu_ls_600_3 = plot_loop(600, 200, -2, 0.002, 3, 73)
#Nf_ls_600_4, mu_ls_600_4 = plot_loop(600, 200, -2, 0.002, 4, 82)
Nf_ls_600_5, mu_ls_600_5 = plot_loop(600, 200, -2, 0.002, 50, 5, 82)
Nf_ls_600_6, mu_ls_600_6 = plot_loop(600, 175, -2, 0.002, 50, 6, 82)

In [ ]:
plt.figure()
#plt.plot(Nf_ls_600[1:], mu_ls_600[1:])
#plt.plot(Nf_ls_600_2, mu_ls_600_2)
plt.plot(Nf_ls_600_3, mu_ls_600_3)
plt.plot(Nf_ls_600_4, mu_ls_600_4)
plt.plot(Nf_ls_600_5, mu_ls_600_5)

In [ ]:
temp, dir = 600, 5
for i in [8]:
    index = i
    plot_log_free(f'./production_run/free_E/test/Nt_{temp}/{dir}/loop_bias_{dir}.log_{index}')

In [ ]:
temp, dir = 600, 2
for i in range(1,2):
    index = i
    plot_log_free(f'./production_run/free_E/test/Nt_{temp}/{dir}/loop_bias_{dir}.log_{index}')

### 500K

In [ ]:
#Nf_ls_500, mu_ls_500 = plot_loop(500, 200, -2, 0.002, 1, 61)
#Nf_ls_500_2, mu_ls_500_2 = plot_loop(500, 200, -2, 0.002, 2, 49)
#Nf_ls_500_3, mu_ls_500_3 = plot_loop(500, 200, -2, 0.002, 3, 60)
#Nf_ls_500_4, mu_ls_500_4 = plot_loop(500, 200, -2, 0.002, 4, 82)
#Nf_ls_500_5, mu_ls_500_5 = plot_loop(500, 200, -2, 0.002, 5, 82)
#Nf_ls_500_6, mu_ls_500_6 = plot_loop(500, 200, -2, 0.002, 6, 82)
Nf_ls_500_7, mu_ls_500_7 = plot_loop(500, 175, -2, 0.002, 50, 7, 82)
Nf_ls_500_8, mu_ls_500_8 = plot_loop(500, 175, -2, 0.002, 50, 8, 82)

In [ ]:
Nf_ls_500_7, mu_ls_500_7 = plot_loop(500, 175, -2, 0.002, 50, 7, 82)
Nf_ls_500_8, mu_ls_500_8 = plot_loop(500, 175, -2, 0.002, 50, 8, 82)
#(400, 175, -2, 0.002, 50, 6, 82)

In [ ]:
plt.figure()
plt.plot(Nf_ls_500_7, mu_ls_500_7)
plt.plot(Nf_ls_500_8, mu_ls_500_8)
plt.ylabel('Chemical potential (eV)')
plt.xlabel('Number of Na')
plt.title('500K')
plt.tight_layout()

In [ ]:
plt.figure()
plt.plot(Nf_ls_500_4, mu_ls_500_4)
plt.plot(Nf_ls_500_5, mu_ls_500_5)
plt.ylabel('Chemical potential (eV)')
plt.xlabel('Number of Na')
plt.title('500K')
plt.tight_layout()

In [ ]:
temp, dir = 500, 4
for i in [54]:
    index = i
    plot_log_free(f'./production_run/free_E/test/Nt_{temp}/{dir}/loop_bias_{dir}.log_{index}')

### 400K

#### Plot

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(singlecol*2,singlecol*1), dpi=300)
plt.subplots_adjust(wspace=0.35)
temp = temp
fi = 0.002
Nf_ls = []
mu_ls = []

temp = 400
for i in range(1, 83):
    
    temp, serie = 400, 1
    file_1 = f'./production_run/free_E/test/Nt_{temp}/{serie}/atoms_log_{i}.txt'
    log_data_1 = parse_print_file(file_1)
    step_1 = np.array(log_data_1['Step'])
    n_Na_1 = np.array(log_data_1['atoms']) - 28672  # Adjust baseline as needed
    
    temp, serie = 400, 4
    file_2 = f'./production_run/free_E/test/Nt_{temp}/{serie}/atoms_log_{i}.txt'
    log_data_2 = parse_print_file(file_2)
    step_2 = np.array(log_data_2['Step'])
    n_Na_2 = np.array(log_data_2['atoms']) - 28672
    
    temp, serie = 400, 5
    file_3 = f'./production_run/free_E/test/Nt_{temp}/{serie}/atoms_log_{i}.txt'
    log_data_3 = parse_print_file(file_3)
    step_3 = np.array(log_data_3['Step'])
    n_Na_3 = np.array(log_data_3['atoms']) - 28672
    
    n_Na = np.concatenate([n_Na_1, n_Na_2, n_Na_3])
    print(f'Data number {i}')
    ax1.plot(n_Na/2048, range(0,2000*n_Na.shape[0],2000),
             color='tab:green')
    ax1.plot(np.concatenate([n_Na_1, n_Na_2])/2048, range(0,2000*np.concatenate([n_Na_1, n_Na_2]).shape[0],2000),
             color='tab:orange')
    ax1.plot(n_Na_1/2048, range(0,2000*n_Na_1.shape[0],2000),
             color='tab:blue')
    
 
ax2.plot(np.array(Nf_ls_400)/2048, mu_ls_400, label='initial')
ax2.plot(np.array(Nf_ls_400_4)/2048, mu_ls_400_4, label='1st resample')
ax2.plot(np.array(Nf_ls_400_5)/2048, mu_ls_400_5, label='2nd resample')
ax2.legend()

# Set plot properties
ax1.grid()
ax1.set_ylabel('Number of GCMC moves')
ax1.set_xlabel(r"$x$ in Na$_x$Mn[Fe(CN)$_6$]")
ax2.set_xlabel(r"$x$ in Na$_x$Mn[Fe(CN)$_6$]")
ax2.set_ylabel('Chemical potential (eV)')
plt.tight_layout()
# Save figure
#plt.savefig(f'./production_run/free_E/test/loop_{temp}.png')

In [ ]:
np.concatenate([n_Na_1, n_Na_2]).shape[0]

In [ ]:
#Nf_ls_400, mu_ls_400 = plot_loop(400, 200, -2, 0.002, 1, 82)
#Nf_ls_400_2, mu_ls_400_2 = plot_loop(400, 200, -2, 0.002, 2, 82)
#Nf_ls_400_3, mu_ls_400_3 = plot_loop(400, 200, -2, 0.002, 3, 82)
Nf_ls_400_4, mu_ls_400_4 = plot_loop(400, 200, -2, 0.002, 4, 82)
Nf_ls_400_5, mu_ls_400_5 = plot_loop(400, 200, -2, 0.002, 5, 82)
Nf_ls_400_6, mu_ls_400_6 = plot_loop(400, 175, -2, 0.002, 6, 82)

In [ ]:
Nf_ls_400_6, mu_ls_400_6 = plot_loop(400, 175, -2, 0.002, 50, 6, 82)
Nf_ls_400_7, mu_ls_400_7 = plot_loop(400, 175, -2, 0.002, 50, 7, 82)


In [ ]:
plt.figure()
plt.plot(Nf_ls_400_6, mu_ls_400_6)
plt.plot(Nf_ls_400_7, mu_ls_400_7)

### Convergence test

In [ ]:
conver_Na = []
converged = []
for i in range(1,76):
    temp, serie = 400, 1
    file_1 = f'./production_run/free_E/test/Nt_{temp}/{serie}/atoms_log_{i}.txt'
    log_data_1 = parse_print_file(file_1)
    step_1 = np.array(log_data_1['Step'])
    n_Na_1 = np.array(log_data_1['atoms']) - 28672  # Adjust baseline as needed
    
    temp, serie = 400, 4
    file_2 = f'./production_run/free_E/test/Nt_{temp}/{serie}/atoms_log_{i}.txt'
    log_data_2 = parse_print_file(file_2)
    step_2 = np.array(log_data_2['Step'])
    n_Na_2 = np.array(log_data_2['atoms']) - 28672

    temp, serie = 400, 5
    file_3 = f'./production_run/free_E/test/Nt_{temp}/{serie}/atoms_log_{i}.txt'
    log_data_3 = parse_print_file(file_3)
    step_3 = np.array(log_data_3['Step'])
    n_Na_3 = np.array(log_data_3['atoms']) - 28672
    
    n_Na = np.concatenate([n_Na_1, n_Na_2, n_Na_3])
    print(f'Data number {i}')
    result = pymser.equilibrate(n_Na, LLM=False, batch_size=1, ADF_test=False, uncertainty='uSD', print_results=True)
    conver_Na.append(result['average'])

    if result['equilibrated'].shape[0] > 10:
        converged.append(i-1)

In [ ]:
mu_b = -2
fi = 0.002
mu_ls=[]
conver_Na = np.array(conver_Na)
converged = np.array(converged)
for i in converged:
    Nt = 200+50*i
    mu_ls.append(mu_b-2*fi*(conver_Na[i]-Nt))

In [ ]:
converged.shape

In [ ]:
plt.figure()
plt.plot(conver_Na[converged], mu_ls)
#plt.plot(conver_Na, mu)
plt.plot(Nf_ls_400_5, mu_ls_400_5)

In [ ]:
i=35
temp, serie = 400, 1
file_1 = f'./production_run/free_E/test/Nt_{temp}/{serie}/atoms_log_{i}.txt'
log_data_1 = parse_print_file(file_1)
step_1 = np.array(log_data_1['Step'])
n_Na_1 = np.array(log_data_1['atoms']) - 28672  # Adjust baseline as needed

temp, serie = 400, 4
file_2 = f'./production_run/free_E/test/Nt_{temp}/{serie}/atoms_log_{i}.txt'
log_data_2 = parse_print_file(file_2)
step_2 = np.array(log_data_2['Step'])
n_Na_2 = np.array(log_data_2['atoms']) - 28672

temp, serie = 400, 5
file_3 = f'./production_run/free_E/test/Nt_{temp}/{serie}/atoms_log_{i}.txt'
log_data_3 = parse_print_file(file_3)
step_3 = np.array(log_data_3['Step'])
n_Na_3 = np.array(log_data_3['atoms']) - 28672

n_Na = np.concatenate([n_Na_1, n_Na_2, n_Na_3])
print(f'Data number {i}')
plt.figure(figsize=(singlecol*1.3,singlecol*1), dpi=300)
plt.plot(n_Na, color='tab:green')
plt.plot(np.concatenate([n_Na_1, n_Na_2]), color='tab:orange')
plt.plot(n_Na_1, color='tab:blue')

plt.ylabel('Number of sodium')
plt.xlabel('Number of sampling')
plt.tight_layout()

In [ ]:
plt.figure(figsize=(singlecol*1.3,singlecol*1), dpi=200)
error = 0.002*2*5
#plt.plot(np.array(Nf_ls_400)/2048, mu_ls_400, label='initial')
plt.plot(np.array(Nf_ls_400_4)/2048, mu_ls_400_4, label='1st resample')
plt.plot(np.array(Nf_ls_400_5)/2048, mu_ls_400_5, label='2nd resample')
plt.plot(np.array(Nf_ls_400_6)/2048, mu_ls_400_6, label='2*nd resample')
plt.legend()
plt.ylabel('Chemical potential (eV)')
plt.xlabel(r"$x$ in Na$_x$Mn[Fe(CN)$_6$]")
plt.title('400K')
plt.tight_layout()

In [ ]:
result = pymser.equilibrate(n_Na_1, LLM=False, batch_size=1,
                            ADF_test=True, uncertainty='uSD', print_results=True)

In [ ]:
result = pymser.equilibrate(np.concatenate([n_Na_1, n_Na_2]), LLM=False, batch_size=1, ADF_test=True, uncertainty='uSD', print_results=True)

In [ ]:
result = pymser.equilibrate(n_Na, LLM=False, batch_size=1, ADF_test=True, uncertainty='uSD', print_results=True)

In [ ]:
result

In [ ]:
result['average']

In [ ]:
result['equilibrated'].shape[0]

In [ ]:
acf_values

In [ ]:
acf_values = acf(n_Na, nlags=len(n_Na)//2, fft=True)
acf_values
#plt.figure(figsize=(10, 5))
plt.xlabel('Lag (Timesteps)')
plt.ylabel('Autocorrelation')
plot_acf(n_Na, lags=50, title='Autocorrelation Function of Sodium Count (Production Data)') [6]
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pymser import mser  # For automated burn-in detection [1, 2, 3, 4]
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import acf

# --- 1. Load your data ---
# Assuming your data is in a CSV file with columns 'timestep' and 'sodium_count'
# Replace 'your_simulation_data.csv' with the actual path to your data file.
try:
    df = pd.read_csv('your_simulation_data.csv')
    timesteps = df['timestep'].values
    sodium_count = df['sodium_count'].values
except FileNotFoundError:
    print("Error: 'your_simulation_data.csv' not found. Creating dummy data for demonstration.")
    # Create dummy data for demonstration if file not found
    total_steps = 100000
    timesteps = np.arange(total_steps)
    # Simulate an equilibration period followed by a stable region
    equilibration_steps = 20000
    initial_fluctuation = np.sin(np.linspace(0, 10 * np.pi, equilibration_steps)) * 50 + 100
    stable_fluctuation = np.random.normal(loc=250, scale=10, size=total_steps - equilibration_steps)
    sodium_count = np.concatenate((initial_fluctuation, stable_fluctuation))
    # Add some noise to the initial part to make MSER more realistic
    sodium_count[:equilibration_steps] += np.random.normal(0, 20, equilibration_steps)
    sodium_count = np.maximum(0, sodium_count).astype(int) # Ensure non-negative counts

print(f"Loaded data with {len(timesteps)} timesteps.")

# --- 2. Automated Burn-in Detection using pyMSER ---
# The Marginal Standard Error Rule (MSER) helps objectively identify the equilibration period.[1, 2, 3, 4]
# It finds the point where the statistical uncertainty of the mean is minimized.[2]

# MSER requires a 1D numpy array of the observable (sodium_count in this case)
# The 'max_block_size_ratio' parameter can be adjusted based on your data length.
# A common practice is to use a fraction of the total data length.
# For GCMC, MSER can significantly reduce computational cost by identifying the optimal truncation point.[1, 2]
equilibration_index = mser.detect_equilibration(sodium_count, max_block_size_ratio=0.5)

print(f"\nAutomated burn-in detection suggests equilibration at index: {equilibration_index}")
print(f"This means the first {equilibration_index} data points should be discarded.")

# Plot the raw data and the detected equilibration point
plt.figure(figsize=(12, 6))
plt.plot(timesteps, sodium_count, label='Instantaneous Sodium Count')
plt.axvline(x=timesteps[equilibration_index], color='r', linestyle='--', label='Equilibration Point (MSER)')
plt.title('Sodium Count Time Series with MSER Equilibration Point')
plt.xlabel('Timestep')
plt.ylabel('Sodium Count')
plt.legend()
plt.grid(True)
plt.show()

# Truncate the data to keep only the production (equilibrated) phase
production_sodium_count = sodium_count[equilibration_index:]
production_timesteps = timesteps[equilibration_index:]
print(f"Production data length: {len(production_sodium_count)} points.")

# --- 3. Autocorrelation Analysis ---
# Autocorrelation quantifies the correlation between samples at different time lags.[5, 6]
# The integrated autocorrelation time (tau) indicates how many raw steps are needed for one independent sample.[7]

# Calculate the Autocorrelation Function (ACF) for the production data
# `nlags` determines the maximum lag to compute the ACF for.
# `fft=True` can speed up computation for long series.
acf_values = acf(production_sodium_count, nlags=len(production_sodium_count)//2, fft=True) [6]

# Plot the Autocorrelation Function
plt.figure(figsize=(10, 5))
plot_acf(production_sodium_count, lags=50, title='Autocorrelation Function of Sodium Count (Production Data)') [6]
plt.xlabel('Lag (Timesteps)')
plt.ylabel('Autocorrelation')
plt.grid(True)
plt.show()

# Estimate the integrated autocorrelation time (tau)
# A common way to estimate tau is by summing the ACF values until they decay to near zero.
# statsmodels' `acf` function can also return the confidence intervals, but for tau estimation,
# we often sum the positive part of the ACF or use a more sophisticated method.
# For simplicity, we'll sum the ACF values until they become negative or very small.
# A more robust method would involve using a statistical package that directly calculates integrated autocorrelation time.
# For demonstration, we'll use a simple sum of positive values.
tau_estimate = 0.5 * (1 + 2 * np.sum(acf_values[1:])) # Formula for integrated autocorrelation time [7]
# A more common and robust way to calculate tau is to sum the ACF until it drops below a certain threshold or becomes negative.
# Let's refine the tau calculation for better accuracy, summing until the ACF is negligible.
# This is a heuristic; for rigorous analysis, consider specialized libraries or methods.
integrated_autocorrelation_time = 0
for i in range(1, len(acf_values)):
    if acf_values[i] > 0.05: # Sum as long as ACF is significantly positive (threshold can be adjusted)
        integrated_autocorrelation_time += acf_values[i]
    else:
        break
integrated_autocorrelation_time = 1 + 2 * integrated_autocorrelation_time # [7]

print(f"\nEstimated Integrated Autocorrelation Time (τ): {integrated_autocorrelation_time:.2f} timesteps")

# --- 4. Effective Sample Size Calculation ---
# The effective sample size (N_eff) accounts for correlations in the data.[7]
# N_eff = N / (1 + 2 * tau) [7] where N is the total number of data points in the production run.

total_production_samples = len(production_sodium_count)
effective_sample_size = total_production_samples / integrated_autocorrelation_time [7]

print(f"Total number of production samples (N): {total_production_samples}")
print(f"Effective Sample Size (N_eff): {effective_sample_size:.2f}")

if effective_sample_size < 100: # A common heuristic for sufficient samples
    print("\nWarning: Effective sample size is relatively low. Consider running longer simulations for better statistics.")
else:
    print("\nEffective sample size appears sufficient for statistical analysis.")

# --- 5. Block Averaging (Optional, for mean and error estimation) ---
# Block averaging is a robust technique for estimating the mean and its statistical uncertainty from correlated data.[8, 9]
# The block size should be greater than the autocorrelation time.[8]

block_size = int(np.ceil(integrated_autocorrelation_time))
if block_size == 0: # Avoid division by zero if tau is very small
    block_size = 1
num_blocks = total_production_samples // block_size

if num_blocks > 1:
    block_averages = [np.mean(production_sodium_count[i * block_size:(i + 1) * block_size]) for i in range(num_blocks)]
    mean_sodium_count = np.mean(block_averages)
    std_err_of_mean = np.std(block_averages, ddof=1) / np.sqrt(num_blocks) # Standard error of the mean

    print(f"\n--- Block Averaging Results (Block Size: {block_size}) ---")
    print(f"Mean Sodium Count (from production data): {mean_sodium_count:.2f}")
    print(f"Standard Error of the Mean: {std_err_of_mean:.2f}")
else:
    print("\nNot enough data or block size too large for meaningful block averaging.")
    print(f"Mean Sodium Count (from production data): {np.mean(production_sodium_count):.2f}")

In [ ]:
plt.figure()
error = 0.002*2*5
plt.plot(Nf_ls_400, mu_ls_400, label='initial')
#plt.fill_between(Nf_ls_400, np.array(mu_ls_400) - error,  np.array(mu_ls_400) + error,
#                 alpha=0.15, label='Error range')
#plt.plot(Nf_ls_400_2, mu_ls_400_2, label='2')
#plt.fill_between(Nf_ls_400_2, np.array(mu_ls_400_2) - error,  np.array(mu_ls_400_2) + error,
#                 alpha=0.15, label='Error range')
#plt.plot(Nf_ls_400_3, mu_ls_400_3, label='3')
#plt.fill_between(Nf_ls_400_3, np.array(mu_ls_400_3) - error,  np.array(mu_ls_400_3) + error,
#                 alpha=0.15, label='Error range')
plt.plot(Nf_ls_400_4, mu_ls_400_4, label='1st resample')
plt.plot(Nf_ls_400_5, mu_ls_400_5, label='2nd resample')
plt.legend()
plt.ylabel('Chemical potential (eV)')
plt.xlabel('Number of Na')
plt.title('400K')
plt.tight_layout()

In [ ]:
temp, dir = 400, 4
for i in range(60,80,4):
    index = i
    plot_log_free(f'./production_run/free_E/test/Nt_{temp}/{dir}/loop_bias_{dir}.log_{index}')

In [ ]:
temp, dir = 400, 5
for i in [11]:
    index = i
    plot_log_free(f'./production_run/free_E/test/Nt_{temp}/{dir}/loop_bias_{dir}.log_{index}')

In [ ]:
log_data = parse_lammps_free(f'./production_run/free_E/test/Nt_{temp}/{dir}/loop_bias_{dir}.log_{index}')
atoms = np.array(log_data['atoms'])-28672
atoms[40:].mean()

### Gibbs Free Energy !!!!!!!!

In [ ]:
plt.rcParams.update({
    'font.size': 8,
    'lines.linewidth': 1,
    'axes.linewidth': 0.5,
    'lines.markersize': 3,
    'savefig.dpi': 300,
    'legend.fontsize': 8
})
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

def linear_function(x):
    return slope * x + intercept

import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate, optimize
import sympy as sp
import warnings
warnings.filterwarnings('ignore')

def derivative_equation(x):
    return polynomial(x) - linear_slope
    
def polynomial(x):
    return np.polyval(coefficients, x)

def integrate_numerically():
    result, error = integrate.quad(polynomial, x0, x_end)
    return result, error

def integrate_analytically():
    x = sp.Symbol('x')
    expr = sum(c * x**i for i, c in enumerate(reversed(coefficients)))
    antiderivative = sp.integrate(expr, x)
    result = float(antiderivative.subs(x, x_end) - antiderivative.subs(x, x0))
    return result, antiderivative

def integrate_manually():
    powers = list(range(8, -1, -1))
    integrated_coeffs = [(coef / (p + 1), p + 1) for coef, p in zip(coefficients, powers)]
    
    def F(x):
        return sum(c * x**p for c, p in integrated_coeffs)

    result = F(x_end) - F(x0)
    return result, integrated_coeffs
    

In [ ]:
# Your data fitting
plt.rcParams.update({'font.size': 16})
plt.rcParams.update({'legend.fontsize': 10})

for temp in range(300,600,100):

#    if temp == 700:
#        x = np.concatenate([Nf_ls_700_4, Nf_ls_700_5])
#        y = np.concatenate([mu_ls_700_4, mu_ls_700_5])
#        min_x1, min_y1 = 264.2250,  -235067.4613
#        min_x2, min_y2 = 3265.4322,  -240782.9127
#        degree = 7
#        
#    if temp == 600:
#        x = np.concatenate([Nf_ls_600_4, Nf_ls_600_5])
#        y = np.concatenate([mu_ls_600_4, mu_ls_600_5])
#        min_x1, min_y1 = 148.4986,  -234824.9286
#        min_x2, min_y2 = 3411.3704,  -240778.0352
#        degree = 7
        
    if temp == 500:
        x = np.concatenate([Nf_ls_500_8, Nf_ls_500_8])
        y = np.concatenate([mu_ls_500_8, mu_ls_500_8])
        min_x1, min_y1 = 0,  -234542.5987
        min_x2, min_y2 = 3796.9546,  -241069.6956
        degree = 9
    
    if temp == 400:
        x = np.concatenate([Nf_ls_400_7, Nf_ls_400_7])
        y = np.concatenate([mu_ls_400_7, mu_ls_400_7])
        min_x1, min_y1 = 0,  -234542.5987
        min_x2, min_y2 = 3802.0753, -240968.3298

        degree = 9
    
    if temp == 300:
        x = np.concatenate([Nf_ls_300_9, Nf_ls_300_9])
        y = np.concatenate([mu_ls_300_9, mu_ls_300_9])
        min_x1, min_y1 = 0, -234542.5987
        min_x2, min_y2 = 4096, -241440.8180
        degree = 9
    
    sort_indices = np.argsort(x)
    x = x[sort_indices]
    y = y[sort_indices]
    coefficients = np.polyfit(x, y, degree)
    poly_func = np.poly1d(coefficients)
    
    # Generate fitted values
    x_fit = np.linspace(0, int(x.max()), 800)
    y_fit = poly_func(x_fit)
    
    # Calculate fitted values for original data points to compute metrics
    y_pred = poly_func(x)
    
    # Calculate fitting quality metrics
    r2 = r2_score(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    mae = mean_absolute_error(y, y_pred)
    residual_std = np.std(y - y_pred)
    
    # Generate the equation string (fixed version)
    equation = "y = "
    for i, c in enumerate(coefficients):
        power = degree - i
        if i > 0:  # Add + or - for subsequent terms
            if c >= 0:
                equation += " + "
            else:
                equation += " - "
                c = abs(c)  # Use absolute value after adding minus sign
        
        if power == 0:
            equation += f"{c}"
        elif power == 1:
            equation += f"{c}x"
        else:
            equation += f"{c}x^{power}"
    
    # Create fitting quality text
    quality_text = f"R² = {r2:.4f}\nRMSE = {rmse:.4f}\nMAE = {mae:.4f}\nResidual σ = {residual_std:.4f}"
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(singlecol*2,singlecol*2), sharex=True)
    
    ax1.scatter(x/2048, y, label='Data', alpha=0.7)
    ax1.plot(x/2048, y, 'b-', alpha=0.5)
    ax1.plot(x_fit/2048, y_fit, 'r-', label=f'Polynomial Fit (degree {degree})', linewidth=2)
    ax1.legend()
    ax2.set_xlabel(r"$x$ in Na$_x$Mn[Fe(CN)$_6$]")
    ax1.set_ylabel('Chemical potential (eV)')
    
    ############################
    coefficients = coefficients
    x0, x_end = 0, 4096
    y0 = -234541.768112556427  # Initial condition
    
    x_cumulative = np.linspace(x0, x_end, 8000)
    dx = x_cumulative[1] - x_cumulative[0]
    integrand_vals = polynomial(x_cumulative)
    cumulative_integral = y0 + np.cumsum(integrand_vals) * dx
    
    slope = (min_y2 - min_y1) / (min_x2 - min_x1)
    intercept = min_y1 - slope * min_x1
    
    difference = cumulative_integral - linear_function(x_cumulative)
    difference_derivative = np.gradient(difference, dx)
    linear_slope = slope 
    
    zero_points = []
    starting_points = np.linspace(x0, x_end, 50)
    for start in starting_points:
        root = optimize.fsolve(derivative_equation, start)[0]
        if abs(derivative_equation(root)) < 1e-8 and x0 <= root <= x_end:
            if all(abs(root - zp) > 10 for zp in zero_points):
                zero_points.append(root)
    zero_points = sorted(zero_points)
    
    # Plot 2: Cumulative integral and linear interpolation
    ax2.plot(x_cumulative/2048, cumulative_integral/2048, 'r-', label='G(x) = G(x₀) + ∫₀ˣ mu(x)dx')
    ax2.plot(min_x1/2048, min_y1/2048, 'ro')
    ax2.plot(min_x2/2048, min_y2/2048, 'ro')
    ax2.plot(x_cumulative/2048, linear_function(x_cumulative)/2048, 'b--', label='L(x) = Cotangent line')
    #ax2.set_title('Free Energy')
    ax2.set_ylabel('Mixing energy\nper formula (eV)')
    ax2.legend()
    
    ax22=ax2.twinx()
    
    # Plot 3: Difference function
    ax22.plot(x_cumulative/2048, difference/2048, 'g-')#, label='Difference = G(x) - L(x)')
    ax22.axhline(0, color='k', linestyle='--', alpha=0.5)
    for zp in [zero_points[0], zero_points[-1]]:
        idx = np.argmin(np.abs(x_cumulative - zp))
        ax22.plot(zp/2048, difference[idx]/2048, 'ro')
    ax2.set_xlabel(r"$x$ in Na$_x$Mn[Fe(CN)$_6$]")
    #ax3.set_title('Difference Function')
    #ax3.grid(True, alpha=0.3)
    ax22.set_ylabel('G(x) - L(x) (eV)', color='green')
    plt.tight_layout()
    plt.savefig(f'Free Energy_{temp}.svg')

In [ ]:

coefficients = np.polyfit(x_fit[500:700]/2048, y_fit[500:700], 2)
poly_func = np.poly1d(coefficients)

# Generate fitted values
x_fit_2 = np.linspace(x_fit[500]/2048, 2, 300)
y_fit_2 = poly_func(x_fit_2)

In [ ]:
i=700
plt.figure()
plt.plot(x_fit/2048, y_fit, label='original fit')
plt.plot(x_fit[500:700]/2048,y_fit[500:700])
plt.plot(x_fit_2, y_fit_2, label='extrapolate fit')
plt.plot(x_fit_2[-100:], y_fit_2[-100:], label='extrapolate fit_2')
plt.legend()

In [ ]:
plt.figure()
plt.plot(x_fit[:7000]/2048, y_fit[:7000], label='original fit')
plt.plot(x_fit_2[-1000:], y_fit_2[-1000:], label='extrapolate fit_2')

In [ ]:
y = np.concatenate((y_fit[:6500], y_fit_2[-1500:]))
x = np.concatenate((x_fit[:6500]/2048, x_fit_2[-1500:]))
plt.figure()
plt.plot(x, y)
data = np.column_stack((x,y))
temp=400
np.savetxt(f"./extrapolate_{temp}K_chemical_potential.csv", data, delimiter=",", header="x,y", comments="", fmt="%.6f")

In [ ]:
plt.figure()
#plt.plot(y_fit)
#plt.plot(x_fit[5000:7000]/2048, y_fit[5000:7000])
#plt.plot(x_fit_2, y_fit_2)
plt.plot(y_fit_2)

In [ ]:
np.concatenate((a, b))
data = np.column_stack((x_fit/2048, y_fit))
np.savetxt(f"./fit_{temp}K_chemical_potential_2.csv", data, delimiter=",", header="x,y", comments="", fmt="%.6f")

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(singlecol*2,singlecol*2), sharex=True)

colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple']

k=0
for temp in range(300,600,100):

    if temp == 700:
        x = np.concatenate([Nf_ls_700_4, Nf_ls_700_5])
        y = np.concatenate([mu_ls_700_4, mu_ls_700_5])
        min_x1, min_y1 = 264.2250,  -235067.4613
        min_x2, min_y2 = 3265.4322,  -240782.9127
        degree = 7
        
    if temp == 600:
        x = np.concatenate([Nf_ls_600_4, Nf_ls_600_5])
        y = np.concatenate([mu_ls_600_4, mu_ls_600_5])
        min_x1, min_y1 = 148.4986,  -234824.9286
        min_x2, min_y2 = 3411.3704,  -240778.0352
        degree = 7
    
    if temp == 500:
        x = np.concatenate([Nf_ls_500_8, Nf_ls_500_8])
        y = np.concatenate([mu_ls_500_8, mu_ls_500_8])
        min_x1, min_y1 = 0,  -234542.5987
        min_x2, min_y2 = 3796.9546,  -241069.6956
        degree = 9
    
    if temp == 400:
        x = np.concatenate([Nf_ls_400_7, Nf_ls_400_7])
        y = np.concatenate([mu_ls_400_7, mu_ls_400_7])
        min_x1, min_y1 = 0,  -234542.5987
        min_x2, min_y2 = 3802.0753, -240968.3298

        degree = 9
    
    if temp == 300:
        x = np.concatenate([Nf_ls_300_9, Nf_ls_300_9])
        y = np.concatenate([mu_ls_300_9, mu_ls_300_9])
        min_x1, min_y1 = 0, -234542.5987
        min_x2, min_y2 = 4096, -241440.8180
        degree = 9
    
    sort_indices = np.argsort(x)
    x = x[sort_indices]
    y = y[sort_indices]
    coefficients = np.polyfit(x, y, degree)
    poly_func = np.poly1d(coefficients)
    
    # Generate fitted values
    x_fit = np.linspace(x0, x_end, 8000)
    y_fit = poly_func(x_fit)
    y_pred = poly_func(x)
    
    # Calculate fitting quality metrics
    r2 = r2_score(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    mae = mean_absolute_error(y, y_pred)
    residual_std = np.std(y - y_pred)
    
    p1, p2 = y_fit[0:2000].argmax(), y_fit[2000:].argmin()+2000
    
    # Create fitting quality text
    quality_text = f"R² = {r2:.4f}\nRMSE = {rmse:.4f}\nMAE = {mae:.4f}\nResidual σ = {residual_std:.4f}"
    
    ax1.scatter(x/2048, y,  alpha=0.15, color=colors[k])
    ax1.plot(x/2048, y, 'b-', alpha=0.2)
    ax1.plot(x_fit/2048, y_fit, linewidth=1, color=colors[k], label=f'{temp} K')
    
    data = np.column_stack((x/2048, y))
    np.savetxt(f"./raw_{temp}K_chemical_potential.csv", data, delimiter=",", header="x,y", comments="", fmt="%.6f")

    data = np.column_stack((x_fit/2048, y_fit))
    np.savetxt(f"./fit_{temp}K_chemical_potential.csv", data, delimiter=",", header="x,y", comments="", fmt="%.6f")

    
    if temp != 600:
        ax1.scatter(x_fit[p1]/2048, y_fit[p1], color=colors[k], s=25)
        ax1.scatter(x_fit[p2]/2048, y_fit[p2], color=colors[k], s=25)
    else:
        ax1.scatter(x_fit[p1]/2048, y_fit[p1], color=colors[k], s=25)
        ax1.scatter(x_fit[3545]/2048, y_fit[3545], color=colors[k], s=25)

    #print(temp,y_fit[p1],y_fit[p2])
    ax1.legend()
    ax2.set_xlabel(r"$x$ in Na$_x$Mn[Fe(CN)$_6$]")
    ax1.set_ylabel('Chemical potential (eV)')
    
    ############################
    coefficients = coefficients
    x0, x_end = 0, 4096
    y0 = -234541.768112556427  # Initial condition
    
    x_cumulative = np.linspace(x0, x_end, 8000)
    dx = x_cumulative[1] - x_cumulative[0]
    integrand_vals = polynomial(x_cumulative)
    cumulative_integral = y0 + np.cumsum(integrand_vals) * dx
    
    slope = (min_y2 - min_y1) / (min_x2 - min_x1)
    intercept = min_y1 - slope * min_x1
    
    difference = cumulative_integral - linear_function(x_cumulative)
    difference_derivative = np.gradient(difference, dx)
    linear_slope = slope 
    
    ############
    zero_points = []
    starting_points = np.linspace(x0, x_end, 50)
    for start in starting_points:
        root = optimize.fsolve(derivative_equation, start)[0]
        if abs(derivative_equation(root)) < 1e-8 and x0 <= root <= x_end:
            if all(abs(root - zp) > 10 for zp in zero_points):
                zero_points.append(root)
    zero_points = sorted(zero_points)
    for zp in [zero_points[0],zero_points[-1]]:
        idx = np.argmin(np.abs(x_cumulative - zp))
        ax2.scatter(zp/2048, difference[idx]/2048, color=colors[k], s=15)
        print(temp, zp/2048)
    ax2.set_ylabel('Mixing energy\nper formula (eV)')
    #ax2.legend()

    # Plot 3: Difference function
    ax2.plot(x_cumulative/2048, difference/2048, color=colors[k])#, 'g-')#, label='Difference = G(x) - L(x)')
    ax2.axhline(0, color='k', linestyle='--', alpha=0.5)
    ax2.set_xlabel(r"$x$ in Na$_x$Mn[Fe(CN)$_6$]")
    #ax3.set_title('Difference Function')
    plt.tight_layout()
    k+=1
    plt.savefig(f'Free Energy.svg')

In [ ]:
x_array = x_fit / 2048
y_array = y_fit

dy_dx = np.gradient(y_array, x_array)

In [ ]:
plt.figure()
plt.plot(dy_dx)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.optimize import curve_fit

class PiecewisePolynomial:
    def __init__(self, degree1=2, degree2=2):
        """
        Initialize piecewise polynomial fitter
        
        Parameters:
        degree1: polynomial degree for first segment
        degree2: polynomial degree for second segment
        """
        self.degree1 = degree1
        self.degree2 = degree2
        self.breakpoint = None
        self.coeffs1 = None
        self.coeffs2 = None
        
    def poly_func(self, x, coeffs):
        """Evaluate polynomial with given coefficients"""
        return sum(coeff * x**i for i, coeff in enumerate(coeffs))
    
    def piecewise_func(self, x, breakpoint, coeffs1, coeffs2):
        """Evaluate piecewise polynomial function"""
        result = np.zeros_like(x)
        mask1 = x <= breakpoint
        mask2 = x > breakpoint
        
        if np.any(mask1):
            result[mask1] = self.poly_func(x[mask1], coeffs1)
        if np.any(mask2):
            result[mask2] = self.poly_func(x[mask2], coeffs2)
            
        return result
    
    def objective_function(self, params, x_data, y_data):
        """Objective function for optimization"""
        breakpoint = params[0]
        coeffs1 = params[1:self.degree1+2]
        coeffs2 = params[self.degree1+2:]
        
        # Predicted values
        y_pred = self.piecewise_func(x_data, breakpoint, coeffs1, coeffs2)
        
        # Mean squared error
        mse = np.mean((y_data - y_pred)**2)
        
        # Add continuity constraint penalty
        continuity_penalty = 0
        if len(coeffs1) > 0 and len(coeffs2) > 0:
            val1 = self.poly_func(breakpoint, coeffs1)
            val2 = self.poly_func(breakpoint, coeffs2)
            continuity_penalty = 1000 * (val1 - val2)**2
        
        return mse + continuity_penalty
    
    def fit(self, x_data, y_data, initial_breakpoint=None):
        """
        Fit piecewise polynomial to data
        
        Parameters:
        x_data: array of x values
        y_data: array of y values
        initial_breakpoint: initial guess for breakpoint (default: median of x)
        """
        if initial_breakpoint is None:
            initial_breakpoint = np.median(x_data)
        
        # Initial parameter guess
        initial_params = [initial_breakpoint]
        initial_params.extend([0] * (self.degree1 + 1))  # coeffs1
        initial_params.extend([0] * (self.degree2 + 1))  # coeffs2
        
        # Bounds for breakpoint
        x_min, x_max = np.min(x_data), np.max(x_data)
        bounds = [(x_min + 0.1*(x_max-x_min), x_max - 0.1*(x_max-x_min))]
        bounds.extend([(None, None)] * (self.degree1 + self.degree2 + 2))
        
        # Optimize
        result = minimize(self.objective_function, initial_params, 
                         args=(x_data, y_data), bounds=bounds, 
                         method='L-BFGS-B')
        
        # Extract results
        self.breakpoint = result.x[0]
        self.coeffs1 = result.x[1:self.degree1+2]
        self.coeffs2 = result.x[self.degree1+2:]
        
        return result.success
    
    def predict(self, x):
        """Predict values for new x data"""
        if self.breakpoint is None:
            raise ValueError("Model not fitted yet")
        
        return self.piecewise_func(x, self.breakpoint, self.coeffs1, self.coeffs2)
    
    def plot_fit(self, x_data, y_data, x_pred=None, title="Piecewise Polynomial Fit"):
        """Plot the fitted piecewise polynomial"""
        if x_pred is None:
            x_pred = np.linspace(np.min(x_data), np.max(x_data), 200)
        
        y_pred = self.predict(x_pred)
        
        plt.figure(figsize=(10, 6))
        plt.scatter(x_data, y_data, alpha=0.7, label='Data', color='blue')
        plt.plot(x_pred, y_pred, 'r-', linewidth=2, label='Piecewise Fit')
        plt.axvline(x=self.breakpoint, color='green', linestyle='--', 
                   label=f'Breakpoint: {self.breakpoint:.3f}')
        
        plt.xlabel('x')
        plt.ylabel('y')
        plt.title(title)
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()
    
    def get_summary(self):
        """Get summary of the fitted model"""
        if self.breakpoint is None:
            return "Model not fitted yet"
        
        summary = f"Piecewise Polynomial Fit Summary:\n"
        summary += f"Breakpoint: {self.breakpoint:.6f}\n"
        summary += f"Segment 1 (degree {self.degree1}): "
        summary += " + ".join([f"{coeff:.6f}*x^{i}" for i, coeff in enumerate(self.coeffs1)])
        summary += f"\nSegment 2 (degree {self.degree2}): "
        summary += " + ".join([f"{coeff:.6f}*x^{i}" for i, coeff in enumerate(self.coeffs2)])
        
        return summary

# Example usage and demonstration
if __name__ == "__main__":
    # Generate sample data with a clear breakpoint
    np.random.seed(42)
    x = np.linspace(0, 10, 100)
    
    # Create piecewise data: quadratic before x=5, linear after x=5
    y = np.where(x < 5, 
                 0.5*x**2 - 2*x + 3,  # quadratic segment
                 2*x - 5)             # linear segment
    
    # Add some noise
    y += np.random.normal(0, 0.5, len(x))
    
    # Fit piecewise polynomial
    fitter = PiecewisePolynomial(degree1=2, degree2=1)
    success = fitter.fit(x, y)
    
    if success:
        print(fitter.get_summary())
        fitter.plot_fit(x, y)
        
        # Calculate R-squared
        y_pred = fitter.predict(x)
        ss_res = np.sum((y - y_pred)**2)
        ss_tot = np.sum((y - np.mean(y))**2)
        r_squared = 1 - (ss_res / ss_tot)
        print(f"\nR-squared: {r_squared:.4f}")
    else:
        print("Fitting failed!")

In [ ]:
x = np.concatenate([Nf_ls_300_9, Nf_ls_300_9])
y = np.concatenate([mu_ls_300_9, mu_ls_300_9])
sort_indices = np.argsort(x)
x = x[sort_indices]
y = y[sort_indices]
fitter = PiecewisePolynomial(degree1=4, degree2=4)  # quadratic + linear
fitter.fit(x, y, initial_breakpoint=2000)
#y_fitted = fitter.predict(x_new)

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Your data fitting
x = np.concatenate([Nf_ls_400_7, Nf_ls_400_7])[sort_indices]
y = np.concatenate([mu_ls_400_7, mu_ls_400_7])[sort_indices]
degree = 9
coefficients = np.polyfit(x, y, degree)
poly_func = np.poly1d(coefficients)

# Generate fitted values
x_fit = np.linspace(int(x.min()), int(x.max()), 100)
y_fit = poly_func(x_fit)

# Calculate fitted values for original data points to compute metrics
y_pred = poly_func(x)

# Calculate fitting quality metrics
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
mae = mean_absolute_error(y, y_pred)
residual_std = np.std(y - y_pred)

# Generate the equation string (fixed version)
equation = "y = "
for i, c in enumerate(coefficients):
    power = degree - i
    if i > 0:  # Add + or - for subsequent terms
        if c >= 0:
            equation += " + "
        else:
            equation += " - "
            c = abs(c)  # Use absolute value after adding minus sign
    
    if power == 0:
        equation += f"{c}"
    elif power == 1:
        equation += f"{c}x"
    else:
        equation += f"{c}x^{power}"

# Create fitting quality text
quality_text = f"R² = {r2:.4f}\nRMSE = {rmse:.4f}\nMAE = {mae:.4f}\nResidual σ = {residual_std:.4f}"

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(singlecol*2,singlecol*2), sharex=True)

ax1.scatter(x/2048, y, label='Data', alpha=0.7)
ax1.plot(x/2048, y, 'b-', alpha=0.5)
ax1.plot(x_fit/2048, y_fit, 'r-', label=f'Polynomial Fit (degree {degree})', linewidth=2)
ax1.legend()
ax2.set_xlabel(r"$x$ in Na$_x$Mn[Fe(CN)$_6$]")
ax1.set_ylabel('Chemical potential (eV)')

############################
coefficients = coefficients
x0, x_end = 0, 4096
y0 = -234541.768112556427  # Initial condition

x_cumulative = np.linspace(x0, x_end, 2000)
dx = x_cumulative[1] - x_cumulative[0]
integrand_vals = polynomial(x_cumulative)
cumulative_integral = y0 + np.cumsum(integrand_vals) * dx

### 400K (4+5) ###
min_x2, min_x1 = 2122.7894, 458.9815
min_y2, min_y1 = -238031.6572, -235242.1625

### 400K (5) ###
#min_x2, min_x1 = 3378.5983, 14.8499
#min_y2, min_y1 = -240267.2852, -234568.1466

### 300K ###
#min_x2, min_x1 = 3835.3594, 0.0000
#min_y2, min_y1 = -240892.1041, -234542.5764

slope = (min_y2 - min_y1) / (min_x2 - min_x1)
intercept = min_y1 - slope * min_x1

difference = cumulative_integral - linear_function(x_cumulative)
difference_derivative = np.gradient(difference, dx)
linear_slope = slope 

zero_points = []
starting_points = np.linspace(x0, x_end, 50)
for start in starting_points:
    root = optimize.fsolve(derivative_equation, start)[0]
    if abs(derivative_equation(root)) < 1e-8 and x0 <= root <= x_end:
        if all(abs(root - zp) > 10 for zp in zero_points):
            zero_points.append(root)
zero_points = sorted(zero_points)

# Plot 2: Cumulative integral and linear interpolation
ax2.plot(x_cumulative/2048, cumulative_integral/2048, 'r-', label='G(x) = G(x₀) + ∫₀ˣ mu(x)dx')
ax2.plot(min_x1/2048, min_y1/2048, 'ro')
ax2.plot(min_x2/2048, min_y2/2048, 'ro')
ax2.plot(x_cumulative/2048, linear_function(x_cumulative)/2048, 'b--', label='L(x) = Cotangent line')
#ax2.set_title('Free Energy')
ax2.set_ylabel('Energy per formula (eV)')
ax2.legend()

ax22=ax2.twinx()

# Plot 3: Difference function
ax22.plot(x_cumulative/2048, difference/2048, 'g-')#, label='Difference = G(x) - L(x)')
ax22.axhline(0, color='k', linestyle='--', alpha=0.5)
for zp in zero_points:
    idx = np.argmin(np.abs(x_cumulative - zp))
    ax22.plot(zp/2048, difference[idx]/2048, 'ro')
ax2.set_xlabel(r"$x$ in Na$_x$Mn[Fe(CN)$_6$]")
#ax3.set_title('Difference Function')
#ax3.grid(True, alpha=0.3)
ax22.set_ylabel('G(x) - L(x) (eV)', color='green')
plt.tight_layout()


In [ ]:
coeff_300,coeff_400,coeff_500

In [ ]:
x = np.concatenate([Nf_ls_300_9, Nf_ls_300_9])
y = np.concatenate([mu_ls_300_9, mu_ls_300_9])
sort_indices = np.argsort(x)
coeff_300 = fit_chemical_potential(x,y,9)

In [ ]:
x = np.concatenate([Nf_ls_400_7, Nf_ls_400_7])
y = np.concatenate([mu_ls_400_7, mu_ls_400_7])
sort_indices = np.argsort(x)
coeff_400 = fit_chemical_potential(x,y,9)

In [ ]:
x = np.concatenate([Nf_ls_500_8, Nf_ls_500_8])
y = np.concatenate([mu_ls_500_8, mu_ls_500_8])
sort_indices = np.argsort(x)
coeff_500 = fit_chemical_potential(x,y,9)

In [ ]:
x = np.concatenate([Nf_ls_600_4, Nf_ls_600_5])
y = np.concatenate([mu_ls_600_4, mu_ls_600_5])
sort_indices = np.argsort(x)
coeff_600 = fit_chemical_potential(x,y,7)

In [ ]:
x = np.concatenate([Nf_ls_700_4, Nf_ls_700_5])
y = np.concatenate([mu_ls_700_4, mu_ls_700_5])
sort_indices = np.argsort(x)
coeff_700 = fit_chemical_potential(x,y,7)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate, optimize
import sympy as sp
import warnings
warnings.filterwarnings('ignore')

# =====================================================
# 1. CONFIGURATION
# =====================================================

# Polynomial coefficients (from highest to lowest degree)
coefficients = coeff_500
x0, x_end = 0, 4096
y0 = -234541.768112556427  # Initial condition

# =====================================================
# 2. POLYNOMIAL DEFINITION
# =====================================================

def polynomial(x):
    return np.polyval(coefficients, x)

# =====================================================
# 3. INTEGRATION METHODS
# =====================================================

def integrate_numerically():
    result, error = integrate.quad(polynomial, x0, x_end)
    return result, error

def integrate_analytically():
    x = sp.Symbol('x')
    expr = sum(c * x**i for i, c in enumerate(reversed(coefficients)))
    antiderivative = sp.integrate(expr, x)
    result = float(antiderivative.subs(x, x_end) - antiderivative.subs(x, x0))
    return result, antiderivative

def integrate_manually():
    powers = list(range(8, -1, -1))
    integrated_coeffs = [(coef / (p + 1), p + 1) for coef, p in zip(coefficients, powers)]
    
    def F(x):
        return sum(c * x**p for c, p in integrated_coeffs)

    result = F(x_end) - F(x0)
    return result, integrated_coeffs

# =====================================================
# 4. COMPUTE AND COMPARE RESULTS
# =====================================================

numerical_result, numerical_error = integrate_numerically()
analytical_result, antiderivative_expr = integrate_analytically()
manual_result, integrated_coeffs = integrate_manually()

print("="*60)
print("INTEGRATION RESULTS")
print("="*60)
print(f"Numerical Integration:  {numerical_result:.10e} (±{numerical_error:.2e})")
print(f"Analytical Integration: {analytical_result:.10e}")
print(f"Manual Integration:     {manual_result:.10e}")
print(f"Difference Num-Ana:     {abs(numerical_result - analytical_result):.2e}")
print(f"Difference Num-Man:     {abs(numerical_result - manual_result):.2e}")

# =====================================================
# 5. INITIAL VALUE PROBLEM
# =====================================================

y4096 = y0 + numerical_result
print("\n" + "="*60)
print("INITIAL VALUE PROBLEM")
print("="*60)
print(f"y(4096) = y(0) + ∫₀⁴⁰⁹⁶ f(x) dx = {y0:.6e} + {numerical_result:.6e} = {y4096:.6e}")

# =====================================================
# 6. CUMULATIVE INTEGRAL & DIFFERENCE
# =====================================================

#min = polynomial(np.arange(2000,3000))

x_cumulative = np.linspace(x0, x_end, 8000)
dx = x_cumulative[1] - x_cumulative[0]
integrand_vals = polynomial(x_cumulative)
cumulative_integral = y0 + np.cumsum(integrand_vals) * dx

### 400K (4+5) ###
#min_x2, min_x1 = 3304.8611, 14.8499
#min_y2, min_y1 = -240117.6307, -234567.9971

### 400K (5) ###
#min_x2, min_x1 = 3378.5983, 14.8499
#min_y2, min_y1 = -240267.2852, -234568.1466

### 300K ###
#min_x2, min_x1 = 3835.3594, 0.0000
#min_y2, min_y1 = -240892.1041, -234542.5764

### 600K'
min_x2, min_x1 = 3796.9546, 0
min_y2, min_y1 = -241069.6956, -234542.5987
#x=818.2783, G(x)=-235972.6036

slope = (min_y2 - min_y1) / (min_x2 - min_x1)
intercept = min_y1 - slope * min_x1

def linear_function(x):
    return slope * x + intercept

difference = cumulative_integral - linear_function(x_cumulative)
difference_derivative = np.gradient(difference, dx)
linear_slope = slope 

# =====================================================
# 7. FIND ZEROS OF DIFFERENCE DERIVATIVE
# =====================================================

def derivative_equation(x):
    return polynomial(x) - linear_slope

zero_points = []
starting_points = np.linspace(x0, x_end, 50)
for start in starting_points:
    root = optimize.fsolve(derivative_equation, start)[0]
    if abs(derivative_equation(root)) < 1e-8 and x0 <= root <= x_end:
        if all(abs(root - zp) > 10 for zp in zero_points):
            zero_points.append(root)
zero_points = sorted(zero_points)

print("\n" + "="*60)
print("ZEROS OF DERIVATIVE (CRITICAL POINTS OF THE DIFFERENCE)")
print("="*60)
for i, zp in enumerate(zero_points):
    idx = np.argmin(np.abs(x_cumulative - zp))
    print(f"Point {i+1}: x = {zp:.2f}, diff = {difference[idx]:.2e}, f(x) = {polynomial(zp):.2e}")

# =====================================================
# 8. ANALYZE NATURE OF CRITICAL POINTS
# =====================================================

print("\n" + "="*60)
print("CRITICAL POINT CLASSIFICATION")
print("="*60)
for i, zp in enumerate(zero_points):
    h = 1.0
    if x0 + h <= zp <= x_end - h:
        d2 = (derivative_equation(zp + h) - derivative_equation(zp - h)) / (2 * h)
        nature = "Min" if d2 > 1e-6 else "Max" if d2 < -1e-6 else "Inflection"
        print(f"x = {zp:.2f}, 2nd derivative ≈ {d2:.2e}, Type: {nature}")

# =====================================================
# 9. VISUALIZATION
# =====================================================
singlecol=3.5
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(singlecol*2,singlecol*2))

# Plot 1: Original polynomial
ax1.plot(x_cumulative, polynomial(x_cumulative), 'b-', label='Fitted chemical potential: mu(x)')
#ax1.axhline(min_y,color='r', linestyle='--',)
ax1.plot(min_x1, polynomial(min_x1), 'ro')
ax1.plot(min_x2, polynomial(min_x2), 'ro')
ax1.set_ylabel('Chemical potential (eV)')
ax2.set_ylabel('Energy (eV)')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Plot 2: Cumulative integral and linear interpolation
ax2.plot(x_cumulative, cumulative_integral, 'r-', label='G(x) = G(x₀) + ∫₀ˣ mu(x)dx')
ax2.plot(min_x1, min_y1, 'ro')
ax2.plot(min_x2, min_y2, 'ro')
ax2.plot(x_cumulative, linear_function(x_cumulative), 'b--', label='L(x) = Cotangent line')
ax2.set_title('Free Energy')
ax2.set_xlabel('Energy (eV)')
ax2.grid(True, alpha=0.3)
ax2.legend()

ax22=ax2.twinx()

# Plot 3: Difference function
ax22.plot(x_cumulative, difference, 'g-')#, label='Difference = G(x) - L(x)')
ax22.axhline(0, color='k', linestyle='--', alpha=0.5)
for zp in zero_points:
    idx = np.argmin(np.abs(x_cumulative - zp))
    ax22.plot(zp, difference[idx], 'ro')
ax2.set_xlabel(r"$x$ in Na$_x$Mn[Fe(CN)$_6$]")
#ax3.set_title('Difference Function')
#ax3.grid(True, alpha=0.3)
ax22.set_ylabel('G(x) - L(x) (eV)', color='green')
plt.tight_layout()
# Plot 4: Derivative of difference

# =====================================================
# 10. STATISTICS
# =====================================================

print("\n" + "="*60)
print("STATISTICS OF DIFFERENCE")
print("="*60)
print(f"Max Difference: {np.max(difference):.2e} at x = {x_cumulative[np.argmax(difference)]:.1f}")
print(f"Min Difference: {np.min(difference):.2e} at x = {x_cumulative[np.argmin(difference)]:.1f}")
print(f"Mean: {np.mean(difference):.2e}, Std: {np.std(difference):.2e}, Range: {np.ptp(difference):.2e}")

# =====================================================
# 11. ANTIDERIVATIVE FUNCTION
# =====================================================

def antiderivative_function(x):
    return sum(c * x**p for c, p in integrated_coeffs)

print("\n" + "="*60)
print("ANTIDERIVATIVE FUNCTION")
print("="*60)
print(f"F(0) = {antiderivative_function(0):.10e}")
print(f"F(4096) = {antiderivative_function(4096):.10e}")
print(f"F(4096) - F(0) = {antiderivative_function(4096) - antiderivative_function(0):.10e}")


### Convex Hull

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares


def find_common_tangents(
    coefficients,
    x_bounds=(0.0, 4096.0),
    G0=0.0,
    n_grid=12001,
    min_skipped_points=5,
):
    """
    Find common-tangent regions of G(x), where

        mu(x) = polynomial defined by coefficients
        G(x)  = G0 + integral[mu(s) ds]

    Polynomial coefficients follow np.polyval convention:
        [highest power, ..., constant]

    Returns
    -------
    analysis : dict
        analysis["tangents"] contains the detected tangent regions.
    """

    coefficients = np.asarray(coefficients, dtype=float)

    mu_poly = np.poly1d(coefficients)
    G_primitive = np.polyint(mu_poly)

    x_min, x_max = map(float, x_bounds)

    def mu(x):
        return mu_poly(x)

    # Relative free energy is numerically better for hull calculations.
    def G_relative(x):
        return G_primitive(x) - G_primitive(x_min)

    def G(x):
        return G0 + G_relative(x)

    # -------------------------------------------------
    # Sample G(x)
    # -------------------------------------------------
    x_grid = np.linspace(x_min, x_max, n_grid)
    G_relative_grid = G_relative(x_grid)
    G_grid = G0 + G_relative_grid

    # -------------------------------------------------
    # Construct lower convex envelope
    # -------------------------------------------------
    lower_hull = []

    for i in range(n_grid):
        while len(lower_hull) >= 2:
            i0 = lower_hull[-2]
            i1 = lower_hull[-1]

            slope_01 = (
                G_relative_grid[i1] - G_relative_grid[i0]
            ) / (
                x_grid[i1] - x_grid[i0]
            )

            slope_12 = (
                G_relative_grid[i] - G_relative_grid[i1]
            ) / (
                x_grid[i] - x_grid[i1]
            )

            # Convex function must have increasing slopes.
            if slope_12 <= slope_01:
                lower_hull.pop()
            else:
                break

        lower_hull.append(i)

    lower_hull = np.asarray(lower_hull, dtype=int)

    # -------------------------------------------------
    # Find hull segments that skip original points
    # -------------------------------------------------
    dx = x_grid[1] - x_grid[0]
    tangent_results = []

    for index_left, index_right in zip(
        lower_hull[:-1], lower_hull[1:]
    ):
        skipped_points = index_right - index_left - 1

        if skipped_points < min_skipped_points:
            continue

        x1_guess = x_grid[index_left]
        x2_guess = x_grid[index_right]
        initial_width = x2_guess - x1_guess

        left_boundary = index_left == 0
        right_boundary = index_right == n_grid - 1

        # Restrict optimization to a small neighborhood around
        # the convex-hull estimates. This prevents convergence to x1=x2.
        refinement_width = min(5 * dx, 0.2 * initial_width)

        x1_lower = max(x_min, x1_guess - refinement_width)
        x1_upper = min(x_max, x1_guess + refinement_width)

        x2_lower = max(x_min, x2_guess - refinement_width)
        x2_upper = min(x_max, x2_guess + refinement_width)

        # ---------------------------------------------
        # Interior-interior common tangent
        # ---------------------------------------------
        if not left_boundary and not right_boundary:

            def residual(values):
                x1, x2 = values
                common_slope = (
                    G_relative(x2) - G_relative(x1)
                ) / (
                    x2 - x1
                )

                return np.array([
                    mu(x1) - common_slope,
                    mu(x2) - common_slope,
                ])

            solution = least_squares(
                residual,
                x0=[x1_guess, x2_guess],
                bounds=(
                    [x1_lower, x2_lower],
                    [x1_upper, x2_upper],
                ),
                xtol=1e-13,
                ftol=1e-13,
                gtol=1e-13,
                max_nfev=2000,
            )

            x1, x2 = solution.x
            residual_norm = np.linalg.norm(residual([x1, x2]))
            tangent_type = "common tangent"

        # ---------------------------------------------
        # Left endpoint constrained by physical domain
        # ---------------------------------------------
        elif left_boundary and not right_boundary:
            x1 = x_min

            def residual_x2(value):
                x2 = value[0]
                common_slope = (
                    G_relative(x2) - G_relative(x1)
                ) / (
                    x2 - x1
                )
                return np.array([mu(x2) - common_slope])

            solution = least_squares(
                residual_x2,
                x0=[x2_guess],
                bounds=([x2_lower], [x2_upper]),
                xtol=1e-13,
                ftol=1e-13,
                gtol=1e-13,
            )

            x2 = solution.x[0]
            residual_norm = np.linalg.norm(residual_x2([x2]))
            tangent_type = "left-boundary tangent"

        # ---------------------------------------------
        # Right endpoint constrained by physical domain
        # ---------------------------------------------
        elif right_boundary and not left_boundary:
            x2 = x_max

            def residual_x1(value):
                x1 = value[0]
                common_slope = (
                    G_relative(x2) - G_relative(x1)
                ) / (
                    x2 - x1
                )
                return np.array([mu(x1) - common_slope])

            solution = least_squares(
                residual_x1,
                x0=[x1_guess],
                bounds=([x1_lower], [x1_upper]),
                xtol=1e-13,
                ftol=1e-13,
                gtol=1e-13,
            )

            x1 = solution.x[0]
            residual_norm = np.linalg.norm(residual_x1([x1]))
            tangent_type = "right-boundary tangent"

        else:
            x1 = x_min
            x2 = x_max
            residual_norm = np.nan
            tangent_type = "boundary-to-boundary chord"

        common_slope = (
            G_relative(x2) - G_relative(x1)
        ) / (
            x2 - x1
        )

        intercept = G(x1) - common_slope * x1

        # Verify that the line lies below G between x1 and x2.
        x_check = np.linspace(x1, x2, 2001)
        tangent_line = G(x1) + common_slope * (x_check - x1)
        difference = G(x_check) - tangent_line

        numerical_tolerance = (
            1e-8 * max(1.0, np.ptp(G_relative_grid))
        )

        if np.min(difference) < -numerical_tolerance:
            continue

        tangent_results.append({
            "x1": float(x1),
            "x2": float(x2),
            "G1": float(G(x1)),
            "G2": float(G(x2)),
            "mu1": float(mu(x1)),
            "mu2": float(mu(x2)),
            "slope": float(common_slope),
            "intercept": float(intercept),
            "residual": float(residual_norm),
            "type": tangent_type,
            "skipped_points": int(skipped_points),
        })

    return {
        "x": x_grid,
        "mu": mu(x_grid),
        "G": G_grid,
        "lower_hull_indices": lower_hull,
        "tangents": tangent_results,
        "mu_function": mu,
        "G_function": G,
    }

In [ ]:
analysis = find_common_tangents(
    coefficients=coeff_500,
    x_bounds=(0, 4096),
    G0=-234541.768112556427,
    n_grid=20001,
)

for i, tangent in enumerate(analysis["tangents"], start=1):
    print(f"\nTangent region {i}: {tangent['type']}")
    print(f"  N1 = {tangent['x1']:.6f}")
    print(f"  N2 = {tangent['x2']:.6f}")
    print(f"  Na composition x1 = {tangent['x1']/2048:.6f}")
    print(f"  Na composition x2 = {tangent['x2']/2048:.6f}")
    print(f"  Chemical potential = {tangent['slope']:.8f} eV")
    print(f"  Refinement residual = {tangent['residual']:.3e}")

In [ ]:
N = analysis["x"]
G = analysis["G"]
hull_indices = analysis["lower_hull_indices"]

fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(8, 8),
    sharex=True,
)
x = np.concatenate([Nf_ls_300_9, Nf_ls_300_9])

#x = np.concatenate([Nf_ls_400_7, Nf_ls_400_7])
#x = np.concatenate([Nf_ls_500_8, Nf_ls_500_8])
# Chemical-potential fit
ax1.scatter(x/ 2048, y, s=12, alpha=0.6, label="Data")
ax1.plot(
    N / 2048,
    analysis["mu"],
    color="red",
    linewidth=2,
    label=f"Polynomial fit, degree {degree}",
)

ax1.set_ylabel("Chemical potential (eV)")
ax1.legend()
ax1.grid(alpha=0.3)

# Free energy
ax2.plot(
    N / 2048,
    G / 2048,
    color="orange",
    label=r"$G(N)=G_0+\int_0^N\mu(s)\,ds$",
)

ax2.plot(
    N[hull_indices] / 2048,
    G[hull_indices] / 2048,
    "b--",
    linewidth=1.5,
    label="Lower convex envelope",
)

for i, tangent in enumerate(analysis["tangents"]):
    N1 = tangent["x1"]
    N2 = tangent["x2"]

    N_line = np.linspace(N1, N2, 500)
    G_line = (
        tangent["G1"]
        + tangent["slope"] * (N_line - N1)
    )

    ax2.plot(
        N_line / 2048,
        G_line / 2048,
        color="red",
        linewidth=2.5,
        label="Common tangent" if i == 0 else None,
    )

    ax2.scatter(
        [N1 / 2048, N2 / 2048],
        [tangent["G1"] / 2048, tangent["G2"] / 2048],
        color="red",
        s=50,
        zorder=10,
    )

ax2.set_xlabel(r"$x$ in Na$_x$Mn[Fe(CN)$_6$]")
ax2.set_ylabel("Energy per formula unit (eV)")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# If scipy is available, use it; otherwise use a fallback convex-hull implementation
try:
    from scipy.spatial import ConvexHull
    use_scipy = True
except ImportError:
    use_scipy = False

# =====================================================
# 1. CONFIGURATION
# =====================================================
# Polynomial coefficients (highest to lowest degree)
coefficients = coeff_500  # You'll need to define this
x0, x_end = 0, 4096
y0 = -234541.768112556427  # Initial offset for cumulative integral

# =====================================================
# 2. POLYNOMIAL FUNCTION
# =====================================================
def polynomial(x):
    return np.polyval(coefficients, x)

# =====================================================
# 3. BUILD CUMULATIVE INTEGRAL
# =====================================================
# Discretize x over 8000 points
num_points = 8000
x_cumulative = np.linspace(x0, x_end, num_points)
dx = x_cumulative[1] - x_cumulative[0]

# Evaluate polynomial at each x
mu_vals = polynomial(x_cumulative)

# Compute cumulative integral via a simple Riemann sum
cumulative_integral = y0 + np.cumsum(mu_vals) * dx

# =====================================================
# 4. CONVEX HULL COMPUTATION
# =====================================================
# Stack (x, G(x)) pairs into an (N, 2) array
points = np.column_stack((x_cumulative, cumulative_integral))

if use_scipy:
    # Use scipy.spatial.ConvexHull if available
    hull = ConvexHull(points)
    hull_vertices = points[hull.vertices]
else:
    # Fallback: monotonic chain algorithm (a.k.a. Andrew's algorithm)
    def monotonic_chain(pts):
        """Return convex hull of a set of 2D points using the Monotonic Chain algorithm."""
        # Sort by x, then by y
        pts = sorted(pts.tolist())
        if len(pts) <= 1:
            return np.array(pts)
        
        def cross(o, a, b):
            return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])
        
        lower = []
        for p in pts:
            while len(lower) >= 2 and cross(lower[-2], lower[-1], p) <= 0:
                lower.pop()
            lower.append(p)
        
        upper = []
        for p in reversed(pts):
            while len(upper) >= 2 and cross(upper[-2], upper[-1], p) <= 0:
                upper.pop()
            upper.append(p)
        
        # Concatenate lower and upper hull, excluding last point of each to avoid duplication
        hull_pts = np.array(lower[:-1] + upper[:-1])
        return hull_pts
    
    hull_vertices = monotonic_chain(points)

# Sort hull vertices by x-coordinate for analysis
hull_sorted = hull_vertices[np.argsort(hull_vertices[:, 0])]

# =====================================================
# 5. FIND DISCONTINUOUS VERTICES
# =====================================================
def find_discontinuous_vertices(hull_vertices, original_dx, threshold_multiplier=5):
    """
    Find vertices in the convex hull that represent discontinuities.
    
    Parameters:
    - hull_vertices: sorted hull vertices (N, 2) array
    - original_dx: spacing between original data points
    - threshold_multiplier: multiplier for dx to determine what constitutes a gap
    
    Returns:
    - discontinuous_segments: list of tuples (start_vertex, end_vertex) for each gap
    - gap_info: list of dictionaries with gap information
    """
    if len(hull_vertices) < 2:
        return [], []
    
    threshold = threshold_multiplier * original_dx
    discontinuous_segments = []
    gap_info = []
    
    for i in range(len(hull_vertices) - 1):
        x1, y1 = hull_vertices[i]
        x2, y2 = hull_vertices[i + 1]
        
        dx_gap = x2 - x1
        
        if dx_gap > threshold:
            discontinuous_segments.append((hull_vertices[i], hull_vertices[i + 1]))
            gap_info.append({
                'start_point': (x1, y1),
                'end_point': (x2, y2),
                'x_gap': dx_gap,
                'y_gap': y2 - y1,
                'gap_ratio': dx_gap / original_dx,
                'segment_index': i
            })
    
    return discontinuous_segments, gap_info

# Find discontinuous vertices
discontinuous_segments, gap_info = find_discontinuous_vertices(hull_sorted, dx)

# =====================================================
# 6. ALTERNATIVE METHOD: FIND VERTICES NOT ON ORIGINAL CURVE
# =====================================================
def find_vertices_off_curve(hull_vertices, x_original, y_original, tolerance=1e-6):
    """
    Find hull vertices that don't lie close to the original curve.
    This identifies vertices that are "corners" of the convex hull.
    """
    vertices_off_curve = []
    
    for hx, hy in hull_vertices:
        # Find closest x-value in original data
        closest_idx = np.argmin(np.abs(x_original - hx))
        closest_x = x_original[closest_idx]
        closest_y = y_original[closest_idx]
        
        # Check if hull vertex is significantly different from curve
        x_diff = abs(hx - closest_x)
        y_diff = abs(hy - closest_y)
        
        if x_diff > tolerance or y_diff > tolerance:
            vertices_off_curve.append({
                'hull_vertex': (hx, hy),
                'closest_curve_point': (closest_x, closest_y),
                'x_difference': x_diff,
                'y_difference': y_diff,
                'is_corner': True
            })
    
    return vertices_off_curve

# Find vertices that represent corners (not on the original curve)
corner_vertices = find_vertices_off_curve(hull_sorted, x_cumulative, cumulative_integral)

# =====================================================
# 7. OUTPUT RESULTS
# =====================================================
print("="*60)
print("CONVEX HULL ANALYSIS")
print("="*60)

print(f"\nTotal hull vertices: {len(hull_sorted)}")
print(f"Original data spacing (dx): {dx:.6f}")

#print("\nAll Convex Hull Vertices (x, G(x)):")
#for i, (x_val, g_val) in enumerate(hull_sorted):
#    print(f"  {i:2d}: x = {x_val:10.4f},  G(x) = {g_val:12.4f}")

if discontinuous_segments:
    print(f"\nDISCONTINUOUS SEGMENTS FOUND: {len(discontinuous_segments)}")
    print("-" * 50)
    for i, gap in enumerate(gap_info):
        print(f"Gap {i+1}:")
        print(f"  From: x={gap['start_point'][0]:.4f}, G(x)={gap['start_point'][1]:.4f}")
        print(f"  To:   x={gap['end_point'][0]:.4f}, G(x)={gap['end_point'][1]:.4f}")
        print(f"  X-gap: {gap['x_gap']:.4f} (ratio: {gap['gap_ratio']:.1f}x normal spacing)")
        print(f"  Y-gap: {gap['y_gap']:.4f}")
        print()
else:
    print("\nNo significant discontinuities found in convex hull.")

if corner_vertices:
    print(f"\nCORNER VERTICES (not on original curve): {len(corner_vertices)}")
    print("-" * 50)
    for i, corner in enumerate(corner_vertices):
        hx, hy = corner['hull_vertex']
        cx, cy = corner['closest_curve_point']
        print(f"Corner {i+1}:")
        print(f"  Hull vertex: x={hx:.4f}, G(x)={hy:.4f}")
        print(f"  Nearest curve point: x={cx:.4f}, G(x)={cy:.4f}")
        print(f"  Differences: Δx={corner['x_difference']:.6f}, Δy={corner['y_difference']:.4f}")
        print()

# =====================================================
# 8. ENHANCED PLOTTING
# =====================================================
plt.figure(figsize=(10,6))

# Plot original curve
plt.plot(x_cumulative, cumulative_integral, 
         label='Cumulative Integral $G(x)$', color='orange', alpha=0.7, linewidth=1)

# Plot convex hull
plt.plot(hull_sorted[:, 0], hull_sorted[:, 1], 
         'b--', label='Convex Hull', linewidth=2)

# Plot all hull vertices
plt.scatter(hull_sorted[:, 0], hull_sorted[:, 1], 
            color='blue', s=30, label='Hull Vertices', zorder=5)

# Highlight discontinuous segments
if discontinuous_segments:
    for i, (start, end) in enumerate(discontinuous_segments):
        plt.plot([start[0], end[0]], [start[1], end[1]], 
                'r-', linewidth=3, alpha=0.8, 
                label='Discontinuous Segment' if i == 0 else "")
        plt.scatter([start[0], end[0]], [start[1], end[1]], 
                   color='red', s=60, marker='s', zorder=10)

# Highlight corner vertices
if corner_vertices:
    corner_x = [c['hull_vertex'][0] for c in corner_vertices]
    corner_y = [c['hull_vertex'][1] for c in corner_vertices]
    plt.scatter(corner_x, corner_y, 
               color='green', s=80, marker='^', 
               label='Corner Vertices', zorder=15)

plt.xlabel('x')
plt.ylabel('G(x)')
plt.title('Cumulative Integral and Its Convex Hull\nwith Discontinuity Analysis')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# =====================================================
# 9. SUMMARY STATISTICS
# =====================================================
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Data points: {num_points}")
print(f"Hull vertices: {len(hull_sorted)}")
print(f"Discontinuous segments: {len(discontinuous_segments)}")
print(f"Corner vertices: {len(corner_vertices)}")
print(f"Hull efficiency: {len(hull_sorted)/num_points*100:.2f}% of original points")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# If scipy is available, use it; otherwise use a fallback convex-hull implementation
try:
    from scipy.spatial import ConvexHull
    use_scipy = True
except ImportError:
    use_scipy = False

# =====================================================
# 1. CONFIGURATION
# =====================================================
# Polynomial coefficients (highest to lowest degree)
coefficients = coeff_600  # You'll need to define this
x0, x_end = 0, 4096
y0 = -234541.768112556427  # Initial offset for cumulative integral

# =====================================================
# 2. POLYNOMIAL FUNCTION
# =====================================================
def polynomial(x):
    return np.polyval(coefficients, x)

# =====================================================
# 3. BUILD CUMULATIVE INTEGRAL
# =====================================================
# Discretize x over 8000 points
num_points = 8000
x_cumulative = np.linspace(x0, x_end, num_points)
dx = x_cumulative[1] - x_cumulative[0]

# Evaluate polynomial at each x
mu_vals = polynomial(x_cumulative)

# Compute cumulative integral via a simple Riemann sum
cumulative_integral = y0 + np.cumsum(mu_vals) * dx

# =====================================================
# 4. CONVEX HULL COMPUTATION
# =====================================================
# Stack (x, G(x)) pairs into an (N, 2) array
points = np.column_stack((x_cumulative, cumulative_integral))

if use_scipy:
    # Use scipy.spatial.ConvexHull if available
    hull = ConvexHull(points)
    hull_vertices = points[hull.vertices]
else:
    # Fallback: monotonic chain algorithm (a.k.a. Andrew's algorithm)
    def monotonic_chain(pts):
        """Return convex hull of a set of 2D points using the Monotonic Chain algorithm."""
        # Sort by x, then by y
        pts = sorted(pts.tolist())
        if len(pts) <= 1:
            return np.array(pts)
        
        def cross(o, a, b):
            return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])
        
        lower = []
        for p in pts:
            while len(lower) >= 2 and cross(lower[-2], lower[-1], p) <= 0:
                lower.pop()
            lower.append(p)
        
        upper = []
        for p in reversed(pts):
            while len(upper) >= 2 and cross(upper[-2], upper[-1], p) <= 0:
                upper.pop()
            upper.append(p)
        
        # Concatenate lower and upper hull, excluding last point of each to avoid duplication
        hull_pts = np.array(lower[:-1] + upper[:-1])
        return hull_pts
    
    hull_vertices = monotonic_chain(points)

# Sort hull vertices by x-coordinate for analysis
hull_sorted = hull_vertices[np.argsort(hull_vertices[:, 0])]

# =====================================================
# 5. FIND DISCONTINUOUS VERTICES
# =====================================================
def find_discontinuous_vertices(hull_vertices, original_dx, threshold_multiplier=5):
    """
    Find vertices in the convex hull that represent discontinuities.
    
    Parameters:
    - hull_vertices: sorted hull vertices (N, 2) array
    - original_dx: spacing between original data points
    - threshold_multiplier: multiplier for dx to determine what constitutes a gap
    
    Returns:
    - discontinuous_segments: list of tuples (start_vertex, end_vertex) for each gap
    - gap_info: list of dictionaries with gap information
    """
    if len(hull_vertices) < 2:
        return [], []
    
    threshold = threshold_multiplier * original_dx
    discontinuous_segments = []
    gap_info = []
    
    for i in range(len(hull_vertices) - 1):
        x1, y1 = hull_vertices[i]
        x2, y2 = hull_vertices[i + 1]
        
        dx_gap = x2 - x1
        
        if dx_gap > threshold:
            discontinuous_segments.append((hull_vertices[i], hull_vertices[i + 1]))
            gap_info.append({
                'start_point': (x1, y1),
                'end_point': (x2, y2),
                'x_gap': dx_gap,
                'y_gap': y2 - y1,
                'gap_ratio': dx_gap / original_dx,
                'segment_index': i
            })
    
    return discontinuous_segments, gap_info

# Find discontinuous vertices
discontinuous_segments, gap_info = find_discontinuous_vertices(hull_sorted, dx)

# =====================================================
# 6. ALTERNATIVE METHOD: FIND VERTICES NOT ON ORIGINAL CURVE
# =====================================================
def find_vertices_off_curve(hull_vertices, x_original, y_original, tolerance=1e-6):
    """
    Find hull vertices that don't lie close to the original curve.
    This identifies vertices that are "corners" of the convex hull.
    """
    vertices_off_curve = []
    
    for hx, hy in hull_vertices:
        # Find closest x-value in original data
        closest_idx = np.argmin(np.abs(x_original - hx))
        closest_x = x_original[closest_idx]
        closest_y = y_original[closest_idx]
        
        # Check if hull vertex is significantly different from curve
        x_diff = abs(hx - closest_x)
        y_diff = abs(hy - closest_y)
        
        if x_diff > tolerance or y_diff > tolerance:
            vertices_off_curve.append({
                'hull_vertex': (hx, hy),
                'closest_curve_point': (closest_x, closest_y),
                'x_difference': x_diff,
                'y_difference': y_diff,
                'is_corner': True
            })
    
    return vertices_off_curve

# Find vertices that represent corners (not on the original curve)
corner_vertices = find_vertices_off_curve(hull_sorted, x_cumulative, cumulative_integral)

# =====================================================
# 7. OUTPUT RESULTS
# =====================================================
print("="*60)
print("CONVEX HULL ANALYSIS")
print("="*60)

print(f"\nTotal hull vertices: {len(hull_sorted)}")
print(f"Original data spacing (dx): {dx:.6f}")

#print("\nAll Convex Hull Vertices (x, G(x)):")
#for i, (x_val, g_val) in enumerate(hull_sorted):
#    print(f"  {i:2d}: x = {x_val:10.4f},  G(x) = {g_val:12.4f}")

if discontinuous_segments:
    print(f"\nDISCONTINUOUS SEGMENTS FOUND: {len(discontinuous_segments)}")
    print("-" * 50)
    for i, gap in enumerate(gap_info):
        print(f"Gap {i+1}:")
        print(f"  From: x={gap['start_point'][0]:.4f}, G(x)={gap['start_point'][1]:.4f}")
        print(f"  To:   x={gap['end_point'][0]:.4f}, G(x)={gap['end_point'][1]:.4f}")
        print(f"  X-gap: {gap['x_gap']:.4f} (ratio: {gap['gap_ratio']:.1f}x normal spacing)")
        print(f"  Y-gap: {gap['y_gap']:.4f}")
        print()
else:
    print("\nNo significant discontinuities found in convex hull.")

if corner_vertices:
    print(f"\nCORNER VERTICES (not on original curve): {len(corner_vertices)}")
    print("-" * 50)
    for i, corner in enumerate(corner_vertices):
        hx, hy = corner['hull_vertex']
        cx, cy = corner['closest_curve_point']
        print(f"Corner {i+1}:")
        print(f"  Hull vertex: x={hx:.4f}, G(x)={hy:.4f}")
        print(f"  Nearest curve point: x={cx:.4f}, G(x)={cy:.4f}")
        print(f"  Differences: Δx={corner['x_difference']:.6f}, Δy={corner['y_difference']:.4f}")
        print()

# =====================================================
# 8. ENHANCED PLOTTING
# =====================================================
plt.figure(figsize=(10,6))

# Plot original curve
plt.plot(x_cumulative, cumulative_integral, 
         label='Cumulative Integral $G(x)$', color='orange', alpha=0.7, linewidth=1)

# Plot convex hull
plt.plot(hull_sorted[:, 0], hull_sorted[:, 1], 
         'b--', label='Convex Hull', linewidth=2)

# Plot all hull vertices
plt.scatter(hull_sorted[:, 0], hull_sorted[:, 1], 
            color='blue', s=30, label='Hull Vertices', zorder=5)

# Highlight discontinuous segments
if discontinuous_segments:
    for i, (start, end) in enumerate(discontinuous_segments):
        plt.plot([start[0], end[0]], [start[1], end[1]], 
                'r-', linewidth=3, alpha=0.8, 
                label='Discontinuous Segment' if i == 0 else "")
        plt.scatter([start[0], end[0]], [start[1], end[1]], 
                   color='red', s=60, marker='s', zorder=10)

# Highlight corner vertices
if corner_vertices:
    corner_x = [c['hull_vertex'][0] for c in corner_vertices]
    corner_y = [c['hull_vertex'][1] for c in corner_vertices]
    plt.scatter(corner_x, corner_y, 
               color='green', s=80, marker='^', 
               label='Corner Vertices', zorder=15)

plt.xlabel('x')
plt.ylabel('G(x)')
plt.title('Cumulative Integral and Its Convex Hull\nwith Discontinuity Analysis')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# =====================================================
# 9. SUMMARY STATISTICS
# =====================================================
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Data points: {num_points}")
print(f"Hull vertices: {len(hull_sorted)}")
print(f"Discontinuous segments: {len(discontinuous_segments)}")
print(f"Corner vertices: {len(corner_vertices)}")
print(f"Hull efficiency: {len(hull_sorted)/num_points*100:.2f}% of original points")

### 300K

In [ ]:
#Nf_ls_300, mu_ls_300 = plot_loop(300, 200, -2, 0.002, 1, 24)
#Nf_ls_300_2, mu_ls_300_2 = plot_loop(300, 200, -2, 0.002, 2, 82)
#Nf_ls_300_3, mu_ls_300_3 = plot_loop(300, 200, -2, 0.002, 3, 82)
Nf_ls_300_4, mu_ls_300_4 = plot_loop(300, 200, -2, 0.002, 4, 82)
Nf_ls_300_5, mu_ls_300_5 = plot_loop(300, 200, -2, 0.002, 5, 82)
Nf_ls_300_6, mu_ls_300_6 = plot_loop(300, 200, -2, 0.002, 6, 82)
Nf_ls_300_7, mu_ls_300_7 = plot_loop(300, 200, -2, 0.002, 7, 82)
Nf_ls_300_8, mu_ls_300_8 = plot_loop(300, 175, -2, 0.002, 8, 82)
Nf_ls_300_8, mu_ls_300_8 = plot_loop(300, 175, -2, 0.002, 8, 82)

In [ ]:
def plot_loop(temp, Nt0, mu_b, fi, serie, curr_N):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,5))
    plt.subplots_adjust(wspace=0.35)
    temp = temp
    fi = fi
    Nf_ls = []
    mu_ls = []
    
    for i in range(1, curr_N):
        Nt = Nt0 + (i-1) * 50
        
        # Load atoms log file
        file = f'./production_run/free_E/test/Nt_{temp}/{serie}/atoms_log_{i}.txt'
        try:
            log_data = parse_print_file(file)
            step = np.array(log_data['Step'])
            n_Na = np.array(log_data['atoms']) - 28672  # Adjust baseline as needed
            ax1.plot(step, n_Na, label=f'Nf_{i}={n_Na[-1]}, Nt={Nt}')
        except Exception as e:
            print(f"Warning: Could not load atoms log file {file}: {e}")
            continue
        
        # Load convergence file with error handling
        file = f'./production_run/free_E/test/Nt_{temp}/{serie}/{temp}_converge_{i}.txt'
        try:
            data = np.loadtxt(file, skiprows=2)
            
            # Check if data is empty
            if data.size == 0:
                print(f"Warning: Empty data in file {file}, skipping iteration {i}")
                continue
            
            # Handle different data shapes
            if data.ndim == 1:
                # 1D array - single row of data
                if len(data) >= 2:  # Make sure we have at least 2 elements
                    ts = data[0]
                    Nf = data[-1] - 28672
                    Nf = data[:].mean - 28672
                else:
                    print(f"Warning: Insufficient data in file {file}, skipping iteration {i}")
                    continue
            else:
                # 2D array - multiple rows
                if data.shape[0] > 0 and data.shape[1] >= 2:  # Check dimensions
                    ts = data[-1, 0] 
                    Nf = data[-1, -1] - 28672
                    Nf = data[:].mean - 28672
                else:
                    print(f"Warning: Invalid data shape in file {file}, skipping iteration {i}")
                    continue
            
            # Calculate chemical potential
            mu = mu_b - 2 * fi * (Nf - Nt)
            
            # Store results
            Nf_ls.append(Nf)
            mu_ls.append(mu)
            
            # Add scatter points and text
            ax1.scatter(ts, Nf)
            ax1.text(ts, Nf-5, s=f'{Nf}', fontsize=8)
            ax2.scatter(Nf, mu)
            
        except FileNotFoundError:
            print(f"Warning: File {file} not found, skipping iteration {i}")
            continue
        except Exception as e:
            print(f"Warning: Error loading file {file}: {e}, skipping iteration {i}")
            continue
    
    # Only plot if we have data
    if len(Nf_ls) > 0:
        ax2.plot(Nf_ls, mu_ls)
    else:
        print("Warning: No valid data found for plotting")
        
    # Set plot properties
    plt.suptitle(f'{temp} K mu_bias = {mu_b}')
    ax1.grid()
    ax1.set_xlabel('Timestep (0.4 fs/step)')
    ax1.set_ylabel('Number of Na+')
    ax2.set_xlabel('Number of Na+')
    ax2.set_ylabel('Chemical potential (eV)')
    plt.tight_layout()
    
    # Save figure
    plt.savefig(f'./production_run/free_E/test/loop_{temp}.png')
    
    return Nf_ls, mu_ls

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

def plot_loop(temp, Nt0, mu_b, fi, step_size, serie, curr_N, vac=False):

    # Create figure with two subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    plt.subplots_adjust(wspace=0.35)
    
    # Initialize lists to store results
    Nf_ls = []
    mu_ls = []
    
    # Loop through simulation runs
    for i in range(1, curr_N):
        #print(f"Processing run {i}/{curr_N-1}")
        
        # Calculate target number of atoms for this run
        Nt = Nt0 + (i-1) * step_size

        if vac == False:
            # Define file paths
            atoms_file = f'./production_run/free_E/test/Nt_{temp}/{serie}/atoms_log_{i}.txt'
            converge_file = f'./production_run/free_E/test/Nt_{temp}/{serie}/{temp}_converge_{i}.txt'
        else:
            atoms_file = f'./production_run/free_E/test/Nt_{temp}/{serie}/atoms_log_{i}.txt'
            converge_file = f'./production_run/free_E/test/Nt_{temp}/{serie}/{temp}_converge_{i}.txt'
            
        # Check and process atoms log file
        if not os.path.exists(atoms_file) or os.path.getsize(atoms_file) == 0:
            print(f"Warning: Skipped empty or missing file: {atoms_file}")
            continue
        
        try:
            # Parse atom count data over time
            log_data = parse_print_file(atoms_file)
            timesteps = np.array(log_data['Step'])
            n_Na = np.array(log_data['atoms']) - 28672  # Subtract baseline
            
            # Plot time evolution of atom count
            ax1.plot(timesteps, n_Na, label=f'Run {i}: Nf={n_Na[-1]}, Nt={Nt}')
            ax1.axhline(y=Nt, color='black', linestyle='-', alpha=0.5, linewidth=0.8)
            
        except Exception as e:
            print(f"Error processing {atoms_file}: {e}")
            continue
        
        # Check and process convergence file
        if not os.path.exists(converge_file) or os.path.getsize(converge_file) == 0:
            print(f"Warning: Skipped empty or missing file: {converge_file}")
            continue
        
        try:
            # Load convergence data
            data = np.loadtxt(converge_file, skiprows=2)
            if data.size == 0:
                print(f"Warning: No data in file: {converge_file}")
                continue
            
            # Extract final timestep and atom count
            if data.ndim == 1:
                ts_final = data[0] if len(data) > 0 else 0
                #Nf = data[-1] - 28672 if len(data) > 0 else 0
                Nf = n_Na[:].mean()
            else:
                ts_final = data[-1, 0]
                #Nf = data[-1, -1] - 28672
                Nf = n_Na[:].mean()
            
            # Calculate chemical potential
            mu = mu_b - 2 * fi * (Nf - Nt)
            
            # Store results
            Nf_ls.append(Nf)
            mu_ls.append(mu)
            
            print(f"{i}  Final: ts={ts_final}, Nf={Nf}, Nt={Nt}, mu={mu:.4f}")
            
            # Add markers and annotations to plots
            ax1.scatter(ts_final, Nf, color='red', s=30, zorder=5)
            ax1.annotate(f'{Nf}', (ts_final, Nf), xytext=(5, 5), 
                        textcoords='offset points', fontsize=8, 
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
            
            ax2.scatter(Nf, mu, s=50, alpha=0.7)
            
        except Exception as e:
            print(f"Error processing {converge_file}: {e}")
            continue
    
    # Plot chemical potential trend
    if len(Nf_ls) > 1:
        ax2.plot(Nf_ls, mu_ls, 'b-', linewidth=2, alpha=0.7)
    
    # Format plots
    ax1.grid(True, alpha=0.3)
    ax1.set_xlabel('Timestep (0.4 fs/step)')
    ax1.set_ylabel('Number of Na+ (relative to baseline)')
    ax1.set_title(f'Atom Count Evolution - {temp} K')
    #ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    
    ax2.grid(True, alpha=0.3)
    ax2.set_xlabel('Final Number of Na+ (relative to baseline)')
    ax2.set_ylabel('Chemical Potential (eV)')
    ax2.set_title(f'Chemical Potential vs Final Count - {temp} K')
    
    # Add overall title
    fig.suptitle(f'Simulation Analysis: {temp} K, μ_bias = {mu_b} eV', fontsize=14)
    
    plt.tight_layout()
    
    # Save figure
    output_dir = f'./Nt_{temp}/{serie}/'
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(f'{output_dir}loop_{temp}.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    #print(f"\nSummary:")
    #print(f"  Final atom counts: {Nf_ls}")
    #print(f"  Chemical potentials: {[f'{mu:.4f}' for mu in mu_ls]}")
    
    return Nf_ls, mu_ls

# Additional helper function for better error handling
def safe_file_check(filepath):
    """Safely check if file exists and has content."""
    return os.path.exists(filepath) and os.path.getsize(filepath) > 0

In [ ]:
Nf_ls_300_8, mu_ls_300_8 = plot_loop(300, 175, -2, 0.002, 50, 8, 82)
Nf_ls_300_9, mu_ls_300_9 = plot_loop(300, 175, -2, 0.002, 50, 9, 82)

In [ ]:
plt.figure()
plt.plot(Nf_ls_300_8, mu_ls_300_8)
plt.plot(Nf_ls_300_9, mu_ls_300_9)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Configuration
temp, Nt0, mu_b, fi, curr_N = 300, 200, -2, 0.002, 82
series_list = [4, 5, 6]

# Create figure with 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(singlecol*3, singlecol), sharey=True)
plt.subplots_adjust(wspace=0.2)

cmap = plt.get_cmap("viridis")
for ax in axes:
    ax.ticklabel_format(style='sci', axis='x', scilimits=(0,0))
# Loop through each series and its corresponding axis
for idx, serie in enumerate(series_list):
    ax = axes[idx]
    
    for i in range(1, curr_N):
        Nt = Nt0 + (i-1) * 50
        color = cmap(i / curr_N)
        
        # --- Load atoms log file ---
        file_atoms = f'./production_run/free_E/test/Nt_{temp}/{serie}/atoms_log_{i}.txt'
        try:
            log_data = parse_print_file(file_atoms)
            step = np.array(log_data['Step'])
            n_Na = np.array(log_data['atoms']) - 28672  
            ax.plot(step, n_Na, color=color, alpha=0.7)
        except Exception:
            continue
        
        # --- Load convergence file for scatter points ---
        file_conv = f'./production_run/free_E/test/Nt_{temp}/{serie}/{temp}_converge_{i}.txt'
        try:
            data = np.loadtxt(file_conv, skiprows=2)
            if data.size > 0:
                if data.ndim == 1 and len(data) >= 2:
                    ts, Nf = data[0], data[-1] - 28672
                elif data.ndim > 1:
                    ts, Nf = data[-1, 0], data[-1, -1] - 28672
                ax.scatter(ts, Nf, color=color, s=10)
        except Exception:
            continue

    # Subplot formatting
    ax.set_title(f'Sampling iteration = {serie-3}')
    ax.set_xlabel('Timestep (0.4 fs/step)')
    if idx == 0:
        ax.set_ylabel('Number of Na+')

plt.suptitle(f'Temperature: {temp} K | mu_bias: {mu_b}', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure()
error = 0.002*2*5
#plt.plot(Nf_ls_300[5:], mu_ls_300[5:])
#plt.fill_between(Nf_ls_300[5:], np.array(mu_ls_300[5:]) - error,  np.array(mu_ls_300[5:]) + error,
#                 alpha=0.15, label='Error range')
#plt.plot(Nf_ls_300_2, mu_ls_300_2)
#plt.fill_between(Nf_ls_300_2, np.array(mu_ls_300_2) - error,  np.array(mu_ls_300_2) + error,
#                 alpha=0.15, label='Error range')
#plt.plot(Nf_ls_300_3, mu_ls_300_3)
#plt.fill_between(Nf_ls_300_3, np.array(mu_ls_300_3) - error,  np.array(mu_ls_300_3) + error,
#                 alpha=0.15, label='Error range')
#plt.plot(Nf_ls_300_4, mu_ls_300_4)
plt.plot(Nf_ls_300_5, mu_ls_300_5)
plt.plot(Nf_ls_300_6, mu_ls_300_6)
plt.plot(Nf_ls_300_7, mu_ls_300_7)
plt.plot(Nf_ls_300_8, mu_ls_300_8)

plt.ylabel('Chemical potential (eV)')
plt.xlabel('Number of Na')
plt.title('300K')
plt.tight_layout()

In [ ]:
x = np.concatenate([Nf_ls_300_5, Nf_ls_300_6])
y = np.concatenate([mu_ls_300_5, mu_ls_300_6])
sort_indices = np.argsort(x)
coeff_300 = fit_chemical_potential(x,y,11)

In [ ]:
i=45
temp, serie = 300, 4
file_1 = f'./production_run/free_E/test/Nt_{temp}/{serie}/atoms_log_{i}.txt'
log_data_1 = parse_print_file(file_1)
step_1 = np.array(log_data_1['Step'])
n_Na_1 = np.array(log_data_1['atoms']) - 28672  # Adjust baseline as needed

temp, serie = 300, 5
file_2 = f'./production_run/free_E/test/Nt_{temp}/{serie}/atoms_log_{i}.txt'
log_data_2 = parse_print_file(file_2)
step_2 = np.array(log_data_2['Step'])
n_Na_2 = np.array(log_data_2['atoms']) - 28672

temp, serie = 300, 6
file_3 = f'./production_run/free_E/test/Nt_{temp}/{serie}/atoms_log_{i}.txt'
log_data_3 = parse_print_file(file_3)
step_3 = np.array(log_data_3['Step'])
n_Na_3 = np.array(log_data_3['atoms']) - 28672

n_Na = np.concatenate([n_Na_1, n_Na_2, n_Na_3])
print(f'Data number {i}')
plt.figure()
plt.plot(n_Na)
plt.plot(np.concatenate([n_Na_1, n_Na_2]))
plt.plot(n_Na_1)

In [ ]:
temp, dir = 300, 6
for i in range(1,2):
    index = i
    plot_log_free(f'./production_run/free_E/test/Nt_{temp}/{dir}/loop_bias_{dir}.log_{index}')

In [ ]:
temp, dir = 300, 7
for i in range(2,3):
    index = i
    plot_log_free(f'./production_run/free_E/test/Nt_{temp}/{dir}/loop_bias_{dir}.log_{index}')

In [ ]:
temp, dir = 300, 6
for i in range(1,2):
    index = i
    plot_log_free(f'./production_run/free_E/test/Nt_{temp}/{dir}/loop_bias_{dir}.log_{index}')

In [ ]:
x = np.concatenate([Nf_ls_600[1:], Nf_ls_600_2, Nf_ls_600_3])
y = np.concatenate([mu_ls_600[1:], mu_ls_600_2, mu_ls_600_3])
sort_indices = np.argsort(x)
#x = x[sort_indices]
#y = y[sort_indices]
#fit_chemical_potential(x,y,7)
#fit_chemical_potential(x,y,8)
coeff_300 = fit_chemical_potential(x,y,9)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate, optimize
import sympy as sp
import warnings
warnings.filterwarnings('ignore')

# =====================================================
# 1. CONFIGURATION
# =====================================================

# Polynomial coefficients (from highest to lowest degree)
coefficients = coeff_300
x0, x_end = 0, 4096
y0 = -234541.768112556427  # Initial condition

# =====================================================
# 2. POLYNOMIAL DEFINITION
# =====================================================

def polynomial(x):
    return np.polyval(coefficients, x)

# =====================================================
# 3. INTEGRATION METHODS
# =====================================================

def integrate_numerically():
    result, error = integrate.quad(polynomial, x0, x_end)
    return result, error

def integrate_analytically():
    x = sp.Symbol('x')
    expr = sum(c * x**i for i, c in enumerate(reversed(coefficients)))
    antiderivative = sp.integrate(expr, x)
    result = float(antiderivative.subs(x, x_end) - antiderivative.subs(x, x0))
    return result, antiderivative

def integrate_manually():
    powers = list(range(8, -1, -1))
    integrated_coeffs = [(coef / (p + 1), p + 1) for coef, p in zip(coefficients, powers)]
    
    def F(x):
        return sum(c * x**p for c, p in integrated_coeffs)

    result = F(x_end) - F(x0)
    return result, integrated_coeffs

# =====================================================
# 4. COMPUTE AND COMPARE RESULTS
# =====================================================

numerical_result, numerical_error = integrate_numerically()
analytical_result, antiderivative_expr = integrate_analytically()
manual_result, integrated_coeffs = integrate_manually()

print("="*60)
print("INTEGRATION RESULTS")
print("="*60)
print(f"Numerical Integration:  {numerical_result:.10e} (±{numerical_error:.2e})")
print(f"Analytical Integration: {analytical_result:.10e}")
print(f"Manual Integration:     {manual_result:.10e}")
print(f"Difference Num-Ana:     {abs(numerical_result - analytical_result):.2e}")
print(f"Difference Num-Man:     {abs(numerical_result - manual_result):.2e}")

# =====================================================
# 5. INITIAL VALUE PROBLEM
# =====================================================

y4096 = y0 + numerical_result
print("\n" + "="*60)
print("INITIAL VALUE PROBLEM")
print("="*60)
print(f"y(4096) = y(0) + ∫₀⁴⁰⁹⁶ f(x) dx = {y0:.6e} + {numerical_result:.6e} = {y4096:.6e}")

# =====================================================
# 6. CUMULATIVE INTEGRAL & DIFFERENCE
# =====================================================

min = polynomial(np.arange(2000,3000))

x_cumulative = np.linspace(x0, x_end, 8000)
dx = x_cumulative[1] - x_cumulative[0]
integrand_vals = polynomial(x_cumulative)
cumulative_integral = y0 + np.cumsum(integrand_vals) * dx

min_x2, min_x1 = 2361.1271, 201.7532
min_y2, min_y1 = -238705.2597, -234911.9004

slope = (min_y2 - min_y1) / (min_x2 - min_x1)
intercept = min_y1 - slope * min_x1

def linear_function(x):
    return slope * x + intercept

difference = cumulative_integral - linear_function(x_cumulative)
difference_derivative = np.gradient(difference, dx)
linear_slope = slope 

# =====================================================
# 7. FIND ZEROS OF DIFFERENCE DERIVATIVE
# =====================================================

def derivative_equation(x):
    return polynomial(x) - linear_slope

zero_points = []
starting_points = np.linspace(x0, x_end, 50)
for start in starting_points:
    root = optimize.fsolve(derivative_equation, start)[0]
    if abs(derivative_equation(root)) < 1e-8 and x0 <= root <= x_end:
        if all(abs(root - zp) > 10 for zp in zero_points):
            zero_points.append(root)
zero_points = sorted(zero_points)

print("\n" + "="*60)
print("ZEROS OF DERIVATIVE (CRITICAL POINTS OF THE DIFFERENCE)")
print("="*60)
for i, zp in enumerate(zero_points):
    idx = np.argmin(np.abs(x_cumulative - zp))
    print(f"Point {i+1}: x = {zp:.2f}, diff = {difference[idx]:.2e}, f(x) = {polynomial(zp):.2e}")

# =====================================================
# 8. ANALYZE NATURE OF CRITICAL POINTS
# =====================================================

print("\n" + "="*60)
print("CRITICAL POINT CLASSIFICATION")
print("="*60)
for i, zp in enumerate(zero_points):
    h = 1.0
    if x0 + h <= zp <= x_end - h:
        d2 = (derivative_equation(zp + h) - derivative_equation(zp - h)) / (2 * h)
        nature = "Min" if d2 > 1e-6 else "Max" if d2 < -1e-6 else "Inflection"
        print(f"x = {zp:.2f}, 2nd derivative ≈ {d2:.2e}, Type: {nature}")

# =====================================================
# 9. VISUALIZATION
# =====================================================

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10))

# Plot 1: Original polynomial
ax1.plot(x_cumulative, polynomial(x_cumulative), 'b-', label='Fitted chemical potential: mu(x)')
#ax1.axhline(min_y,color='r', linestyle='--',)
ax1.plot(min_x1, polynomial(min_x1), 'ro')
ax1.plot(min_x2, polynomial(min_x2), 'ro')
ax1.set_ylabel('Chemical potential (eV)')
ax2.set_ylabel('Energy (eV)')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Plot 2: Cumulative integral and linear interpolation
ax2.plot(x_cumulative, cumulative_integral, 'r-', label='G(x) = G(x₀) + ∫₀ˣ mu(x)dx')
ax2.plot(min_x1, min_y1, 'ro')
ax2.plot(min_x2, min_y2, 'ro')
ax2.plot(x_cumulative, linear_function(x_cumulative), 'b--', label='L(x) = Cotangent line')
ax2.set_title('Free Energy')
ax2.set_xlabel('Energy (eV)')
ax2.grid(True, alpha=0.3)
ax2.legend()

ax22=ax2.twinx()

# Plot 3: Difference function
ax22.plot(x_cumulative, difference, 'g-')#, label='Difference = G(x) - L(x)')
ax22.axhline(0, color='k', linestyle='--', alpha=0.5)
for zp in zero_points:
    idx = np.argmin(np.abs(x_cumulative - zp))
    ax22.plot(zp, difference[idx], 'ro')
ax2.set_xlabel('Number of Sodium')
#ax3.set_title('Difference Function')
#ax3.grid(True, alpha=0.3)
ax22.set_ylabel('G(x) - L(x) (eV)', color='green')
plt.tight_layout()
# Plot 4: Derivative of difference

# =====================================================
# 10. STATISTICS
# =====================================================

print("\n" + "="*60)
print("STATISTICS OF DIFFERENCE")
print("="*60)
print(f"Max Difference: {np.max(difference):.2e} at x = {x_cumulative[np.argmax(difference)]:.1f}")
print(f"Min Difference: {np.min(difference):.2e} at x = {x_cumulative[np.argmin(difference)]:.1f}")
print(f"Mean: {np.mean(difference):.2e}, Std: {np.std(difference):.2e}, Range: {np.ptp(difference):.2e}")

# =====================================================
# 11. ANTIDERIVATIVE FUNCTION
# =====================================================

def antiderivative_function(x):
    return sum(c * x**p for c, p in integrated_coeffs)

print("\n" + "="*60)
print("ANTIDERIVATIVE FUNCTION")
print("="*60)
print(f"F(0) = {antiderivative_function(0):.10e}")
print(f"F(4096) = {antiderivative_function(4096):.10e}")
print(f"F(4096) - F(0) = {antiderivative_function(4096) - antiderivative_function(0):.10e}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# If scipy is available, use it; otherwise use a fallback convex-hull implementation
try:
    from scipy.spatial import ConvexHull
    use_scipy = True
except ImportError:
    use_scipy = False

# =====================================================
# 1. CONFIGURATION
# =====================================================
# Polynomial coefficients (highest to lowest degree)
coefficients = coeff_300  # You'll need to define this
x0, x_end = 0, 4096
y0 = -234541.768112556427  # Initial offset for cumulative integral

# =====================================================
# 2. POLYNOMIAL FUNCTION
# =====================================================
def polynomial(x):
    return np.polyval(coefficients, x)

# =====================================================
# 3. BUILD CUMULATIVE INTEGRAL
# =====================================================
# Discretize x over 8000 points
num_points = 8000
x_cumulative = np.linspace(x0, x_end, num_points)
dx = x_cumulative[1] - x_cumulative[0]

# Evaluate polynomial at each x
mu_vals = polynomial(x_cumulative)

# Compute cumulative integral via a simple Riemann sum
cumulative_integral = y0 + np.cumsum(mu_vals) * dx

# =====================================================
# 4. CONVEX HULL COMPUTATION
# =====================================================
# Stack (x, G(x)) pairs into an (N, 2) array
points = np.column_stack((x_cumulative, cumulative_integral))

if use_scipy:
    # Use scipy.spatial.ConvexHull if available
    hull = ConvexHull(points)
    hull_vertices = points[hull.vertices]
else:
    # Fallback: monotonic chain algorithm (a.k.a. Andrew's algorithm)
    def monotonic_chain(pts):
        """Return convex hull of a set of 2D points using the Monotonic Chain algorithm."""
        # Sort by x, then by y
        pts = sorted(pts.tolist())
        if len(pts) <= 1:
            return np.array(pts)
        
        def cross(o, a, b):
            return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])
        
        lower = []
        for p in pts:
            while len(lower) >= 2 and cross(lower[-2], lower[-1], p) <= 0:
                lower.pop()
            lower.append(p)
        
        upper = []
        for p in reversed(pts):
            while len(upper) >= 2 and cross(upper[-2], upper[-1], p) <= 0:
                upper.pop()
            upper.append(p)
        
        # Concatenate lower and upper hull, excluding last point of each to avoid duplication
        hull_pts = np.array(lower[:-1] + upper[:-1])
        return hull_pts
    
    hull_vertices = monotonic_chain(points)

# Sort hull vertices by x-coordinate for analysis
hull_sorted = hull_vertices[np.argsort(hull_vertices[:, 0])]

# =====================================================
# 5. FIND DISCONTINUOUS VERTICES
# =====================================================
def find_discontinuous_vertices(hull_vertices, original_dx, threshold_multiplier=5):
    """
    Find vertices in the convex hull that represent discontinuities.
    
    Parameters:
    - hull_vertices: sorted hull vertices (N, 2) array
    - original_dx: spacing between original data points
    - threshold_multiplier: multiplier for dx to determine what constitutes a gap
    
    Returns:
    - discontinuous_segments: list of tuples (start_vertex, end_vertex) for each gap
    - gap_info: list of dictionaries with gap information
    """
    if len(hull_vertices) < 2:
        return [], []
    
    threshold = threshold_multiplier * original_dx
    discontinuous_segments = []
    gap_info = []
    
    for i in range(len(hull_vertices) - 1):
        x1, y1 = hull_vertices[i]
        x2, y2 = hull_vertices[i + 1]
        
        dx_gap = x2 - x1
        
        if dx_gap > threshold:
            discontinuous_segments.append((hull_vertices[i], hull_vertices[i + 1]))
            gap_info.append({
                'start_point': (x1, y1),
                'end_point': (x2, y2),
                'x_gap': dx_gap,
                'y_gap': y2 - y1,
                'gap_ratio': dx_gap / original_dx,
                'segment_index': i
            })
    
    return discontinuous_segments, gap_info

# Find discontinuous vertices
discontinuous_segments, gap_info = find_discontinuous_vertices(hull_sorted, dx)

# =====================================================
# 6. ALTERNATIVE METHOD: FIND VERTICES NOT ON ORIGINAL CURVE
# =====================================================
def find_vertices_off_curve(hull_vertices, x_original, y_original, tolerance=1e-6):
    """
    Find hull vertices that don't lie close to the original curve.
    This identifies vertices that are "corners" of the convex hull.
    """
    vertices_off_curve = []
    
    for hx, hy in hull_vertices:
        # Find closest x-value in original data
        closest_idx = np.argmin(np.abs(x_original - hx))
        closest_x = x_original[closest_idx]
        closest_y = y_original[closest_idx]
        
        # Check if hull vertex is significantly different from curve
        x_diff = abs(hx - closest_x)
        y_diff = abs(hy - closest_y)
        
        if x_diff > tolerance or y_diff > tolerance:
            vertices_off_curve.append({
                'hull_vertex': (hx, hy),
                'closest_curve_point': (closest_x, closest_y),
                'x_difference': x_diff,
                'y_difference': y_diff,
                'is_corner': True
            })
    
    return vertices_off_curve

# Find vertices that represent corners (not on the original curve)
corner_vertices = find_vertices_off_curve(hull_sorted, x_cumulative, cumulative_integral)

# =====================================================
# 7. OUTPUT RESULTS
# =====================================================
print("="*60)
print("CONVEX HULL ANALYSIS")
print("="*60)

print(f"\nTotal hull vertices: {len(hull_sorted)}")
print(f"Original data spacing (dx): {dx:.6f}")

#print("\nAll Convex Hull Vertices (x, G(x)):")
#for i, (x_val, g_val) in enumerate(hull_sorted):
#    print(f"  {i:2d}: x = {x_val:10.4f},  G(x) = {g_val:12.4f}")

if discontinuous_segments:
    print(f"\nDISCONTINUOUS SEGMENTS FOUND: {len(discontinuous_segments)}")
    print("-" * 50)
    for i, gap in enumerate(gap_info):
        print(f"Gap {i+1}:")
        print(f"  From: x={gap['start_point'][0]:.4f}, G(x)={gap['start_point'][1]:.4f}")
        print(f"  To:   x={gap['end_point'][0]:.4f}, G(x)={gap['end_point'][1]:.4f}")
        print(f"  X-gap: {gap['x_gap']:.4f} (ratio: {gap['gap_ratio']:.1f}x normal spacing)")
        print(f"  Y-gap: {gap['y_gap']:.4f}")
        print()
else:
    print("\nNo significant discontinuities found in convex hull.")

if corner_vertices:
    print(f"\nCORNER VERTICES (not on original curve): {len(corner_vertices)}")
    print("-" * 50)
    for i, corner in enumerate(corner_vertices):
        hx, hy = corner['hull_vertex']
        cx, cy = corner['closest_curve_point']
        print(f"Corner {i+1}:")
        print(f"  Hull vertex: x={hx:.4f}, G(x)={hy:.4f}")
        print(f"  Nearest curve point: x={cx:.4f}, G(x)={cy:.4f}")
        print(f"  Differences: Δx={corner['x_difference']:.6f}, Δy={corner['y_difference']:.4f}")
        print()

# =====================================================
# 8. ENHANCED PLOTTING
# =====================================================
plt.figure(figsize=(12, 8))

# Plot original curve
plt.plot(x_cumulative, cumulative_integral, 
         label='Cumulative Integral $G(x)$', color='orange', alpha=0.7, linewidth=1)

# Plot convex hull
plt.plot(hull_sorted[:, 0], hull_sorted[:, 1], 
         'b--', label='Convex Hull', linewidth=2)

# Plot all hull vertices
plt.scatter(hull_sorted[:, 0], hull_sorted[:, 1], 
            color='blue', s=30, label='Hull Vertices', zorder=5)

# Highlight discontinuous segments
if discontinuous_segments:
    for i, (start, end) in enumerate(discontinuous_segments):
        plt.plot([start[0], end[0]], [start[1], end[1]], 
                'r-', linewidth=3, alpha=0.8, 
                label='Discontinuous Segment' if i == 0 else "")
        plt.scatter([start[0], end[0]], [start[1], end[1]], 
                   color='red', s=60, marker='s', zorder=10)

# Highlight corner vertices
if corner_vertices:
    corner_x = [c['hull_vertex'][0] for c in corner_vertices]
    corner_y = [c['hull_vertex'][1] for c in corner_vertices]
    plt.scatter(corner_x, corner_y, 
               color='green', s=80, marker='^', 
               label='Corner Vertices', zorder=15)

plt.xlabel('x')
plt.ylabel('G(x)')
plt.title('Cumulative Integral and Its Convex Hull\nwith Discontinuity Analysis')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# =====================================================
# 9. SUMMARY STATISTICS
# =====================================================
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Data points: {num_points}")
print(f"Hull vertices: {len(hull_sorted)}")
print(f"Discontinuous segments: {len(discontinuous_segments)}")
print(f"Corner vertices: {len(corner_vertices)}")
print(f"Hull efficiency: {len(hull_sorted)/num_points*100:.2f}% of original points")

## Structure inspection

In [ ]:
import re

def parse_lammps_box(filename):
    """
    Simple LAMMPS data parser to extract only box bounds.
    Returns dict with keys: xlo, xhi, ylo, yhi, zlo, zhi, xy, xz, yz
    """
    box_params = {}
    
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
                
            # xlo xhi
            if 'xlo xhi' in line:
                parts = line.split()
                box_params['xlo'] = float(parts[0])
                box_params['xhi'] = float(parts[1])
                
            # ylo yhi  
            elif 'ylo yhi' in line:
                parts = line.split()
                box_params['ylo'] = float(parts[0])
                box_params['yhi'] = float(parts[1])
                
            # zlo zhi
            elif 'zlo zhi' in line:
                parts = line.split()
                box_params['zlo'] = float(parts[0])
                box_params['zhi'] = float(parts[1])
                
            # xy xz yz (tilt factors)
            elif 'xy xz yz' in line:
                parts = line.split()
                box_params['xy'] = float(parts[0])
                box_params['xz'] = float(parts[1])
                box_params['yz'] = float(parts[2])

            elif 'atoms' in line:
                parts = line.split()
                atoms = float(parts[0])
    
    # Set tilt factors to 0 if not found (orthogonal box)
    if 'xy' not in box_params:
        box_params['xy'] = 0.0
        box_params['xz'] = 0.0
        box_params['yz'] = 0.0
    
    return box_params, atoms

def compute_triclinic_angles(xlo, xhi, ylo, yhi, zlo, zhi, xy, xz, yz):
    """
    Compute the angles between the three axes of a triclinic simulation box
    from LAMMPS data file parameters.
    
    Returns: alpha, beta, gamma (angles in degrees)
    """
    import numpy as np
    import math
    
    # Calculate box lengths
    lx = xhi - xlo
    ly = yhi - ylo
    lz = zhi - zlo
    
    # Define the three lattice vectors
    # In LAMMPS triclinic format:
    # a = (lx, 0, 0)
    # b = (xy, ly, 0)
    # c = (xz, yz, lz)
    
    a = np.array([lx, 0, 0])
    b = np.array([xy, ly, 0])
    c = np.array([xz, yz, lz])
    
    # Calculate magnitudes
    mag_a = np.linalg.norm(a)
    mag_b = np.linalg.norm(b)
    mag_c = np.linalg.norm(c)
    
    # Calculate angles using dot product formula: cos(θ) = (u·v)/(|u||v|)
    # α is angle between b and c
    # β is angle between a and c  
    # γ is angle between a and b
    
    cos_alpha = np.dot(b, c) / (mag_b * mag_c)
    cos_beta = np.dot(a, c) / (mag_a * mag_c)
    cos_gamma = np.dot(a, b) / (mag_a * mag_b)
    
    # Convert to degrees
    alpha = math.degrees(math.acos(cos_alpha))
    beta = math.degrees(math.acos(cos_beta))
    gamma = math.degrees(math.acos(cos_gamma))
    
    return lx, ly, lz, alpha, beta, gamma

In [ ]:

for i in range(60,):
# Test with your data
    filename = f'./hydration/Nt_300/2/300loop_bias_{i}.data'
    print(filename)
    #500K_Na{i}_O_32.data'  # Replace with your file
    #filename = f'./production_run/free_E/test/full_Na_big_300K.data'
    try:
        box, atoms = parse_lammps_box(filename)
        print("Box parameters:")
        for key, value in box.items():
            print(f"{key}: {value}")
        
        # Compute angles
        lx, ly, lz, alpha, beta, gamma = compute_triclinic_angles(**box)
        print(f"\nTriclinic angles:")
        print(f"α = {alpha:.3f}°")
        print(f"β = {beta:.3f}°") 
        print(f"γ = {gamma:.3f}°")
        print(f"a = {lx:.3f} A")
        print(f"b = {ly:.3f} A")
        print(f"c = {lz:.3f} A")
        
    except FileNotFoundError:
        print("File not found - testing with sample data instead:")
        
    print(f'atoms = {(atoms-17280)/1728*2}')
    print("")
    print("")

In [ ]:
temp=300
for O in [4,16,32]:
    for Na in [4,16,28]:
        filename = f'./hydration/{temp}_diffusion_4/{temp}K_Na_{Na}_O_{O}.data'
        print(filename)
        #500K_Na{i}_O_32.data'  # Replace with your file
        #filename = f'./production_run/free_E/test/full_Na_big_300K.data'
        try:
            box, atoms = parse_lammps_box(filename)
            print("Box parameters:")
            for key, value in box.items():
                print(f"{key}: {value}")
            
            # Compute angles
            lx, ly, lz, alpha, beta, gamma = compute_triclinic_angles(**box)
            print(f"\nTriclinic angles:")
            print(f"α = {alpha:.3f}°")
            print(f"β = {beta:.3f}°") 
            print(f"γ = {gamma:.3f}°")
            print(f"a = {lx:.3f} Å")
            print(f"b = {ly:.3f} Å")
            print(f"c = {lz:.3f} Å")
            
        except FileNotFoundError:
            print("File not found - testing with sample data instead:")

        print(f'atoms = {atoms}')
        print("")
        print("")

In [ ]:
temp=300
for i in [4,6,8,10]:
    filename = f'./hydration/Nt_{temp}/1/{temp}loop_bias_{i}.data'
    print(filename)
    #500K_Na{i}_O_32.data'  # Replace with your file
    #filename = f'./production_run/free_E/test/full_Na_big_300K.data'
    try:
        box, atoms = parse_lammps_box(filename)
        print("Box parameters:")
        for key, value in box.items():
            print(f"{key}: {value}")
        
        # Compute angles
        lx, ly, lz, alpha, beta, gamma = compute_triclinic_angles(**box)
        print(f"\nTriclinic angles:")
        print(f"α = {alpha:.3f}°")
        print(f"β = {beta:.3f}°") 
        print(f"γ = {gamma:.3f}°")
        print(f"a = {lx:.3f} Å")
        print(f"b = {ly:.3f} Å")
        print(f"c = {lz:.3f} Å")
        
    except FileNotFoundError:
        print("File not found - testing with sample data instead:")

    print(f'atoms = {atoms}')
    print("")
    print("")

In [ ]:
temp=300
for i in [60,65,70]:
    filename = f'./hydration/Nt_{temp}/6/{temp}loop_bias_{i}.data'
    print(filename)
    #500K_Na{i}_O_32.data'  # Replace with your file
    #filename = f'./production_run/free_E/test/full_Na_big_300K.data'
    try:
        box, atoms = parse_lammps_box(filename)
        print("Box parameters:")
        for key, value in box.items():
            print(f"{key}: {value}")
        
        # Compute angles
        lx, ly, lz, alpha, beta, gamma = compute_triclinic_angles(**box)
        print(f"\nTriclinic angles:")
        print(f"α = {alpha:.3f}°")
        print(f"β = {beta:.3f}°") 
        print(f"γ = {gamma:.3f}°")
        print(f"a = {lx:.3f} Å")
        print(f"b = {ly:.3f} Å")
        print(f"c = {lz:.3f} Å")
        
    except FileNotFoundError:
        print("File not found - testing with sample data instead:")

    print(f'atoms = {atoms}')
    print("")
    print("")

In [ ]:

for i in [4,16,28]:
    filename = f'./hydration/RDF_lv12_E0_sf1_f0.01//Na_{i}_O_4.data'
    print(filename)
    #500K_Na{i}_O_32.data'  # Replace with your file
    #filename = f'./production_run/free_E/test/full_Na_big_300K.data'
    try:
        box, atoms = parse_lammps_box(filename)
        print("Box parameters:")
        for key, value in box.items():
            print(f"{key}: {value}")
        
        # Compute angles
        lx, ly, lz, alpha, beta, gamma = compute_triclinic_angles(**box)
        print(f"\nTriclinic angles:")
        print(f"α = {alpha:.3f}°")
        print(f"β = {beta:.3f}°") 
        print(f"γ = {gamma:.3f}°")
        print(f"a = {lx:.3f} A")
        print(f"b = {ly:.3f} A")
        print(f"c = {lz:.3f} A")
        
    except FileNotFoundError:
        print("File not found - testing with sample data instead:")
        
        # Manual test with your sample data
        box = {
            'xlo': 0.3978689266002127,
            'xhi': 78.19184915340703,
            'ylo': 1.0384583124443063,
            'yhi': 77.88243463953448,
            'zlo': 0.9508765165162215,
            'zhi': 77.07847509948887,
            'xy': -11.708768420173849,
            'xz': -10.922211430507614,
            'yz': 9.385480787100692
        }
        
        print("Sample box parameters:")
        for key, value in box.items():
            print(f"{key}: {value}")
        
        # Compute angles with sample data
        lx, ly, lz, alpha, beta, gamma = compute_triclinic_angles(**box)
        print(f"\nTriclinic angles:")
        print(f"α = {alpha:.3f}°")
        print(f"β = {beta:.3f}°") 
        print(f"γ = {gamma:.3f}°")
        print(f"a = {lx:.3f} A")
        print(f"b = {ly:.3f} A")
        print(f"c = {lz:.3f} A")
        print("")
    print(f'atoms = {atoms}')
    print("")
    print("")

In [ ]:
import numpy as np

# Given box bounds and tilts
xlo = 0.5943966731274347
xhi = 77.99532140687175
ylo = 1.1051467927041465
yhi = 77.81574615929499
zlo = 0.9228936939827597
zhi = 77.10645792201697

xy = 10.162191876835358
xz = 10.138761045047923
yz = 8.900335651933078

# Box lengths
lx = xhi - xlo
ly = yhi - ylo
lz = zhi - zlo

# Triclinic box vectors
a = np.array([lx, 0, 0])
b = np.array([xy, ly, 0])
c = np.array([xz, yz, lz])

# Volume via scalar triple product
volume_R = np.dot(a, np.cross(b, c))
print(f"Volume = {volume_R:.6f} nm³")


In [ ]:
volume_T/volume_R

In [ ]:
import numpy as np

# Box lengths
lx = 79.56233257227349 - (-0.24092302027324383)
ly = 79.48883774847997 - (-0.3024137724807617)
lz = 79.24231722245256 - (-0.5902652864530197)

xy = -0.10486226391299333
xz =  0.1150747277449955
yz = -0.12431132021093048

# Vectors defining the box
a = np.array([lx, 0, 0])
b = np.array([xy, ly, 0])
c = np.array([xz, yz, lz])

# Compute volume via scalar triple product
volume_T = np.dot(a, np.cross(b, c))
print(f"Volume = {volume_T:.6f} nm³")


In [ ]:
import numpy as np
import math

def compute_triclinic_angles(xlo, xhi, ylo, yhi, zlo, zhi, xy, xz, yz):
    """
    Compute the angles between the three axes of a triclinic simulation box
    from LAMMPS data file parameters.
    
    Parameters:
    xlo, xhi, ylo, yhi, zlo, zhi: box bounds
    xy, xz, yz: tilt factors
    
    Returns:
    alpha, beta, gamma: angles in degrees
    """
    
    # Calculate box lengths
    lx = xhi - xlo
    ly = yhi - ylo
    lz = zhi - zlo
    
    print(f"Box dimensions:")
    print(f"lx = {lx:.6f}")
    print(f"ly = {ly:.6f}")
    print(f"lz = {lz:.6f}")
    print(f"xy = {xy:.6f}")
    print(f"xz = {xz:.6f}")
    print(f"yz = {yz:.6f}")
    print()
    
    # Define the three lattice vectors
    # In LAMMPS triclinic format:
    # a = (lx, 0, 0)
    # b = (xy, ly, 0)
    # c = (xz, yz, lz)
    
    a = np.array([lx, 0, 0])
    b = np.array([xy, ly, 0])
    c = np.array([xz, yz, lz])
    
    print("Lattice vectors:")
    print(f"a = [{a[0]:.6f}, {a[1]:.6f}, {a[2]:.6f}]")
    print(f"b = [{b[0]:.6f}, {b[1]:.6f}, {b[2]:.6f}]")
    print(f"c = [{c[0]:.6f}, {c[1]:.6f}, {c[2]:.6f}]")
    print()
    
    # Calculate magnitudes
    mag_a = np.linalg.norm(a)
    mag_b = np.linalg.norm(b)
    mag_c = np.linalg.norm(c)
    
    print("Vector magnitudes:")
    print(f"|a| = {mag_a:.6f}")
    print(f"|b| = {mag_b:.6f}")
    print(f"|c| = {mag_c:.6f}")
    print()
    
    # Calculate angles using dot product formula: cos(θ) = (u·v)/(|u||v|)
    # α is angle between b and c
    # β is angle between a and c  
    # γ is angle between a and b
    
    cos_alpha = np.dot(b, c) / (mag_b * mag_c)
    cos_beta = np.dot(a, c) / (mag_a * mag_c)
    cos_gamma = np.dot(a, b) / (mag_a * mag_b)
    
    # Convert to degrees
    alpha = math.degrees(math.acos(cos_alpha))
    beta = math.degrees(math.acos(cos_beta))
    gamma = math.degrees(math.acos(cos_gamma))
    
    print("Cosines:")
    print(f"cos(α) = {cos_alpha:.6f}")
    print(f"cos(β) = {cos_beta:.6f}")
    print(f"cos(γ) = {cos_gamma:.6f}")
    print()
    
    print("Angles:")
    print(f"α (angle between b and c) = {alpha:.3f}°")
    print(f"β (angle between a and c) = {beta:.3f}°")
    print(f"γ (angle between a and b) = {gamma:.3f}°")
    
    return alpha, beta, gamma

# Your LAMMPS data
xlo, xhi = box_bounds[0][0], box_bounds[0][1]
ylo, yhi = box_bounds[1][0], box_bounds[1][1]
zlo, zhi = box_bounds[2][0], box_bounds[2][1]
xy, xz, yz = -11.708768420173849, -10.922211430507614, 9.385480787100692

print("LAMMPS Triclinic Box Angle Calculator")
print("=" * 40)
print()

# Compute angles
alpha, beta, gamma = compute_triclinic_angles(**box_bounds)

print()
print("Summary:")
print(f"α = {alpha:.3f}°")
print(f"β = {beta:.3f}°") 
print(f"γ = {gamma:.3f}°")
print(f"Sum = {alpha + beta + gamma:.3f}°")

In [ ]:
plt.figure()
plt.plot(Nf_ls_1100, mu_ls_1100, label='1100K')
plt.plot(Nf_ls_1000, mu_ls_1000, label='1000K')
plt.plot(Nf_ls_900, mu_ls_900, label='900K')
plt.plot(Nf_ls_800, mu_ls_800, label='800K')
plt.plot(Nf_ls_700, mu_ls_700, label='700K')
plt.legend()
#plt.xlim(0,1300)
plt.xlabel('Number of Na+')
plt.ylabel('Chemical potential (eV)')
plt.tight_layout()

In [ ]:
plt.figure()
plt.plot(Nf_ls_300, mu_ls_300, label='300K')
plt.plot(Nf_ls_400, mu_ls_400, label='400K')
plt.plot(Nf_ls_500, mu_ls_500, label='500K')
plt.plot(Nf_ls_600, mu_ls_600, label='600K')
plt.plot(Nf_ls_700, mu_ls_700, label='700K')
plt.legend()
#plt.xlim(0,1300)
plt.xlabel('Number of Na+')
plt.ylabel('Chemical potential (eV)')
plt.tight_layout()

In [ ]:
%matplotlib widget
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,5))
plt.subplots_adjust(wspace=0.35)
temp = 1000
fi = 0.002
for i in range(1,25):
    Nt = 150+(i-1)*50
    file = f'./production_run/free_E/test/Nt_{temp}/3/atoms_log_{i}.txt'
    log_data = parse_print_file(file)
    step = np.array(log_data['Step'])
    n_Na = np.array(log_data['atoms']) - 28672  # Adjust baseline as needed
    ax1.plot(step, n_Na, label=f'Nf_{i}={n_Na[-1]}, Nt={Nt}')
    ax1.plot([step[0],step[-1]], [Nt, Nt], color='black')
    #plt.vlines(step[-1], 0, 1000, color='black', linestyles='--', linewidth=0.5)
    file = f'./production_run/free_E/test/Nt_{temp}/3/{temp}_converge_{i}.txt'
    data = np.loadtxt(file, skiprows=2) 
    ts = data[-1,0] 
    Nf = data[-1,-1] - 28672
    mu = -3-2*fi*(Nf-Nt)
    ax1.scatter(ts,Nf)
    ax1.text(ts,Nf-5,s=f'{Nf}',fontsize=8)
    ax2.scatter(Nf,mu)
    
ax1.legend(bbox_to_anchor=(0,1))
plt.title(f'{temp} K mu_bias = -3')
ax1.grid()
ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')
ax2.set_xlabel('Number of Na+')
ax2.set_ylabel('Chemical potential (eV)')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,5))
plt.subplots_adjust(wspace=0.35)
temp = 900
fi = 0.002
for i in range(1,20):
    Nt = 150+(i-1)*50
    file = f'./production_run/free_E/test/Nt_{temp}/1/atoms_log_{i}.txt'
    log_data = parse_print_file(file)
    step = np.array(log_data['Step'])
    n_Na = np.array(log_data['atoms']) - 28672  # Adjust baseline as needed
    ax1.plot(step, n_Na, label=f'Nf_{i}={n_Na[-1]}, Nt={Nt}')
    ax1.plot([step[0],step[-1]], [Nt, Nt], color='black')
    #plt.vlines(step[-1], 0, 1000, color='black', linestyles='--', linewidth=0.5)
    file = f'./production_run/free_E/test/Nt_{temp}/1/{temp}_converge_{i}.txt'
    data = np.loadtxt(file, skiprows=2) 
    ts = data[-1,0] 
    Nf = data[-1,-1] - 28672
    mu = -3-2*fi*(Nf-Nt)
    ax1.scatter(ts,Nf)
    ax1.text(ts,Nf-5,s=f'{Nf}',fontsize=8)
    ax2.scatter(Nf,mu)
    
#ax1.legend(bbox_to_anchor=(0,1))
plt.title(f'{temp} K mu_bias = -3')
ax1.grid()
ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')
ax2.set_xlabel('Number of Na+')
ax2.set_ylabel('Chemical potential (eV)')

In [ ]:
%matplotlib inline
fig = plt.figure()
for i in range(1,10):
    Nt = 4000-(i-1)*50
    file = f'./production_run/free_E/test/Nt_1100/2/atoms_log_{i}.txt'
    log_data = parse_print_file(file)
    step = np.array(log_data['Step'])
    n_Na = np.array(log_data['atoms']) - 28672  # Adjust baseline as needed
    plt.plot(step, n_Na, label=f'Nf_{i}={n_Na[-1]}, Nt={Nt}')
    plt.plot([step[0],step[-1]], [Nt, Nt], color='black')
    #plt.vlines(step[-1], 4000, 3000, color='black', linestyles='--', linewidth=0.5)
plt.legend(bbox_to_anchor=(1, 1))
plt.title('1100 K mu_bias = -2')
plt.grid()

In [ ]:
%matplotlib inline
fig = plt.figure()
for i in range(1,10):
    Nt = 4000-(i-1)*50
    file = f'./production_run/free_E/test/Nt_1000/2/atoms_log_{i}.txt'
    log_data = parse_print_file(file)
    step = np.array(log_data['Step'])
    n_Na = np.array(log_data['atoms']) - 28672  # Adjust baseline as needed
    plt.plot(step, n_Na, label=f'Nf_{i}={n_Na[-1]}, Nt={Nt}')
    plt.plot([step[0],step[-1]], [Nt, Nt], color='black')
    #plt.vlines(step[-1], 0, 1000, color='black', linestyles='--', linewidth=0.5)
plt.legend(bbox_to_anchor=(1, 1))
plt.title('1000 K mu_bias = -2')
plt.grid()

In [ ]:
import re
import numpy as np
from lammps import lammps

def parse_print_file(file_path):
    data = {'Step': [], 'atoms': []}

    with open(file_path, 'r') as file:
        for line in file:
            if re.match(r"^\s*Step\s+Atoms", line):
                next(file)
                for data_line in file:
                    if re.match(r"^\s*\d+", data_line) and len(data_line.split()) == 2:
                        values = data_line.split()
                        data['Step'].append(int(values[0]))
                        data['atoms'].append(float(values[1]))
                    else:
                        break
    return data

def detect_equilibration_time(file, precision=10, window_fraction=0.2):
    log_data = parse_print_file(file)
    if len(log_data['Step']) < 50:
        print(f"[Equilibrium] Not enough data yet in {file}, skipping check.")
        return False, None, None, None

    step = np.array(log_data['Step'])
    window_size = int((step.shape[0]) * window_fraction)
    n_Na = np.array(log_data['atoms']) - 28672  # Adjust baseline as needed

    Q_mean_ls = []
    Q_std_ls = []
    for i in range(int(1 / window_fraction)):
        start = window_size * i
        end = window_size * (i + 1)
        if end > len(n_Na): break  # Avoid overrun
        Q_mean_ls.append(np.mean(n_Na[start:end]))
        Q_std_ls.append(np.std(n_Na[start:end]))

    Q_mean_ls = np.array(Q_mean_ls)
    Q_std_ls = np.array(Q_std_ls)

    Q_std_B1_B2_ls = Q_std_ls[:-1] + Q_std_ls[1:]
    Q_diff_ls = np.abs(Q_mean_ls[1:] - Q_mean_ls[:-1])

    precon_1 = np.std(n_Na)>5
    precon_2 = Q_diff_ls[-1]>5
    ## the CHNAGE of number of Na in the lAST block < 8
    con_1 = Q_diff_ls[-1] < precision
    ## the FLUCTUATION of number of Na in the lAST block < 15
    con_2 = Q_std_B1_B2_ls[-1] < 5
    ## the CHNAGE of number of Na in the lAST block is the smallest
    con_3 = Q_diff_ls.argmin() == Q_diff_ls.shape[0]-1
    ## the FLUCTUATION of number of Na in the lAST block is the smallest
    con_4 = Q_std_B1_B2_ls.argmin() == Q_std_B1_B2_ls.shape[0]-1

    ##### if Na profile is NOT FLAT & the last block has delta(Na) > 8  #####
    if precon_1 and precon_2:
        if con_1 and con_2 and con_3 and con_4:
            print('The simulation HAS reached equilibrium. type 1.')
            return True, Q_mean_ls, Q_diff_ls, Q_std_B1_B2_ls
        else:
            print('The simulation HAS NOT reached equilibrium. Run for another 10000 steps.')
            return False,Q_mean_ls, Q_diff_ls, Q_std_B1_B2_ls
    ##### if last block has delta(Na) < 8 #####
    else:
        print('The simulation HAS reached equilibrium. type 2.')
        return True, Q_mean_ls, Q_diff_ls, Q_std_B1_B2_ls

def check_equilibrium_lmp(lmp):
    step = lmp.get_thermo("step")
    if step < 30000:
        print(f"[Python/Invoke] Step {step} < 30000: skipping equilibrium check.")
        return

    try:
        N = lmp.extract_variable("N", None, 0)
        log_filename = f"atoms_log_{int(N)}.txt"
        print(f"[Python/Invoke] Checking equilibrium from: {log_filename} at {step} steps")

        reached, Q_mean, Q_diff, Q_std = detect_equilibration_time(log_filename)

        if reached:
            print(f"[Python/Invoke] Equilibrium reached at loop {N} at {step} steps. Unfixing fgcmc and pycheck.")
            lmp.command("unfix fgcmc")
            lmp.command("unfix pycheck")
            lmp.command("variable skip_loop equal 1")
        else:
            print(f"[Python/Invoke] Not yet equilibrated at loop {N} at {step} steps. Continuing simulation.")

    except Exception as e:
        print(f"[Python/Invoke] Error reading or processing file: {e}")


In [ ]:
import re
import numpy as np
from lammps import lammps

def parse_print_file(file_path):
    data = {'Step': [], 'atoms': []}

    with open(file_path, 'r') as file:
        for line in file:
            if re.match(r"^\s*Step\s+Atoms", line):
                next(file)
                for data_line in file:
                    if re.match(r"^\s*\d+", data_line) and len(data_line.split()) == 2:
                        values = data_line.split()
                        data['Step'].append(int(values[0]))
                        data['atoms'].append(float(values[1]))
                    else:
                        break
    return data

def detect_equilibration_time(file, precision=10, window_fraction=0.2):
    log_data = parse_print_file(file)
    if len(log_data['Step']) < 50:
        print(f"[Equilibrium] Not enough data yet in {file}, skipping check.")
        return False, None, None, None

    step = np.array(log_data['Step'])
    window_size = int((step.shape[0]) * window_fraction)
    n_Na = np.array(log_data['atoms']) - 28672  # Adjust baseline as needed

    Q_mean_ls = []
    Q_std_ls = []
    for i in range(int(1 / window_fraction)):
        start = window_size * i
        end = window_size * (i + 1)
        if end > len(n_Na): break  # Avoid overrun
        Q_mean_ls.append(np.mean(n_Na[start:end]))
        Q_std_ls.append(np.std(n_Na[start:end]))

    Q_mean_ls = np.array(Q_mean_ls)
    Q_std_ls = np.array(Q_std_ls)

    Q_std_B1_B2_ls = Q_std_ls[:-1] + Q_std_ls[1:]
    Q_diff_ls = np.abs(Q_mean_ls[1:] - Q_mean_ls[:-1])

    precon_1 = np.std(n_Na)>5
    precon_2 = Q_diff_ls[-1]>5
    ## the CHNAGE of number of Na in the lAST block < 8
    con_1 = Q_diff_ls[-1] < precision
    ## the FLUCTUATION of number of Na in the lAST block < 15
    con_2 = Q_std_B1_B2_ls[-1] < 15
    ## the CHNAGE of number of Na in the lAST block is the smallest
    con_3 = Q_diff_ls.argmin() == Q_diff_ls.shape[0]-1
    ## the FLUCTUATION of number of Na in the lAST block is the smallest
    con_4 = Q_std_B1_B2_ls.argmin() == Q_std_B1_B2_ls.shape[0]-1

    ##### if Na profile is NOT FLAT & the last block has delta(Na) > 8  #####
    if precon_1 and precon_2:
        if con_1 and con_2 and con_3 and con_4:
            print('The simulation HAS reached equilibrium. type 1.')
            return True, Q_mean_ls, Q_diff_ls, Q_std_B1_B2_ls
        else:
            print('The simulation HAS NOT reached equilibrium. Run for another 10000 steps.')
            return False,Q_mean_ls, Q_diff_ls, Q_std_B1_B2_ls
    ##### if last block has delta(Na) < 8 #####
    else:
        print('The simulation HAS reached equilibrium. type 2.')
        return True, Q_mean_ls, Q_diff_ls, Q_std_B1_B2_ls
        

def check_equilibrium_lmp(lmp):
    step = lmp.get_thermo("step")
    if step < 30000:
        print(f"[Python/Invoke] Step {step} < 30000: skipping equilibrium check.")
        return

    try:
        N = lmp.extract_variable("N", None, 0)
        log_filename = f"atoms_log_{int(N)}.txt"
        print(f"[Python/Invoke] Checking equilibrium from: {log_filename} at {step} steps")

        reached, Q_mean, Q_diff, Q_std = detect_equilibration_time(log_filename)

        if reached:
            print(f"[Python/Invoke] Equilibrium reached at loop {N} at {step} steps. Unfixing fgcmc and pycheck.")
            lmp.command("unfix fgcmc")
            lmp.command("unfix pycheck")
            lmp.command("variable skip_loop equal 1")
        else:
            print(f"[Python/Invoke] Not yet equilibrated at loop {N} at {step} steps. Continuing simulation.")

    except Exception as e:
        print(f"[Python/Invoke] Error reading or processing file: {e}")

if __name__ == "__main__":
    # Access the current LAMMPS instance that's calling this script
    check_equilibrium_lmp(lmp)



In [ ]:
i=2
file = f'./production_run/free_E/test/Nt_1100/1/atoms_log_{i}.txt'
plot_convergence(file, window_fraction=0.2, f=1)

In [ ]:
i=2
file = f'./production_run/free_E/test/Nt_1100/1/atoms_log_{i}.txt'

reached,Q_mean_ls, Q_diff_ls, Q_std_B1_B2_ls = detect_equilibration_time(file, precision=10, window_fraction=0.2)

In [ ]:
Q_mean_ls, Q_diff_ls, Q_std_B1_B2_ls

In [ ]:
file = f'./production_run/free_E/test/bias_1100/bias_279.log'

reached,Q_mean_ls, Q_diff_ls, Q_std_B1_B2_ls = detect_equilibration_time(file, precision=10, window_fraction=0.1)

In [ ]:
file = f'./production_run/free_E/test/bias_1100/bias_261.log'

reached,Q_mean_ls, Q_diff_ls, Q_std_B1_B2_ls = detect_equilibration_time(file, precision=8, window_fraction=0.2)

In [ ]:
file = f'./production_run/free_E/test/bias_1100/bias_232.log'

reached,Q_mean_ls, Q_diff_ls, Q_std_B1_B2_ls = detect_equilibration_time(file, precision=8, window_fraction=0.2)

In [ ]:
file = f'./production_run/free_E/test/1100/free_55.log'
log_data = parse_lammps_free(file)

n_Na = np.array(log_data['atoms']) - 28672

np.std(n_Na)>5


In [ ]:
Q_std_B1_B2_ls[-1]<15

In [ ]:
Q_diff_ls.argmin()==Q_diff_ls.shape[0]-1

In [ ]:
Q_std_B1_B2_ls.argmin()==Q_std_B1_B2_ls.shape[0]-1

In [ ]:
import math

def estimate_required_steps(sigma, rho, epsilon, confidence=0.99):
    """
    Estimate the number of Markov-chain steps (L) needed so that
    the statistical error in the mean of Q is <= epsilon with the
    given confidence level.

    Parameters
    ----------
    sigma : float
        Standard deviation of the observable Q (assumed known or estimated).
    rho : float
        Autocorrelation parameter (0 <= rho < 1).
        The correlation time is taken as tau = 1 / (1 - rho).
    epsilon : float
        Desired absolute error tolerance for the estimate of Q.
    confidence : float
        Confidence level in decimal form (e.g. 0.99 for 99% confidence).

    Returns
    -------
    L : int
        Estimated minimal number of steps in the Markov chain.
    """

    # Compute the correlation time tau ~ 1/(1 - rho).
    # This assumes a geometric decay of correlations.
    if not (0 <= rho < 1):
        raise ValueError("rho must be between 0 and 1 (not including 1).")
    tau = 1.0 / (1.0 - rho)  # approximate correlation time

    # Determine the Z-value from the confidence level.
    # For 99% confidence, Z ~ 2.576
    # For 95% confidence, Z ~ 1.96, etc.
    # Here we just handle the 99% case by default; otherwise you could
    # look up or compute Z via inverse error functions.
    if abs(confidence - 0.99) < 1e-6:
        z_value = 2.576
    else:
        # A simple fallback using the inverse CDF of the normal distribution.
        # from math import erf, sqrt
        # def inv_erf(x): ...  # or use scipy.special.erfinv
        # But for brevity, let us do a common approximation for 95%.
        if abs(confidence - 0.95) < 1e-6:
            z_value = 1.96
        else:
            raise ValueError("This example code only demonstrates 99% or 95% confidence.")

    # Apply the approximate formula:
    #    L >= Z^2 * (2 * sigma^2 * tau) / epsilon^2
    numerator = z_value**2 * 2.0 * (sigma**2) * tau
    L_float = numerator / (epsilon**2)

    # Round up, because you can't run a fractional number of steps
    L = math.ceil(L_float)
    return L

# ---------------------------------------------------------------------------
# Example usage:
if __name__ == "__main__":
    # Suppose we have estimated:
    # - sigma = 1.0 (std dev of Q)
    # - rho = 0.9 (autocorrelation parameter)
    # - epsilon = 0.01 (we want the mean of Q to be accurate within ±0.01)
    # - 99% confidence

    required_L = estimate_required_steps(sigma=3.0, rho=0.9, epsilon=0.03, confidence=0.95)
    print("Estimated number of steps L:", required_L)


In [ ]:
for i in range(len(Q_std_ls)-1):
    step_1 = 20+window_size*(i+0)
    step_2 = 20+window_size*(i+2)

In [ ]:
np.var(n_Na)/n_Na.shape[0]

In [ ]:
from scipy import special
p=0.001
alpha=0.99
z_a = np.sqrt(2)*special.erfinv(alpha)
np.square((p/z_a))

In [ ]:
temp, i = 1100, 8
file = f'./production_run/free_E/test/{temp}/free_{i}.log'
log_data = parse_lammps_free(file)
precision=0.005
window_fraction=0.1
step = np.array(log_data['Step'])
window_size = int((step.shape[0]-40) * window_fraction)
n_Na = np.array(log_data['atoms'])-28672

plt.figure(figsize=(10, 6))
plt.plot(log_data['Step'], -28672+natom)
plt.scatter(log_data['Step'][19],-28672+natom[19], color='r')
plt.scatter(log_data['Step'][-21],-28672+natom[-21], color='r')

Q_mean_ls=[]
Q_std_ls=[]
for i in range(int(1/window_fraction)-1):
    # Initial window
    step_1 = 20+window_size*(i+1)
    step_2 = 20+window_size*(i+2)
    Q_mean = np.mean(n_Na[step_1:step_2])
    Q_mean_ls.append(Q_mean)
    Q_std = np.std(n_Na[step_1:step_2])
    Q_std_ls.append(Q_std)
    plt.hlines(Q_mean, log_data['Step'][step_1],log_data['Step'][step_2], color='r', linestyles='--')
    plt.text(log_data['Step'][step_1], Q_mean+2, f'{round(Q_mean,0)}', color='r', size=10)
    plt.text(log_data['Step'][step_1], Q_mean-5, f'std={round(Q_std,2)}', color='r', size=10)
    plt.vlines(log_data['Step'][step_1], n_Na.min(), n_Na.max(), color='black', linestyles='--', linewidth=0.5)
    
plt.xlabel('Timestep (0.4 fs/step)')
plt.ylabel('Number of Na+')
plt.title(f'{file}')


In [ ]:
len(log_data['Step'])

In [ ]:
Q_std_ls, Q_diff_ls = detect_equilibration_time(file)

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

class EquilibrationTimeDetector:
    def __init__(self, atoms_data):
        """
        Initialize the equilibration time detector with atoms trajectory.
        
        Parameters:
        -----------
        atoms_data : list or numpy.ndarray
            Trajectory of atom numbers
        """
        self.atoms_trajectory = np.array(atoms_data)
        self.num_steps = len(self.atoms_trajectory)
    
    def detect_equilibration_time(self, confidence=0.90, window_fraction=0.2):
        """
        Detect equilibration time for the atoms trajectory.
        
        Parameters:
        -----------
        confidence : float, optional
            Confidence level for statistical tests
        window_fraction : float, optional
            Fraction of trajectory to use for initial and comparison windows
        
        Returns:
        --------
        dict
            Equilibration time and analysis details
        """
        # Calculate window sizes
        window_size = int(self.num_steps * window_fraction)
        
        # Initialize variables
        best_equilibration_time = 0
        max_consistency_score = 0
        
        # Slide windows across trajectory
        for start in range(self.num_steps - 2 * window_size-20):
            # Initial window
            initial_window = self.atoms_trajectory[start+20:start+window_size]
            
            # Comparison window
            comp_window = self.atoms_trajectory[start+20+window_size:start+2*window_size]
            
            # Perform statistical tests
            Q_initial, Q_compared = np.mean(initial_window), np.mean(comp_window)
            
            # Check if windows are statistically similar
            Q_diff = 
            
            # Update best equilibration time
            if consistency_score > max_consistency_score:
                max_consistency_score = consistency_score
                best_equilibration_time = start + window_size
        
        return {
            'equilibration_time': best_equilibration_time,
            'consistency_score': max_consistency_score,
            'mean_before': np.mean(self.atoms_trajectory[:best_equilibration_time]),
            'mean_after': np.mean(self.atoms_trajectory[best_equilibration_time:]),
            'std_before': np.std(self.atoms_trajectory[:best_equilibration_time]),
            'std_after': np.std(self.atoms_trajectory[best_equilibration_time:]),
            'total_steps': self.num_steps
        }
    
    def plot_trajectory(self, equilibration_time=None, save_path=None):
        """
        Plot trajectory for visual equilibration analysis.
        
        Parameters:
        -----------
        equilibration_time : int, optional
            Equilibration time to highlight on the plot
        save_path : str, optional
            Path to save the plot
        """
        plt.figure(figsize=(12, 6))
        plt.plot(self.atoms_trajectory, label='Atoms Trajectory')
        
        if equilibration_time is not None:
            plt.axvline(x=equilibration_time, color='r', linestyle='--', 
                        label=f'Equilibration Time ({equilibration_time})')
            
            # Highlight before and after regions
            plt.axvspan(0, equilibration_time, alpha=0.2, color='green', 
                        label='Pre-Equilibration')
            plt.axvspan(equilibration_time, self.num_steps, alpha=0.2, color='blue', 
                        label='Post-Equilibration')
        
        plt.title('Number of Atoms Trajectory')
        plt.xlabel('Step')
        plt.ylabel('Number of Atoms')
        plt.legend()
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path)
        else:
            plt.show()

# Function to perform complete analysis
def analyze_atoms_equilibration(log_file_path):
    """
    Perform complete equilibration analysis for atoms trajectory.
    
    Parameters:
    -----------
    log_file_path : str
        Path to the LAMMPS log file
    
    Returns:
    --------
    tuple
        Parsed data and equilibration results
    """
    # Parse log file
    parsed_data = parse_lammps_free(log_file_path)
    
    # Create equilibration detector
    detector = EquilibrationTimeDetector(parsed_data['atoms'])
    
    # Detect equilibration time
    equilibration_results = detector.detect_equilibration_time()
    
    # Print detailed results
    print("Equilibration Analysis for Number of Atoms:")
    print(f"Equilibration Time: {equilibration_results['equilibration_time']} steps")
    print(f"Consistency Score: {equilibration_results['consistency_score']:.4f}")
    print("\nStatistical Summary:")
    print(f"Mean (Before Equilibration): {equilibration_results['mean_before']:.2f}")
    print(f"Mean (After Equilibration): {equilibration_results['mean_after']:.2f}")
    print(f"Std Dev (Before Equilibration): {equilibration_results['std_before']:.2f}")
    print(f"Std Dev (After Equilibration): {equilibration_results['std_after']:.2f}")
    print(f"Total Simulation Steps: {equilibration_results['total_steps']}")
    
    # Plot trajectory
    detector.plot_trajectory(
        equilibration_time=equilibration_results['equilibration_time']
    )
    
    return parsed_data, equilibration_results

In [ ]:
# Replace with the path to your LAMMPS log file
temp, i = 1100, 7
file = f'./production_run/free_E/test/{temp}/free_{i+1}.log'

# Run the analysis
parsed_data, equilibration_results = analyze_atoms_equilibration(file)

In [ ]:
L=1e6
window_fraction=0.1
window_size = int(L*window_fraction/500)

averaged_Q = []
for i in range(int(1/window_fraction)):
    

In [ ]:
log_data['Step'][19], log_data['Step'][-21]

## 600K

In [ ]:
plot_Na_profile_from_dump(600, 0, 29, -1.96, 0.0025)

In [ ]:
plot_Na_profile(600, 0, 11, -1.92, 0.005)

In [ ]:
plot_Na_profile(600, 22,33, -1.98, 0.005)

In [ ]:
plot_Na_profile(600, 12, 22, -1.95, 0.005)

In [ ]:
plot_Na_profile(600, 33, 44, -1.85, 0.005)

In [ ]:
plot_Na_profile_from_dump(600, 14, 29, -1.925, 0.0025)

## 300K

In [ ]:
plot_Na_profile(300, 0, 11, -1.86, 0.005)

In [ ]:
plot_Na_profile(300, 22,33, -1.87, 0.005)

In [ ]:
plot_Na_profile(300, 11, 22, -1.86, 0.005)

In [ ]:
plot_Na_profile(300, 33, 44, -1.80, 0.005)

In [ ]:
plot_Na_profile_from_dump(600, 0, 11, -1.92, 0.005)

In [ ]:
plot_Na_profile_from_dump(600, 11, 22, -1.91, 0.005)

In [ ]:
plot_Na_profile(600, 0, 29, -1.96, 0.0025)

In [ ]:
plot_Na_profile(600, 14, 29, -1.925, 0.0025)

In [ ]:
plot_Na_profile(300, 0, 11, -1.86, 0.005)

In [ ]:
plot_Na_profile(300, 1, 13, -1.89, 0.0025)

### 300K

In [ ]:
temp, n_1, n_2, mu_1, mu_2, step = 300,  0, 21, -2, -1.95, 0.0025

timestep_ls = []
num_atoms_ls = []
for i in range(n_1, n_2):
    trj_filename = f'./production_run/free_E/test/dump.gcmc_{i+1}.trj'  # Replace with your actual file name
    timestep, num_atoms = extract_timestep_and_atom_count(trj_filename)
    timestep_ls.append(timestep)
    num_atoms_ls.append(num_atoms)
    
fig, ax1 = plt.subplots(figsize=(10, 6))
for i in range(n_1, n_2):
    nNa=-3584+num_atoms_ls[i][-1]
    ax1.plot(timestep_ls[i], -3584+num_atoms_ls[i], color=cm.jet((i-n_1)/(n_2-n_1)),
             label=f'mu_{i+1} = {round(np.arange(mu_1, mu_2+step, step)[i-n_1],4)}  Na_f = {nNa}')#  V = {round(V_ave,2)}')#, marker='*')
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')   
plt.legend(bbox_to_anchor=(1, 1))
plt.grid()
ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')

In [ ]:
temp, n_1, n_2, mu_1, mu_2, step = 300,  21, 40, -1.9, -1, 0.05

timestep_ls = []
num_atoms_ls = []
for i in range(n_1, n_2):
    trj_filename = f'./production_run/free_E/test/300/dump.gcmc_{i+1}.trj'  # Replace with your actual file name
    timestep, num_atoms = extract_timestep_and_atom_count(trj_filename)
    timestep_ls.append(timestep)
    num_atoms_ls.append(num_atoms)
    
fig, ax1 = plt.subplots(figsize=(10, 6))
for i in range(n_1, n_2):
    nNa=-3584+num_atoms_ls[i-n_1][-1]
    ax1.plot(timestep_ls[i-n_1], -3584+num_atoms_ls[i-n_1], color=cm.jet((i-n_1)/(n_2-n_1)),
             label=f'mu_{i+1} = {round(np.arange(mu_1, mu_2+step, step)[i-n_1],4)}  Na_f = {nNa}')#  V = {round(V_ave,2)}')#, marker='*')
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')   
plt.legend(bbox_to_anchor=(1, 1))
plt.grid()
ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')

In [ ]:
temp, n_1, n_2, mu_1, mu_2, step = 300,  41, 52, -1.975, -1.95, 0.05

timestep_ls = []
num_atoms_ls = []
for i in range(n_1, n_2):
    trj_filename = f'./production_run/free_E/test/dump.gcmc_{i+1}.trj'  # Replace with your actual file name
    timestep, num_atoms = extract_timestep_and_atom_count(trj_filename)
    timestep_ls.append(timestep)
    num_atoms_ls.append(num_atoms)
    
fig, ax1 = plt.subplots(figsize=(10, 6))
for i in range(n_1, n_2):
    nNa=-3584+num_atoms_ls[i-n_1][-1]
    ax1.plot(timestep_ls[i-n_1], -3584+num_atoms_ls[i-n_1], color=cm.jet((i-n_1)/(n_2-n_1)),
             label=f'mu_{i+1} = {round(np.arange(mu_1, mu_2+step, step)[i-n_1],4)}  Na_f = {nNa}')#  V = {round(V_ave,2)}')#, marker='*')
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')   
plt.legend(bbox_to_anchor=(1, 1))
plt.grid()
ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')

In [ ]:
plot_Na_profile(300, 9, 21, -1.9775, -1.95, 0.0025)

In [ ]:
for i in range(18,22):
    file = f'./production_run/free_E/test/600/free_{i+1}.log'
    plot_log_free(file)

## LV20 scale by force

In [ ]:
for i in [1,2,5]:
    file = f'./test_almtp_lv20/gcmc_{i}.log'
    plot_log(file)

In [ ]:
for i in [3,4,6]:
    file = f'./test_almtp_lv20/gcmc_{i}.log'
    plot_log(file)

## NBH

In [ ]:
for i in range(2):
    file = f'./test_almtp_nbh/gcmc_{i+1}.log'
    plot_log_no_V(file)

## compare training set

In [ ]:
for i in range(7):
    file = f'./test_training_set/gcmc_{i+1}.log'
    plot_log(file)

In [ ]:
for i in range(7,11):
    file = f'./test_training_set/gcmc_{i+1}.log'
    plot_log(file)

In [ ]:
for i in range(11,14):
    file = f'./test_training_set/gcmc_{i+1}.log'
    plot_log(file)

In [ ]:
for i in range(14,21):
    file = f'./test_training_set/gcmc_{i+1}.log'
    plot_log(file)

In [ ]:
for i in range(21,27):
    file = f'./test_training_set/gcmc_{i+1}.log'
    plot_log(file)

In [ ]:
for i in range(27,33):
    file = f'./test_training_set/gcmc_{i+1}.log'
    plot_log(file)

## Make_cfg

In [ ]:
for i in range(10):
    file = f'./test_training_set/make_cfg_{i+1}.log'
    plot_log(file)

# Diffusion

### 400K

In [ ]:
for i in [400,1200,2000,2800,3600]:
    file = f'./production_run/free_E/test/diffusion_400/400_diffusion_{i}.log'
    plot_log_diff(file)

### 500K

In [ ]:
for i in [400,3600]:
    file = f'./production_run/free_E/test/diffusion_500/500_diffusion_{i}.log'
    plot_log_diff(file)

### 600K

In [ ]:
for i in [400,3600]:
    file = f'./production_run/free_E/test/diffusion_600/600_diffusion_{i}.log'
    plot_log_diff(file)

### 700K

In [ ]:
for i in [400,3600]:
    file = f'./production_run/free_E/test/diffusion_700/700_diffusion_{i}.log'
    plot_log_diff(file)

### 800K

In [ ]:
for i in [400,3600]:
    file = f'./production_run/free_E/test/diffusion_800/800_diffusion_{i}.log'
    plot_log_diff(file)

### 900K

In [ ]:
for i in [400,3600]:
    file = f'./production_run/free_E/test/diffusion_900/900_diffusion_{i}.log'
    plot_log_diff(file)

### 1000K

In [ ]:
for i in [400,3600]:
    file = f'./production_run/free_E/test/diffusion_1000/1000_diffusion_{i}.log'
    plot_log_diff(file)

### Raise Temperature

In [ ]:
file = f'raise_temp.log'
path = './test_training_set/phase_diagram/'
log_data = parse_lammps_log_diff(path+file)
log_data_2 = parse_lammps_log(path+file)

In [ ]:
file = f'raise_temp.log'
path = './test_training_set/phase_diagram/'
log_data = parse_lammps_log_diff(path+file)
log_data_2 = parse_lammps_log(path+file)
num_Na = int(log_data['v_nNa'][0])

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (1 fs/step)')
ax1.set_ylabel('Volume (A^3)', color=color)
ax1.plot(log_data['Step'], log_data['Volume'], color=color, marker='o', markersize=3)
ax1.plot(log_data_2['Step'], log_data_2['Volume'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Temperature (K) (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Temp'][10:], color=color, marker='s', markersize=3)
ax2.plot(log_data_2['Step'][10:], log_data_2['Temp'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

# MSD(PB) (Third y-axis, second right)
ax3 = ax1.twinx()
color3 = 'tab:green'
ax3.spines['right'].set_position(('outward', 80))
ax3.set_ylabel('PB MSD A^2', color=color3)
ax3.plot(log_data['Step'], log_data['MSD(PB)'], color=color3, marker='*', markersize=3)
ax3.hlines(3,log_data['Step'][0],log_data['Step'][-1], color='r')
ax3.tick_params(axis='y')
ax3.grid(True, linestyle='--', alpha=0.7)

ax4 = ax1.twinx()
color4 = 'tab:purple'
ax4.spines['left'].set_position(('outward', 80))
ax4.set_ylabel('Na MSD A^2', color=color3)
ax4.plot(log_data['Step'], log_data['MSD(Na)'], color=color4, marker='*', markersize=3)
#ax4.tick_params(axis='y')

plt.title(file + f'    Number of Na = {num_Na}')
plt.tight_layout()
#plt.savefig(log_file_path+'.jpg', dpi=300, bbox_inches='tight')

In [ ]:
plt.plot(log_data['Step'], log_data['Temp'], label='Temperature')
plt2=plt.twinx()
plt2.plot(log_data['Step'], log_data['Volume'],color='orange', label'Volume')

In [ ]:
plt.plot(log_data['Step'], log_data['Temp'])
plt2=plt.twinx()
plt2.plot(log_data['Step'], log_data['MSD(Na)'],color='orange')

In [ ]:
filename = f'dump_raise_temp_Na.dat'
path = './test_training_set/diffusion/'
data_Na = pd.read_csv(path + filename, delim_whitespace=True, comment='#', header=None)
data_Na.columns = ['TimeStep', 'c_msd_Na[1]', 'c_msd_Na[2]', 'c_msd_Na[3]', 'c_msd_Na[4]']

filename = f'dump_raise_temp_PB.dat'
path = './test_training_set/diffusion/'
data_PB = pd.read_csv(path + filename, delim_whitespace=True, comment='#', header=None)
data_PB.columns = ['TimeStep', 'c_msd_PB[1]', 'c_msd_PB[2]', 'c_msd_PB[3]', 'c_msd_PB[4]']

In [ ]:
time = range(0, data_PB['TimeStep'].iloc[-1]+40000,40000)
temp = range(800,1600,50)
len(time),len(temp)

In [ ]:
plt.figure(figsize=(8,5))
#plt.plot(data_Na['TimeStep'], data_Na['c_msd_Na[1]'], label='MSD(x)')
#plt.plot(data_Na['TimeStep'], data_Na['c_msd_Na[2]'], label='MSD(y)')
#plt.plot(data_Na['TimeStep'], data_Na['c_msd_Na[3]'], label='MSD(z)')
plt.plot(data_Na['TimeStep'], data_Na['c_msd_Na[4]'], label='TMSD_Na')
#plt.plot(data_PB['TimeStep'], data_PB['c_msd_PB[1]'], label='MSD(x)')
#plt.plot(data_PB['TimeStep'], data_PB['c_msd_PB[2]'], label='MSD(y)')
#plt.plot(data_PB['TimeStep'], data_PB['c_msd_PB[3]'], label='MSD(z)')
plt.plot(data_PB['TimeStep'], data_PB['c_msd_PB[4]'], label='TMSD_PB')
plt.xlabel('Timestep (1 fs/step)')
plt.grid()
plt.ylabel('MSD (A^2)')

plt2 = plt.twinx()
plt2.scatter(time, temp)
plt.legend()


## MSD Analysis

In [ ]:
#data = {'Step': [], 'Temp': [], 'TotEng': [], 'Press': [], 'Volume': [], 'v_nNa': [], 'MSD(PB)' : [], 'MSD(Na)' : []}

In [ ]:
i=400
file = f'./production_run/free_E/test/diffusion_1000/1000_diffusion_{i}.log'
data = parse_lammps_log(file)


In [ ]:
#%matplotlib widget

delta_t_ls = np.array([1, 2, 5, 10,15, 20,30,40, 50,60,70,80,90, 100, 200,300,400, 500])#,2000,5000]) #1, 2.5, 10, 25, 100, 250 ps

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18,5))
plt.subplots_adjust(wspace=1)

temp = 500
Na_list = [3952]#,1144,1944,3280,3600,3952]

diffusivity_list = []
num_Na_list = []
k=0
for index in Na_list:
    file = f'./production_run/free_E/test/diffusion_{temp}/{temp}_diffusion_{index}.log'
    log_data = parse_lammps_log_diff(file)
    MSD_Na = np.array(log_data['MSD(Na)'])
    timestep = np.array(log_data['Step'])
    num_Na = int(log_data['v_nNa'][0])-28672
    num_Na_list.append(num_Na)
    t_intival = []
    MSD_intival = []
    for i in range(delta_t_ls.shape[0]):
        #print(i, delta_t_ls[i]*0.5)
        t_intival.append(np.arange(timestep[0], timestep[-1], int(delta_t_ls[i])*500, dtype='int'))
        MSD_intival.append(np.linspace(MSD_Na[0], MSD_Na[-1], t_intival[i].shape[0]))

    delta_t_all = []
    delta_MSD_all = []
    for i in range(delta_t_ls.shape[0]):
        delta_t = []
        delta_MSD = []
        for j in range(t_intival[i].shape[0]-1):
            delta_t.append(t_intival[i][j+1] - t_intival[i][j])
            delta_MSD.append(MSD_intival[i][j+1] - MSD_intival[i][j])
        delta_t_all.append(delta_t)
        delta_MSD_all.append(delta_MSD)

    final_MSD = []
    final_t = []
    diffusivity = []
    for i in range(delta_t_ls.shape[0]):
        final_t.append(sum(delta_t_all[i])/len(delta_t_all[i]))
        final_MSD.append(sum(delta_MSD_all[i])/len(delta_MSD_all[i]))
        
    final_MSD = np.array(final_MSD)
    final_t = np.array(final_t)
    diffusivity_list.append(final_MSD/final_t/6*1e-5)
    
    ax2.plot(final_t, final_MSD, marker='*', markersize=3, label=f'{temp}K SoC = {round(Na_list[k]/4096,3)}')
    ax1.set_xlabel('Timestep (1 fs/step)')
    ax1.set_ylabel('MSD A^2')
    ax1.plot(log_data['Step'], log_data['MSD(Na)'],  marker='*', markersize=3, label=f'SoC {round(Na_list[k]/4096,3)}')
    k+=1
    
ax1.grid(True, linestyle='--', alpha=0.7)
ax2.grid(True, linestyle='--', alpha=0.7)
ax2.legend()
ax1.legend()
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlabel('delta t (fs)')
ax2.set_ylabel('MSD (A^2)')
plt.title(f'{temp} K')

for i in range(len(diffusivity_list)):
    ax3.plot(final_t, diffusivity_list[i], marker='*', markersize=3, label=f'{temp}K SoC = {round(Na_list[i]/4096,3)}')

ax3.grid(True, linestyle='--', alpha=0.7)
ax3.legend()
#ax3.set_xscale('log')
#ax3.set_yscale('log')
ax3.set_xlabel('delta t (fs)')
ax3.set_ylabel('Diffusivity (m^2/s)')

plt.tight_layout()
plt.savefig(f'./production_run/free_E/test/diffusion_{temp}K'+'.png', bbox_inches='tight')


In [ ]:
dMSD_dt = np.gradient(final_MSD, final_t)
plt.figure()
plt.plot(final_t, dMSD_dt, marker='*', markersize=5)

In [ ]:
#%matplotlib widget
def MSD_analysis(temp, Na_list, start, end):
    delta_t_ls = np.array([4, 5, 6, 7, 8,
                       9, 10, 15, 20, 30,
                       40, 50, 60, 70, 80,
                       90, 100, 200, 300, 400,
                       500, 600])
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(17,5))
    plt.subplots_adjust(wspace=1)
    
    temp = temp
    num_Na_list = []
    diffusivity_list = []
    k=Na_list[0]-1
    for index in Na_list:
        file = f'./production_run/free_E/test/diffusion_{temp}/{temp}_diffusion_{index}.log'
        log_data = parse_lammps_log_diff(file)
        MSD_Na = np.array(log_data['MSD(Na)'])
        timestep = np.array(log_data['Step'])
        num_Na = int(log_data['v_nNa'][0])-28672
        num_Na_list.append(num_Na)
        t_intival_ls = []
        MSD_intival_ls = []
        for i in range(delta_t_ls.shape[0]):
            num_segment = int((MSD_Na.shape[0]-start)/delta_t_ls[i])
            t_intival = []
            MSD_intival = []
            for w in range(num_segment):
                t_intival.append(timestep[start + delta_t_ls[i]*w : start + delta_t_ls[i]*(w+1)] - timestep[start + delta_t_ls[i]*w])
                MSD_intival.append(MSD_Na[start + delta_t_ls[i]*w : start + delta_t_ls[i]*(w+1)] - MSD_Na[start + delta_t_ls[i]*w])
            t_intival_ls.append(t_intival)
            MSD_intival_ls.append(MSD_intival)
    
        delta_t_all = []
        delta_MSD_all = []
        for i in range(delta_t_ls.shape[0]):
            delta_t = []
            delta_MSD = []
            for j in range(len(MSD_intival_ls[i])):
                delta_MSD.append(MSD_intival_ls[i][j][-1])
                delta_t.append(t_intival_ls[i][j][-1])
    
            delta_t_all.append(delta_t)
            delta_MSD_all.append(delta_MSD)
    
        final_MSD = []
        final_t = []
        MSD_std = []
        diffusivity = []
        
        for i in range(delta_t_ls.shape[0]):
            final_t.append(sum(delta_t_all[i])/len(delta_t_all[i]))
            final_MSD.append(sum(delta_MSD_all[i])/len(delta_MSD_all[i]))
            MSD_std.append(np.std(delta_MSD_all[i]))
            
        final_MSD = np.array(final_MSD)
        final_t = np.array(final_t)
        MSD_std = np.array(MSD_std)
        diffusivity_list.append(final_MSD/final_t/6*1e-5)
        ax2.errorbar(final_t, final_MSD, yerr=MSD_std, capsize=5, fmt='-o', alpha=0.8)
        #ax2.plot(final_t, final_MSD, marker='*', markersize=3, label=f'{temp}K x = {round(num_Na_list[k]/864,3)} y = {round(num_H2O_list[k]/864,3)}')
        ax1.set_xlabel('Timestep (1 fs/step)')
        ax1.set_ylabel('MSD A^2')
        ax1.plot(log_data['Step'], log_data['MSD(Na)'],  marker='*', markersize=3)#, label=f'{temp}K SoC = {round(Na_list[k]/4096,3)}')
        ax1.plot(timestep[:start], MSD_Na[:start], color='black', alpha=0.5)
        k+=1
        
    ax1.grid(True, linestyle='--', alpha=0.7)
    ax2.grid(True, linestyle='--', alpha=0.7)
    #ax2.legend()
    ax1.legend()
    #ax2.set_xscale('log')
    #ax2.set_yscale('log')
    ax2.set_xlabel('delta t (fs)')
    ax2.set_ylabel('MSD (A^2)')
    plt.title(f'{temp} K')
    
    k=Na_list[0]-1
    for i in range(len(diffusivity_list)):
        #ax3.errorbar(final_t, diffusivity_list[i], yerr=MSD_std/final_t/6*1e-5 , capsize=5 , fmt='-o')
        ax3.plot(final_t, diffusivity_list[i], marker='*', markersize=3)#, label=f'{temp}K SoC = {round(Na_list[k]/4096,3)}')
        k+=1
    
    ax3.grid(True, linestyle='--', alpha=0.7)
    #ax3.legend()
    #ax3.set_xscale('log')
    #ax3.set_yscale('log')
    ax3.set_xlabel('delta t (fs)')
    ax3.set_ylabel('Diffusivity (m^2/s)')
    
    plt.tight_layout()
    plt.savefig(f'./production_run/free_E/test/diffusion_{temp}K'+'.png', bbox_inches='tight')
    return diffusivity_list

In [ ]:
%matplotlib inline
diff_400=MSD_analysis(400,[400,1200,3600],1000,-1)

In [ ]:
diff_500=MSD_analysis(500,[400,1144,1944,3280,3600,3952],1000,-1)

In [ ]:
diff_600=MSD_analysis(600,[400,1144,1944,3280,3600,3952],1000,-1)

In [ ]:
diff_700=MSD_analysis(700,[400,1144,1944,3280,3600],1000,-1)

In [ ]:
diff_800=MSD_analysis(800,[400,1144,1944,3280,3600],1000,-1)

In [ ]:
diff_900=MSD_analysis(900,[400,3600],1000,-1)

In [ ]:
diff_1000=MSD_analysis(1000,[400,3600],1000,-1)

In [ ]:
diff_1000[0][10:15].mean()

In [ ]:
Na_90_diff_ls=[diff_400[-1][i], diff_500[-2][i], diff_600[-2][i], diff_700[-1][i], diff_800[-1][i], diff_900[-1][i], diff_1000[-1][i]]
Na_90_diff_ls

In [ ]:
big_diff = [diff_400, diff_600, diff_800, diff_1000]
diffusivity_1 = []
diffusivity_2 = []
diffusivity_3 = []
diffusivity_4 = []
temp = np.array([400, 600, 800, 1000])

for i in range(temp.shape[0]):
    print(i)
    diffusivity_1.append(big_diff[i][0][9])
    diffusivity_2.append(big_diff[i][1][9])
    diffusivity_3.append(big_diff[i][2][9])
    diffusivity_4.append(big_diff[i][2][9])


In [ ]:
Na_10_diff_ls

In [ ]:
temp_1 = np.array([400, 500, 600, 700, 800, 900, 1000])
temp_2 = np.array([500, 600, 700, 800])
delta_t_ls = np.array([4, 5, 6, 7, 8,
                       9, 10, 15, 20, 30,
                       40, 50, 60, 70, 80,
                       90, 100, 200, 300, 400,
                       500, 600])
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18,5))

i=10
p1, p2 = -8,-7
Na_90_diff_ls=[diff_400[-1][p1:p2].mean(), diff_500[-2][p1:p2].mean(), diff_600[-2][p1:p2].mean(), diff_700[-1][p1:p2].mean(),
               diff_800[-1][p1:p2].mean(), diff_900[-1][p1:p2].mean(), diff_1000[-1][p1:p2].mean()]

Na_80_diff_ls=[diff_500[-3][p1:p2].mean(), diff_600[-3][p1:p2].mean(),diff_700[-2][p1:p2].mean(),diff_800[-1][p1:p2].mean()]
Na_50_diff_ls=[diff_500[2][p1:p2].mean(), diff_600[2][p1:p2].mean(),diff_700[2][p1:p2].mean(),diff_800[2][p1:p2].mean()]
Na_30_diff_ls=[diff_500[1][p1:p2].mean(), diff_600[1][p1:p2].mean(),diff_700[1][p1:p2].mean(),diff_800[1][p1:p2].mean()]

Na_10_diff_ls=[diff_400[0][p1:p2].mean(), diff_500[0][p1:p2].mean(), diff_600[0][p1:p2].mean(), diff_700[0][p1:p2].mean(),
               diff_800[0][p1:p2].mean(), diff_900[0][p1:p2].mean(), diff_1000[0][p1:p2].mean()]

ax1.plot(1/temp_1*1000, Na_90_diff_ls, marker='*', markersize=3, label='SoC=0.878')
ax1.plot(1/temp_2*1000, Na_80_diff_ls, marker='*', markersize=3, label='SoC=0.801')
ax1.plot(1/temp_2*1000, Na_50_diff_ls, marker='*', markersize=3, label='SoC=0.475')
ax1.plot(1/temp_2*1000, Na_30_diff_ls, marker='*', markersize=3, label='SoC=0.279')
ax1.plot(1/temp_1*1000, Na_10_diff_ls, marker='*', markersize=3, label='SoC=0.098')
ax1.legend()
ax1.grid()
ax1.set_title(f'delta t = {delta_t_ls[i]*0.5} ps')
#ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel('1000/T (1/K)')
ax1.set_ylabel('Diffusivity (m^2/s)')

i=12
p1, p2 = -7,-6
Na_90_diff_ls=[diff_400[-1][p1:p2].mean(), diff_500[-2][p1:p2].mean(), diff_600[-2][p1:p2].mean(), diff_700[-1][p1:p2].mean(), diff_800[-1][p1:p2].mean(), diff_900[-1][p1:p2].mean(), diff_1000[-1][p1:p2].mean()]
Na_80_diff_ls=[diff_500[-3][p1:p2].mean(), diff_600[-3][p1:p2].mean(),diff_700[-2][p1:p2].mean(),diff_800[-1][p1:p2].mean()]
Na_50_diff_ls=[diff_500[2][p1:p2].mean(), diff_600[2][p1:p2].mean(),diff_700[2][p1:p2].mean(),diff_800[2][p1:p2].mean()]
Na_30_diff_ls=[diff_500[1][p1:p2].mean(), diff_600[1][p1:p2].mean(),diff_700[1][p1:p2].mean(),diff_800[1][p1:p2].mean()]
Na_10_diff_ls=[diff_400[0][p1:p2].mean(), diff_500[0][p1:p2].mean(),diff_600[0][p1:p2].mean(),diff_700[0][p1:p2].mean(),diff_800[0][p1:p2].mean(),diff_900[0][p1:p2].mean(),diff_1000[0][p1:p2].mean()]

ax2.plot(1/temp_1*1000, Na_90_diff_ls, marker='*', markersize=3, label='SoC=0.878')
ax2.plot(1/temp_2*1000, Na_80_diff_ls, marker='*', markersize=3, label='SoC=0.801')
ax2.plot(1/temp_2*1000, Na_50_diff_ls, marker='*', markersize=3, label='SoC=0.475')
ax2.plot(1/temp_2*1000, Na_30_diff_ls, marker='*', markersize=3, label='SoC=0.279')
ax2.plot(1/temp_1*1000, Na_10_diff_ls, marker='*', markersize=3, label='SoC=0.098')
ax2.legend()
ax2.grid()
ax2.set_title(f'delta t = {delta_t_ls[i]*0.5} ps')
#ax1.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlabel('1000/T (1/K)')
ax2.set_ylabel('Diffusivity (m^2/s)')

p1, p2 = -6,-5
Na_90_diff_ls=[diff_400[-1][p1:p2].mean(), diff_500[-2][p1:p2].mean(), diff_600[-2][p1:p2].mean(), diff_700[-1][p1:p2].mean(), diff_800[-1][p1:p2].mean(), diff_900[-1][p1:p2].mean(), diff_1000[-1][p1:p2].mean()]
Na_80_diff_ls=[diff_500[-3][p1:p2].mean(), diff_600[-3][p1:p2].mean(),diff_700[-2][p1:p2].mean(),diff_800[-1][p1:p2].mean()]
Na_50_diff_ls=[diff_500[2][p1:p2].mean(), diff_600[2][p1:p2].mean(),diff_700[2][p1:p2].mean(),diff_800[2][p1:p2].mean()]
Na_30_diff_ls=[diff_500[1][p1:p2].mean(), diff_600[1][p1:p2].mean(),diff_700[1][p1:p2].mean(),diff_800[1][p1:p2].mean()]
Na_10_diff_ls=[diff_400[0][p1:p2].mean(), diff_500[0][p1:p2].mean(),diff_600[0][p1:p2].mean(),diff_700[0][p1:p2].mean(),diff_800[0][p1:p2].mean(),diff_900[0][p1:p2].mean(),diff_1000[0][p1:p2].mean()]

ax3.plot(1/temp_1*1000, Na_90_diff_ls, marker='*', markersize=3, label='SoC=0.878')
ax3.plot(1/temp_2*1000, Na_80_diff_ls, marker='*', markersize=3, label='SoC=0.801')
ax3.plot(1/temp_2*1000, Na_50_diff_ls, marker='*', markersize=3, label='SoC=0.475')
ax3.plot(1/temp_2*1000, Na_30_diff_ls, marker='*', markersize=3, label='SoC=0.279')
ax3.plot(1/temp_1*1000, Na_10_diff_ls, marker='*', markersize=3, label='SoC=0.098')
ax3.legend()
ax3.grid()
ax3.set_title(f'delta t = {delta_t_ls[i]*0.5} ps')
#ax1.set_xscale('log')
ax3.set_yscale('log')
ax3.set_xlabel('1000/T (1/K)')
ax3.set_ylabel('Diffusivity (m^2/s)')

In [ ]:
temp_1 = np.array([400, 500, 600, 700, 800, 900, 1000])
temp_2 = np.array([500, 600, 700, 800])
delta_t_ls = np.array([4, 5, 6, 7, 8,
                       9, 10, 15, 20, 30,
                       40, 50, 60, 70, 80,
                       90, 100, 200, 300, 400,
                       500, 600])
fig, (ax1) = plt.subplots(1, 1, figsize=(singlecol*2,singlecol*2))

i=10
p1, p2 = -6,-5
Na_90_diff_ls=[diff_400[-1][p1:p2].mean(), diff_500[-2][p1:p2].mean(), diff_600[-2][p1:p2].mean(), diff_700[-1][p1:p2].mean(),
               diff_800[-1][p1:p2].mean(), diff_900[-1][p1:p2].mean(), diff_1000[-1][p1:p2].mean()]

Na_80_diff_ls=[diff_500[-3][p1:p2].mean(), diff_600[-3][p1:p2].mean(),diff_700[-2][p1:p2].mean(),diff_800[-1][p1:p2].mean()]
Na_50_diff_ls=[diff_500[2][p1:p2].mean(), diff_600[2][p1:p2].mean(),diff_700[2][p1:p2].mean(),diff_800[2][p1:p2].mean()]
Na_30_diff_ls=[diff_500[1][p1:p2].mean(), diff_600[1][p1:p2].mean(),diff_700[1][p1:p2].mean(),diff_800[1][p1:p2].mean()]

Na_10_diff_ls=[diff_400[0][p1:p2].mean(), diff_500[0][p1:p2].mean(), diff_600[0][p1:p2].mean(), diff_700[0][p1:p2].mean(),
               diff_800[0][p1:p2].mean(), diff_900[0][p1:p2].mean(), diff_1000[0][p1:p2].mean()]

ax1.plot(1/temp_1*1000, Na_90_diff_ls, marker='*', markersize=3, label='SoC=0.878')
ax1.plot(1/temp_2*1000, Na_80_diff_ls, marker='*', markersize=3, label='SoC=0.801')
ax1.plot(1/temp_2*1000, Na_50_diff_ls, marker='*', markersize=3, label='SoC=0.475')
ax1.plot(1/temp_2*1000, Na_30_diff_ls, marker='*', markersize=3, label='SoC=0.279')
ax1.plot(1/temp_1*1000, Na_10_diff_ls, marker='*', markersize=3, label='SoC=0.098')
ax1.legend()
ax1.grid()
ax1.set_title(f'delta t = {delta_t_ls[i]*0.5} ps')
#ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel('1000/T (1/K)')
ax1.set_ylabel('Diffusivity (m^2/s)')

In [ ]:
p1, p2 = -6,-5
Na_90_diff_ls=[diff_400[-1][p1:p2].mean(), diff_500[-2][p1:p2].mean(), diff_600[-2][p1:p2].mean(), diff_700[-1][p1:p2].mean(), diff_800[-1][p1:p2].mean(), diff_900[-1][p1:p2].mean(), diff_1000[-1][p1:p2].mean()]
Na_80_diff_ls=[diff_500[-3][p1:p2].mean(), diff_600[-3][p1:p2].mean(),diff_700[-2][p1:p2].mean(),diff_800[-1][p1:p2].mean()]
Na_50_diff_ls=[diff_500[2][p1:p2].mean(), diff_600[2][p1:p2].mean(),diff_700[2][p1:p2].mean(),diff_800[2][p1:p2].mean()]
Na_30_diff_ls=[diff_500[1][p1:p2].mean(), diff_600[1][p1:p2].mean(),diff_700[1][p1:p2].mean(),diff_800[1][p1:p2].mean()]
Na_10_diff_ls=[diff_400[0][p1:p2].mean(), diff_500[0][p1:p2].mean(),diff_600[0][p1:p2].mean(),diff_700[0][p1:p2].mean(),diff_800[0][p1:p2].mean(),diff_900[0][p1:p2].mean(),diff_1000[0][p1:p2].mean()]

In [ ]:
Na_90_diff_ls, np.log10(Na_90_diff_ls)

In [ ]:
import numpy as np

x = 100

log10_x = np.log10(x)         # base-10 log
ln_x = np.log(x)              # natural log

# Convert log10 to ln
converted_ln = log10_x * np.log(10)

# Convert ln to log10
converted_log10 = ln_x / np.log(10)
converted_ln

In [ ]:
x_fit

In [ ]:
plt.rcParams.update({'font.size': 8})
#plt.rcParams.update({'font.sans-serif': 'Helvetica'})
plt.rcParams.update({'lines.linewidth':1})
plt.rcParams.update({'axes.linewidth':1})
plt.rcParams.update({'lines.markersize':3})
plt.rcParams.update({'savefig.dpi':200})
plt.rcParams.update({'legend.fontsize': 8})
fig, ax = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2), dpi=300)
plt.subplots_adjust(wspace=0.35)
temp_1 = np.array([400, 500, 600, 700, 800, 900, 1000])
temp_2 = np.array([600, 700, 800])
colors_ls = [
    "orange",  # very light blue
    "deepskyblue",   # light/bright blue
    "dodgerblue",    # medium bright blue
    "royalblue",     # vivid deep blue
    "mediumblue",    # strong dark blue
    "navy"           # very dark blue
]
temp = 500
Na_list = [400,1144,1944,3280,3600,3952]
k=0
for index in Na_list:
    file = f'./production_run/free_E/test/diffusion_{temp}/{temp}_diffusion_{index}.log'
    log_data = parse_lammps_log_diff(file)
    MSD_Na = np.array(log_data['MSD(Na)'])
    timestep = np.array(log_data['Step'])
    num_Na = int(log_data['v_nNa'][0])-28672
    #num_Na_list.append(num_Na)

    ax[0,0].set_xlabel('Simulation time (ps)')
    ax[0,0].set_ylabel(r'MSD Å$^2$')
    ax[0,0].plot(timestep/1000, MSD_Na, label=f'x = {round(Na_list[k]/2048,3)}', color=colors_ls[k])
    k+=1
ax[0,0].legend(frameon=False)

p1, p2 = -6,-5
Na_90_diff_ls=[diff_400[-1][p1:p2].mean(), diff_500[-2][p1:p2].mean(), diff_600[-2][p1:p2].mean(), diff_700[-1][p1:p2].mean(),
               diff_800[-1][p1:p2].mean(), diff_900[-1][p1:p2].mean(), diff_1000[-1][p1:p2].mean()]

Na_80_diff_ls=[diff_600[-3][p1:p2].mean(),diff_700[-2][p1:p2].mean(),diff_800[-1][p1:p2].mean()]
Na_50_diff_ls=[diff_600[2][p1:p2].mean(),diff_700[2][p1:p2].mean(),diff_800[2][p1:p2].mean()]
Na_30_diff_ls=[diff_600[1][p1:p2].mean(),diff_700[1][p1:p2].mean(),diff_800[1][p1:p2].mean()]

Na_10_diff_ls=[diff_400[0][p1:p2].mean(), diff_500[0][p1:p2].mean(), diff_600[0][p1:p2].mean(), diff_700[0][p1:p2].mean(),
               diff_800[0][p1:p2].mean(), diff_900[0][p1:p2].mean(), diff_1000[0][p1:p2].mean()]


ax[0,1].plot(1/temp_1*1000, Na_10_diff_ls, marker='o', markersize=3, label=f'x = {round(Na_list[-6]/2048,3)}', color=colors_ls[0])
ax[0,1].plot(1/temp_2*1000, Na_30_diff_ls, marker='o', markersize=3, label=f'x = {round(Na_list[-5]/2048,3)}', color=colors_ls[1])
ax[0,1].plot(1/temp_2*1000, Na_50_diff_ls, marker='o', markersize=3, label=f'x = {round(Na_list[-4]/2048,3)}', color=colors_ls[2])
ax[0,1].plot(1/temp_2*1000, Na_80_diff_ls, marker='o', markersize=3, label=f'x = {round(Na_list[-3]/2048,3)}', color=colors_ls[3])
ax[0,1].plot(1/temp_1[1:]*1000, Na_90_diff_ls[1:], marker='*', markersize=3, label=f'x = {round(Na_list[-2]/2048,3)}', color=colors_ls[4])


ax[0,1].legend(frameon=False)
#ax2.set_title(f'delta t = {delta_t_ls[i]*0.5} ps')
#ax1.set_xscale('log')
ax[0,1].set_yscale('log')
ax[0,1].set_xlabel('1000/T (1/K)')
ax[0,1].set_ylabel(r"D(m$^2$/s)")

h = 2.302585
def exp_func(x, a, b):
    return a + b * x

# Prepare your data
x_data = 1/temp_1  # Reciprocal of temperature
y_data = np.log(Na_10_diff_ls)  # Diffusivity values
popt, pcov = curve_fit(exp_func, x_data, y_data, p0=(1, 1))
a_fit, b_fit = popt
perr = np.sqrt(np.diag(pcov))
a_err, b_err = perr
residuals = y_data - exp_func(x_data, a_fit, b_fit)
ss_res = np.sum(residuals**2)
ss_tot = np.sum((y_data - np.mean(y_data))**2)
r_squared = 1 - (ss_res / ss_tot)
print(f"R-squared: {r_squared:.5f}")

x_fit = np.linspace(x_data.min(), x_data.max(), 100)
y_fit = exp_func(x_fit, a_fit, b_fit)
D_300K = exp_func(1/300, a_fit, b_fit)
print(f'(D(300K)) = {np.exp(D_300K)*10000} cm^2/s')

ax[1,0].plot([1000*x_fit[0],1000/300], np.array([y_fit[0],D_300K])/h, 'r--', linewidth=0.5)
ax[1,0].scatter(1000*x_data, y_data/h, color=colors_ls[0],label=f'x = {round(Na_list[-6]/2048,3)}')
ax[1,0].scatter(1000/300, D_300K/h, color='r')

x_data = 1/temp_1[1:]  # Reciprocal of temperature
y_data = np.log(Na_90_diff_ls[1:])  # Diffusivity values
popt, pcov = curve_fit(exp_func, x_data, y_data, p0=(1, 1))
a_fit, b_fit = popt
perr = np.sqrt(np.diag(pcov))
a_err, b_err = perr
residuals = y_data - exp_func(x_data, a_fit, b_fit)
ss_res = np.sum(residuals**2)
ss_tot = np.sum((y_data - np.mean(y_data))**2)
r_squared = 1 - (ss_res / ss_tot)
print(f"R-squared: {r_squared:.5f}")

x_fit = np.linspace(x_data.min(), x_data.max(), 100)
y_fit = exp_func(x_fit, a_fit, b_fit)
D_300K = exp_func(1/300, a_fit, b_fit)
print(f'(D(300K)) = {np.exp(D_300K)*10000} cm^2/s')

ax[1,0].plot([1000*x_fit[0],1000/300], np.array([y_fit[0],D_300K])/h, 'r--', linewidth=0.5)
ax[1,0].scatter(1000*x_data, y_data/h, color=colors_ls[4], label=f'x = {round(Na_list[-2]/2048,3)}')
ax[1,0].scatter(1000/300, D_300K/h, color='r')
#ax[1,0].text(1000/300, D_300K/h, color='r')

ax[1,0].set_xlabel('1000/T (1/K)')
ax[1,0].set_ylabel(r"log(D(m$^2$/s))")
ax[1,0].legend(frameon=False)

x = np.array([400,1144,3600])/2048 
y = np.array([-b_fit_10, -b_fit_30, -b_fit_90])*8.617e-5
yerr = np.array([b_err_10, b_err_30, b_err_90])*8.617e-5
ax[1,1].errorbar(x, y, yerr=yerr, fmt='o', capsize=5)
ax[1,1].plot(x,y)
ax[1,1].set_xlabel(r"$x$ in Na$_x$Mn[Fe(CN)$_6$]")
ax[1,1].set_ylabel('Activation Energy (eV)')

ax[1,1].set_aspect('auto')
ax[1,1].set_box_aspect(1)
ax[0,1].set_aspect('auto')
ax[0,1].set_box_aspect(1)
ax[0,0].set_aspect('auto')
ax[0,0].set_box_aspect(1)
ax[1,0].set_aspect('auto')
ax[1,0].set_box_aspect(1)

#plt.savefig('./production_run/free_E/test/diffusion.svg', bbox_inches='tight')

In [ ]:
y_data[-1]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

def fit_diff(temp_ls, Na_diff_ls, SoC):

    # Define the exponential function
    def exp_func(x, a, b):
        return a + b * x
    
    # Prepare your data
    x_data = 1/temp_ls  # Reciprocal of temperature
    y_data = np.log(Na_diff_ls)  # Diffusivity values
    
    # Perform curve fitting
    popt, pcov = curve_fit(exp_func, x_data, y_data, p0=(1, 1))
    a_fit, b_fit = popt
    
    # Calculate standard errors
    perr = np.sqrt(np.diag(pcov))
    a_err, b_err = perr
    
    # Print fitted parameters with their uncertainties
    print(f"Fitted Parameters:")
    print(f"a = {a_fit} ± {a_err}")
    print(f"b = {b_fit:.5f} ± {b_err:.5f}")
    
    # Calculate R-squared to assess goodness of fit
    residuals = y_data - exp_func(x_data, a_fit, b_fit)
    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((y_data - np.mean(y_data))**2)
    r_squared = 1 - (ss_res / ss_tot)
    print(f"R-squared: {r_squared:.5f}")
    
    # Plotting
    plt.figure(figsize=(6, 6))
    plt.scatter(1000*x_data, y_data, label='Experimental Data')
    x_fit = np.linspace(x_data.min(), x_data.max(), 100)
    y_fit = exp_func(x_fit, a_fit, b_fit)
    D_300K = exp_func(1/300, a_fit, b_fit)
    print(f'(D(300K)) = {np.exp(D_300K)*10000} cm^2/s')
    plt.plot(1000*x_fit, y_fit, 'r-', label='Fitted Curve')
    plt.scatter(1000/300, D_300K)
    plt.xlabel('1000/T')
    plt.ylabel('log(Diffusivity)')
    plt.title(f'Diffusivity Exponential Fit at SoC={SoC}')
    plt.legend()
    plt.savefig(f'./production_run/free_E/test/diffusion_fit_{SoC}'+'.png', bbox_inches='tight')
    #plt.xscale('log')
    #plt.yscale('log')
    plt.grid(True)
    plt.show()
    return a_fit, b_fit, b_err, np.exp(D_300K)

In [ ]:
a_fit_10, b_fit_10, b_err_10, D_300_10 = fit_diff(temp_1, Na_10_diff_ls, 0.1)
a_fit_90, b_fit_90, b_err_90, D_300_90 = fit_diff(np.array([500, 600, 700, 800, 900, 1000]), 
                                        [diff_500[-2][p1:p2].mean(), diff_600[-2][p1:p2].mean(), diff_700[-1][p1:p2].mean(), diff_800[-1][p1:p2].mean(), diff_900[-1][p1:p2].mean(), diff_1000[-1][p1:p2].mean()], 0.9)

In [ ]:
Na_50_diff_ls

In [ ]:
temp_2 = np.array([600, 700, 800])
a_fit_30, b_fit_30, b_err_30, D_300_30 = fit_diff(temp_2, Na_30_diff_ls[1:], 0.3)
a_fit_50, b_fit_50, b_err_50, D_300_50 = fit_diff(temp_2, Na_50_diff_ls[1:], 0.5)
a_fit_80, b_fit_80, b_err_80, D_300_80 = fit_diff(temp_2, Na_80_diff_ls[1:], 0.8)

In [ ]:
-b_fit_10*8.617e-5, -b_fit_90*8.617e-5

In [ ]:
-b_fit_30*8.617e-5, -b_fit_50*8.617e-5, -b_fit_80*8.617e-5

In [ ]:
plt.figure()
x = [0.1,0.3,0.9]
y= np.log10([D_300_10, D_300_30, D_300_90])
plt.plot(x,y, marker='o')
plt.xlabel('SoC')
plt.ylabel('log(Diffusivity) at 300K')
plt.tight_layout()

In [ ]:
D_300_10, D_300_30, D_300_90

In [ ]:
yerr

In [ ]:
y

In [ ]:
plt.figure()
x = [0.1,0.3,0.9]
y = np.array([-b_fit_10, -b_fit_30, -b_fit_90])*8.617e-5
yerr = np.array([b_err_10, b_err_30, b_err_90])*8.617e-5
plt.errorbar(x, y, yerr=yerr, fmt='o', capsize=5)
plt.plot(x,y)
plt.xlabel('SoC')
plt.ylabel('Activation Energy (eV)')
plt.grid()
plt.tight_layout()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Define the exponential function
def exp_func(x, a, b):
    return a * np.exp(b * x)

# Prepare your data
x_data = 1/temp_1  # Reciprocal of temperature
y_data = Na_10_diff_ls  # Diffusivity values

# Perform curve fitting
popt, pcov = curve_fit(exp_func, x_data, y_data, p0=(1, 1))
a_fit, b_fit = popt

# Calculate standard errors
perr = np.sqrt(np.diag(pcov))
a_err, b_err = perr

# Print fitted parameters with their uncertainties
print(f"Fitted Parameters:")
print(f"a = {a_fit:.5f} ± {a_err:.5f}")
print(f"b = {b_fit:.5f} ± {b_err:.5f}")

# Calculate R-squared to assess goodness of fit
residuals = y_data - exp_func(x_data, a_fit, b_fit)
ss_res = np.sum(residuals**2)
ss_tot = np.sum((y_data - np.mean(y_data))**2)
r_squared = 1 - (ss_res / ss_tot)
print(f"R-squared: {r_squared:.5f}")

# Plotting
plt.figure(figsize=(10, 6))
plt.scatter(x_data, y_data, label='Experimental Data')
x_fit = np.linspace(x_data.min(), x_data.max(), 100)
y_fit = exp_func(x_fit, a_fit, b_fit)
plt.plot(x_fit, y_fit, 'r-', label='Fitted Curve')
plt.xlabel('1/T')
plt.ylabel('Diffusivity')
plt.title('Diffusivity vs 1/T - Exponential Fit')
plt.legend()
#plt.xscale('log')
#plt.yscale('log')
plt.grid(True)
plt.show()

In [ ]:
-b_fit*8.617e-5


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Define the exponential function
def exp_func(x, a, b):
    return a * np.exp(b * x)

# Prepare your data
x_data = 1/temp[1:]  # Reciprocal of temperature
y_data = diffusivity_3[:-1]  # Diffusivity values

# Perform curve fitting
popt, pcov = curve_fit(exp_func, x_data, y_data, p0=(1, 1))
a_fit, b_fit = popt

# Calculate standard errors
perr = np.sqrt(np.diag(pcov))
a_err, b_err = perr

# Print fitted parameters with their uncertainties
print(f"Fitted Parameters:")
print(f"a = {a_fit:.5f} ± {a_err:.5f}")
print(f"b = {b_fit:.5f} ± {b_err:.5f}")

# Calculate R-squared to assess goodness of fit
residuals = y_data - exp_func(x_data, a_fit, b_fit)
ss_res = np.sum(residuals**2)
ss_tot = np.sum((y_data - np.mean(y_data))**2)
r_squared = 1 - (ss_res / ss_tot)
print(f"R-squared: {r_squared:.5f}")

# Plotting
plt.figure(figsize=(10, 6))
plt.scatter(x_data, y_data, label='Experimental Data')
x_fit = np.linspace(x_data.min(), x_data.max(), 100)
y_fit = exp_func(x_fit, a_fit, b_fit)
plt.plot(x_fit, y_fit, 'r-', label='Fitted Curve')
plt.xlabel('1/T')
plt.ylabel('Diffusivity')
plt.title('Diffusivity vs 1/T - Exponential Fit')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.grid(True)
plt.show()

In [ ]:
Ea = -b_fit*8.617e-5
Ea

In [ ]:
y_fit = exp_func(x_data, *popt)

plt.scatter(x_data, y_data, label="Data", color="blue")
plt.plot(x_data, y_fit, label="Exponential Fit", color="red")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.title("Exponential Fit")
plt.show()

In [ ]:
delta_t_ls = np.array([1, 2, 5, 10, 20, 50, 100, 200, 500, 1000,2000,5000]) #1, 2.5, 10, 25, 100, 250 ps

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18,8))
plt.subplots_adjust(wspace=3)

temp = 700

diffusivity_list = []
num_Na_list = []
for index in [2, 4, 6]:
    file = f'./test_training_set/diffusion/{temp}K_in_diffuse_{index}.log'
    log_data = parse_lammps_log_diff(file)
    MSD_Na = np.array(log_data['MSD(Na)'])
    timestep = np.array(log_data['Step'])
    num_Na = int(log_data['v_nNa'][0])
    num_Na_list.append(num_Na)
    t_intival = []
    MSD_intival = []
    for i in range(delta_t_ls.shape[0]):
        #print(i, delta_t_ls[i]*0.5)
        t_intival.append(np.arange(timestep[0], timestep[-1], int(delta_t_ls[i])*500, dtype='int'))
        MSD_intival.append(np.linspace(MSD_Na[0], MSD_Na[-1], t_intival[i].shape[0]))

    delta_t_all = []
    delta_MSD_all = []
    for i in range(delta_t_ls.shape[0]):
        delta_t = []
        delta_MSD = []
        for j in range(t_intival[i].shape[0]-1):
            delta_t.append(t_intival[i][j+1] - t_intival[i][j])
            delta_MSD.append(MSD_intival[i][j+1] - MSD_intival[i][j])
        delta_t_all.append(delta_t)
        delta_MSD_all.append(delta_MSD)

    final_MSD = []
    final_t = []
    diffusivity = []
    for i in range(delta_t_ls.shape[0]):
        final_t.append(sum(delta_t_all[i])/len(delta_t_all[i]))
        final_MSD.append(sum(delta_MSD_all[i])/len(delta_MSD_all[i]))
        
    final_MSD = np.array(final_MSD)
    final_t = np.array(final_t)
    diffusivity_list.append(final_MSD/final_t/6*1e-5)
    
    ax1.plot(final_t, final_MSD, marker='*', markersize=3, label=f'{temp}K SoC = {round(num_Na/4096,3)}')
    
ax1.grid(True, linestyle='--', alpha=0.7)
ax1.legend()
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel('delta t (fs)')
ax1.set_ylabel('MSD (A^2)')

for i in range(len(diffusivity_list)):
    plt.plot(final_t, diffusivity_list[i], marker='*', markersize=3, label=f'{temp}K SoC = {round(num_Na_list[i]/4096,3)}')

ax2.grid(True, linestyle='--', alpha=0.7)
ax2.legend()
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlabel('delta t (fs)')
ax2.set_ylabel('Diffusivity (m^2/s)')

plt.tight_layout()

In [ ]:
final_t

In [ ]:
for i in range(len(diffusivity_list)):
    plt.plot(diffusivity_list[i], )

In [ ]:
delta_t_ls = np.array([1, 2, 5, 10, 20, 50, 100, 200, 500, 1000,2000,5000]) #1, 2.5, 10, 25, 100, 250 ps
fig, ax1 = plt.subplots(figsize=(8, 6))
temp = 600
for index in [2, 4, 6, 8]:
    file = f'./test_training_set/diffusion/{temp}_diffuse_{index}.log'
    log_data = parse_lammps_log_diff(file)
    MSD_Na = np.array(log_data['MSD(Na)'])
    timestep = np.array(log_data['Step'])
    num_Na = int(log_data['v_nNa'][0])
    t_intival = []
    MSD_intival = []
    for i in range(delta_t_ls.shape[0]):
        #print(i, delta_t_ls[i]*0.5)
        t_intival.append(np.arange(timestep[0], timestep[-1], int(delta_t_ls[i])*500, dtype='int'))
        MSD_intival.append(np.linspace(MSD_Na[0], MSD_Na[-1], t_intival[i].shape[0]))

    delta_t_all = []
    delta_MSD_all = []
    for i in range(delta_t_ls.shape[0]):
        delta_t = []
        delta_MSD = []
        for j in range(t_intival[i].shape[0]-1):
            delta_t.append(t_intival[i][j+1] - t_intival[i][j])
            delta_MSD.append(MSD_intival[i][j+1] - MSD_intival[i][j])
        delta_t_all.append(delta_t)
        delta_MSD_all.append(delta_MSD)

    final_MSD = []
    final_t = []
    for i in range(delta_t_ls.shape[0]):
        final_t.append(sum(delta_t_all[i])/len(delta_t_all[i]))
        final_MSD.append(sum(delta_MSD_all[i])/len(delta_MSD_all[i]))
        
    ax1.plot(final_t, final_MSD, marker='*', markersize=3, label=f'{temp}K SoC = {round(num_Na/4096,3)}')
    
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
ax1.legend()
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel('delta t (fs)')
ax1.set_ylabel('MSD (A^2)')

In [ ]:
delta_t_ls = np.array([1, 2, 5, 10, 20, 50, 100, 200, 500, 1000,2000,5000]) #1, 2.5, 10, 25, 100, 250 ps
fig, ax1 = plt.subplots(figsize=(8, 6))
temp = 700
for index in [2, 4, 6]:
    file = f'./test_training_set/diffusion/{temp}_diffuse_{index}.log'
    log_data = parse_lammps_log_diff(file)
    MSD_Na = np.array(log_data['MSD(Na)'])
    timestep = np.array(log_data['Step'])
    num_Na = int(log_data['v_nNa'][0])
    t_intival = []
    MSD_intival = []
    for i in range(delta_t_ls.shape[0]):
        #print(i, delta_t_ls[i]*0.5)
        t_intival.append(np.arange(timestep[0], timestep[-1], int(delta_t_ls[i])*500, dtype='int'))
        MSD_intival.append(np.linspace(MSD_Na[0], MSD_Na[-1], t_intival[i].shape[0]))

    delta_t_all = []
    delta_MSD_all = []
    for i in range(delta_t_ls.shape[0]):
        delta_t = []
        delta_MSD = []
        for j in range(t_intival[i].shape[0]-1):
            delta_t.append(t_intival[i][j+1] - t_intival[i][j])
            delta_MSD.append(MSD_intival[i][j+1] - MSD_intival[i][j])
        delta_t_all.append(delta_t)
        delta_MSD_all.append(delta_MSD)

    final_MSD = []
    final_t = []
    for i in range(delta_t_ls.shape[0]):
        final_t.append(sum(delta_t_all[i])/len(delta_t_all[i]))
        final_MSD.append(sum(delta_MSD_all[i])/len(delta_MSD_all[i]))
        
    ax1.plot(final_t, final_MSD, marker='*', markersize=3, label=f'{temp}K SoC = {round(num_Na/4096,3)}')
    
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
ax1.legend()
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel('delta t (fs)')
ax1.set_ylabel('MSD (A^2)')

In [ ]:
delta_t_ls = np.array([1, 2, 5, 10, 20, 50, 100, 200, 500, 1000,2000]) #1, 2.5, 10, 25, 100, 250 ps
fig, ax1 = plt.subplots(figsize=(8, 6))
temp = 800
for index in [2, 4, 6]:
    file = f'./test_training_set/diffusion/{temp}_diffuse_{index}.log'
    log_data = parse_lammps_log_diff(file)
    MSD_Na = np.array(log_data['MSD(Na)'])
    timestep = np.array(log_data['Step'])
    num_Na = int(log_data['v_nNa'][0])
    t_intival = []
    MSD_intival = []
    for i in range(delta_t_ls.shape[0]):
        #print(i, delta_t_ls[i]*0.5)
        t_intival.append(np.arange(timestep[0], timestep[-1], int(delta_t_ls[i])*500, dtype='int'))
        MSD_intival.append(np.linspace(MSD_Na[0], MSD_Na[-1], t_intival[i].shape[0]))

    delta_t_all = []
    delta_MSD_all = []
    for i in range(delta_t_ls.shape[0]):
        delta_t = []
        delta_MSD = []
        for j in range(t_intival[i].shape[0]-1):
            delta_t.append(t_intival[i][j+1] - t_intival[i][j])
            delta_MSD.append(MSD_intival[i][j+1] - MSD_intival[i][j])
        delta_t_all.append(delta_t)
        delta_MSD_all.append(delta_MSD)

    final_MSD = []
    final_t = []
    for i in range(delta_t_ls.shape[0]):
        final_t.append(sum(delta_t_all[i])/len(delta_t_all[i]))
        final_MSD.append(sum(delta_MSD_all[i])/len(delta_MSD_all[i]))
        
    ax1.plot(final_t, final_MSD, marker='*', markersize=3, label=f'{temp}K SoC = {round(num_Na/4096,3)}')
    
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
ax1.legend()
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel('delta t (fs)')
ax1.set_ylabel('MSD (A^2)')

In [ ]:
final_t

In [ ]:
delta_t_ls = np.array([1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]) #1, 2.5, 10, 25, 100, 250 ps
i=6
file = f'./test_training_set/diffusion/600_diffuse_{i}.log'
log_data = parse_lammps_log_diff(file)
MSD_Na = np.array(log_data['MSD(Na)'])
timestep = np.array(log_data['Step'])

In [ ]:
t_intival = []
MSD_intival = []
for i in range(delta_t_ls.shape[0]):
    print(i, delta_t_ls[i]*0.5)
    t_intival.append(np.arange(timestep[0], timestep[-1], int(delta_t_ls[i])*500, dtype='int'))
    MSD_intival.append(np.linspace(MSD_Na[0], MSD_Na[-1], t_intival[i].shape[0]))


In [ ]:
i=0
delta_t_ls.shape[0], t_intival[i].shape[0]-1

In [ ]:
delta_t_all = []
delta_MSD_all = []
for i in range(delta_t_ls.shape[0]):
    delta_t = []
    delta_MSD = []
    for j in range(t_intival[i].shape[0]-1):
        delta_t.append(t_intival[i][j+1] - t_intival[i][j])
        delta_MSD.append(MSD_intival[i][j+1] - MSD_intival[i][j])
    delta_t_all.append(delta_t)
    delta_MSD_all.append(delta_MSD)

In [ ]:
final_MSD = []
final_t = []
for i in range(delta_t_ls.shape[0]):
    final_t.append(sum(delta_t_all[i])/len(delta_t_all[i]))
    final_MSD.append(sum(delta_MSD_all[i])/len(delta_MSD_all[i]))

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 6))

ax1.plot(final_t, final_MSD,  marker='*', markersize=5)
ax1.set_xlabel('delta t (ps)')
ax1.set_ylabel('MSD (A^2)')
ax1.set_xscale('log')
ax1.set_yscale('log')

# Phase diagrame

## 300K

In [ ]:
for i in range(11):
    file = f'./test_training_set/phase_diagram/300_phase_{i+1}.log'
    plot_log_no_V(file)

## 400K

In [ ]:
for i in range(11):
    file = f'./test_training_set/phase_diagram/400_phase_{i+1}.log'
    plot_log(file)

## 500K

In [ ]:
for i in range(11):
    file = f'./test_training_set/phase_diagram/500_phase_{i+1}.log'
    plot_log(file)

## 600K

In [ ]:
for i in range(11):
    file = f'./test_training_set/phase_diagram/600_phase_{i+1}.log'
    plot_log(file)

## 700K

In [ ]:
for i in range(19):
    file = f'./test_training_set/phase_diagram/700_phase_{i+1}.log'
    plot_log(file)

## 800K

In [ ]:
for i in range(19):
    file = f'./test_training_set/phase_diagram/800_phase_{i+1}.log'
    plot_log(file)

## 900K

In [ ]:
for i in range(19):
    file = f'./test_training_set/phase_diagram/900_phase_{i+1}.log'
    plot_log(file)

## 1000K

In [ ]:
for i in range(19):
    file = f'./test_training_set/phase_diagram/1000_phase_{i+1}.log'
    plot_log(file)

## 1100K

In [ ]:
for i in range(8,19):
    file = f'./test_training_set/phase_diagram/1100_phase_{i+1}.log'
    plot_log(file)

## 1200K

In [ ]:
for i in range(8,19):
    file = f'./test_training_set/phase_diagram/1200_phase_{i+1}.log'
    plot_log(file)

## 1300K

In [ ]:
for i in range(8,19):
    file = f'./test_training_set/phase_diagram/1300_phase_{i+1}.log'
    plot_log(file)

## Na profiles

In [ ]:
def plot_Na_profile(temp, n_1, n_2, mu_1, mu_2):
    fig, ax1 = plt.subplots(figsize=(10, 6))
    temp=temp
    mu_1, mu_2, step = mu_1, mu_2, 0.05
    for i in range(n_1, n_2):
        file = f'./test_training_set/phase_diagram/{temp}_phase_{i+1}.log'
        log_data = parse_lammps_log(file)
     
        E_Na = -.14023100E+01
        my_list = log_data['TotEng'][20:40]
        E_i = sum(my_list) / len(my_list)
        my_list = log_data['TotEng'][-20:]
        E_f = sum(my_list) / len(my_list)
        my_list = log_data['v_nNa'][20:40]
        n_Na_i = sum(my_list) / len(my_list)
        my_list = log_data['v_nNa'][-20:]
        n_Na_f = sum(my_list) / len(my_list)   
        n_Na = n_Na_i-n_Na_f
        V_ave = -(E_i-E_f-n_Na*E_Na)/n_Na
        
        ax1.plot(log_data['Step'], log_data['v_nNa'], color=cm.jet((i-n_1)/n_2), label=f'mu_{i+1} = {round(np.arange(mu_1, mu_2+step, step)[i],2)}  V = {round(V_ave,2)}')#, marker='*')
    
    
    #ax1.hlines(512,0,3e6, linestyle='--', color='black')
    #ax1.hlines(0,0,3e6, linestyle='--', color='black')
    
    ax1.set_xlabel('Timestep (0.4 fs/step)')
    ax1.set_ylabel('Number of Na+')
    #ax1.set_xlim(0,3e6)
    plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')
    #ax1.tick_params(axis='y', labelcolor=color)
    #ax1.set_ylim(-5,515)
    ax1.legend()
    plt.grid()

In [ ]:
plot_Na_profile(300)

In [ ]:
plot_Na_profile(400)

In [ ]:
plot_Na_profile(500)

In [ ]:
plot_Na_profile(600)

In [ ]:
plot_Na_profile(700, 0, 19, -2.2, -1.3)

In [ ]:
plot_Na_profile(800, 0, 19, -2.2, -1.3)

In [ ]:
plot_Na_profile(900, 0, 19, -2.2, -1.3)

In [ ]:
plot_Na_profile(1000, 0, 19, -2.2, -1.3)

In [ ]:
plot_Na_profile(1100, 0, 19, -2.2, -1.3)

In [ ]:
plot_Na_profile(1200, 8, 19, -2.2, -1.3)

In [ ]:
plot_Na_profile(1300, 8, 19, -2.2, -1.3)

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
temp=300
mu_1, mu_2, step = -2.2, -1.7, 0.05
for i in range(11):
    file = f'./test_training_set/phase_diagram/{temp}_phase_{i+1}.log'
    log_data = parse_lammps_log(file)
 
    E_Na = -.14023100E+01
    my_list = log_data['TotEng'][20:40]
    E_i = sum(my_list) / len(my_list)
    my_list = log_data['TotEng'][-20:]
    E_f = sum(my_list) / len(my_list)
    my_list = log_data['v_nNa'][20:40]
    n_Na_i = sum(my_list) / len(my_list)
    my_list = log_data['v_nNa'][-20:]
    n_Na_f = sum(my_list) / len(my_list)   
    n_Na = n_Na_i-n_Na_f
    V_ave = -(E_i-E_f-n_Na*E_Na)/n_Na
    
    ax1.plot(log_data['Step'], log_data['v_nNa'], color=cm.jet(i/11), label=f'mu_{i+1} = {round(np.arange(mu_1, mu_2+step, step)[i],2)}  V = {round(V_ave,2)}')#, marker='*')


#ax1.hlines(512,0,3e6, linestyle='--', color='black')
#ax1.hlines(0,0,3e6, linestyle='--', color='black')

ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')
#ax1.set_xlim(0,3e6)
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')
#ax1.tick_params(axis='y', labelcolor=color)
ax1.set_ylim(-5,515)
ax1.legend()
plt.grid()

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
temp=400
mu_1, mu_2, step = -2.2, -1.7, 0.05
for i in range(11):
    file = f'./test_training_set/phase_diagram/{temp}_phase_{i+1}.log'
    log_data = parse_lammps_log(file)
 
    E_Na = -.14023100E+01
    my_list = log_data['TotEng'][20:40]
    E_i = sum(my_list) / len(my_list)
    my_list = log_data['TotEng'][-20:]
    E_f = sum(my_list) / len(my_list)
    my_list = log_data['v_nNa'][20:40]
    n_Na_i = sum(my_list) / len(my_list)
    my_list = log_data['v_nNa'][-20:]
    n_Na_f = sum(my_list) / len(my_list)   
    n_Na = n_Na_i-n_Na_f
    V_ave = -(E_i-E_f-n_Na*E_Na)/n_Na
    
    ax1.plot(log_data['Step'], log_data['v_nNa'], color=cm.jet(i/11), label=f'mu_{i+1} = {round(np.arange(mu_1, mu_2+step, step)[i],2)}  V = {round(V_ave,2)}')#, marker='*')


#ax1.hlines(512,0,3e6, linestyle='--', color='black')
#ax1.hlines(0,0,3e6, linestyle='--', color='black')

ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')
#ax1.set_xlim(0,3e6)
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')
#ax1.tick_params(axis='y', labelcolor=color)
ax1.set_ylim(-5,515)
ax1.legend()
plt.grid()

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
temp=500
mu_1, mu_2, step = -2.2, -1.7, 0.05
for i in range(11):
    file = f'./test_training_set/phase_diagram/{temp}_phase_{i+1}.log'
    log_data = parse_lammps_log(file)
 
    E_Na = -.14023100E+01
    my_list = log_data['TotEng'][20:40]
    E_i = sum(my_list) / len(my_list)
    my_list = log_data['TotEng'][-20:]
    E_f = sum(my_list) / len(my_list)
    my_list = log_data['v_nNa'][20:40]
    n_Na_i = sum(my_list) / len(my_list)
    my_list = log_data['v_nNa'][-20:]
    n_Na_f = sum(my_list) / len(my_list)   
    n_Na = n_Na_i-n_Na_f
    V_ave = -(E_i-E_f-n_Na*E_Na)/n_Na
    
    ax1.plot(log_data['Step'], log_data['v_nNa'], color=cm.jet(i/11), label=f'mu_{i+1} = {round(np.arange(mu_1, mu_2+step, step)[i],2)}  V = {round(V_ave,2)}')#, marker='*')


ax1.hlines(512,0,3e6, linestyle='--', color='black')
ax1.hlines(0,0,3e6, linestyle='--', color='black')

ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')
ax1.set_xlim(0,3e6)
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')
#ax1.tick_params(axis='y', labelcolor=color)
#ax1.set_xlim(0,1e6)
ax1.legend()
plt.grid()

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
temp=600
mu_1, mu_2, step = -2.2, -1.7, 0.05
for i in range(11):
    file = f'./test_training_set/phase_diagram/{temp}_phase_{i+1}.log'
    log_data = parse_lammps_log(file)
 
    E_Na = -.14023100E+01
    my_list = log_data['TotEng'][20:40]
    E_i = sum(my_list) / len(my_list)
    my_list = log_data['TotEng'][-20:]
    E_f = sum(my_list) / len(my_list)
    my_list = log_data['v_nNa'][20:40]
    n_Na_i = sum(my_list) / len(my_list)
    my_list = log_data['v_nNa'][-20:]
    n_Na_f = sum(my_list) / len(my_list)   
    n_Na = n_Na_i-n_Na_f
    V_ave = -(E_i-E_f-n_Na*E_Na)/n_Na
    
    ax1.plot(log_data['Step'], log_data['v_nNa'], color=cm.jet(i/11), label=f'mu_{i+1} = {round(np.arange(mu_1, mu_2+step, step)[i],2)}  V = {round(V_ave,2)}')#, marker='*')


#ax1.hlines(512,0,3e6, linestyle='--', color='black')
#ax1.hlines(0,0,3e6, linestyle='--', color='black')

ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')
#ax1.set_xlim(0,3e6)
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')
#ax1.tick_params(axis='y', labelcolor=color)
#ax1.set_xlim(0,1e6)
ax1.legend()
plt.grid()

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
temp=400
mu_1, mu_2, step = -2.2, -1.7, 0.05
for i in range(11):
    file = f'./test_training_set/phase_diagram/{temp}_phase_{i+1}.log'
    log_data_nbh = parse_lammps_log(file)
    ax1.plot(log_data_nbh['Step'], log_data_nbh['v_nNa'], color=cm.jet(i/11), label=f'mu = {round(np.arange(mu_1, mu_2+step, step)[i],2)}')#, marker='*')

#ax1.hlines(512,0,3e6, linestyle='--', color='black')
#ax1.hlines(0,0,3e6, linestyle='--', color='black')

ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')
#ax1.set_xlim(0,3e6)
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')
#ax1.tick_params(axis='y', labelcolor=color)
#ax1.set_xlim(0,1e6)
ax1.legend()
plt.grid()

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
temp=500
mu_1, mu_2, step = -2.2, -1.7, 0.05
for i in range(11):
    file = f'./test_training_set/phase_diagram/{temp}_phase_{i+1}.log'
    log_data_nbh = parse_lammps_log(file)
    ax1.plot(log_data_nbh['Step'], log_data_nbh['v_nNa'], color=cm.jet(i/11), label=f'mu = {round(np.arange(mu_1, mu_2+step, step)[i],2)}')#, marker='*')

#ax1.hlines(512,0,3e6, linestyle='--', color='black')
#ax1.hlines(0,0,3e6, linestyle='--', color='black')

ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')
#ax1.set_xlim(0,3e6)
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')
#ax1.tick_params(axis='y', labelcolor=color)
#ax1.set_xlim(0,1e6)
ax1.legend()
plt.grid()

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
temp=600
mu_1, mu_2, step = -2.2, -1.7, 0.05
for i in range(7):
    file = f'./test_training_set/phase_diagram/{temp}_phase_{i+1}.log'
    log_data_nbh = parse_lammps_log(file)
    ax1.plot(log_data_nbh['Step'], log_data_nbh['v_nNa'], color=cm.jet(i/11), label=f'mu = {round(np.arange(mu_1, mu_2+step, step)[i],2)}')#, marker='*')

#ax1.hlines(512,0,3e6, linestyle='--', color='black')
#ax1.hlines(0,0,3e6, linestyle='--', color='black')

ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')
#ax1.set_xlim(0,3e6)
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')
#ax1.tick_params(axis='y', labelcolor=color)
#ax1.set_xlim(0,1e6)
ax1.legend()
plt.grid()

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
temp=700
mu_1, mu_2, step = -2.2, -1.7, 0.05
for i in range(11):
    file = f'./test_training_set/phase_diagram/{temp}_phase_{i+1}.log'
    log_data_nbh = parse_lammps_log(file)
    ax1.plot(log_data_nbh['Step'], log_data_nbh['v_nNa'], color=cm.jet(i/11), label=f'mu = {round(np.arange(mu_1, mu_2+step, step)[i],2)}')#, marker='*')

#ax1.hlines(512,0,3e6, linestyle='--', color='black')
#ax1.hlines(0,0,3e6, linestyle='--', color='black')

ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')
#ax1.set_xlim(0,3e6)
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')
#ax1.tick_params(axis='y', labelcolor=color)
#ax1.set_xlim(0,1e6)
ax1.legend()
plt.grid()

In [ ]:
np.arange(mu_2-2*step, mu_2, step)

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
temp=300
mu_1, mu_2, step = -2, -1, 0.1
for i in range(11,22):
    file = f'./test_training_set/phase_diagram/{temp}_phase_{i+1}.log'
    log_data_nbh = parse_lammps_log(file)
    ax1.plot(log_data_nbh['Step'], log_data_nbh['v_nNa'], color=cm.jet((i-11)/11), label=f'mu = {round(np.arange(mu_1, mu_2+step, step)[i-11],2)}')#, marker='*')

ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')
#ax1.tick_params(axis='y', labelcolor=color)
#ax1.set_xlim(0,1e6)
ax1.legend()
plt.grid()

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
temp=350
mu_1, mu_2, step = -3, -2, 0.1
for i in range(11):
    file = f'./test_training_set/phase_diagram/{temp}_phase_{i+1}.log'
    log_data_nbh = parse_lammps_log(file)
    ax1.plot(log_data_nbh['Step'], log_data_nbh['v_nNa'], color=cm.jet(i/11), label=f'mu = {round(np.arange(mu_1, mu_2+step, step)[i],2)}')#, marker='*')

ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')
#ax1.tick_params(axis='y', labelcolor=color)
#ax1.set_xlim(0,1e6)
ax1.legend()
plt.grid()

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
temp=400
mu_1, mu_2, step = -3, -2, 0.1
for i in range(11):
    file = f'./test_training_set/phase_diagram/{temp}_phase_{i+1}.log'
    log_data_nbh = parse_lammps_log(file)
    ax1.plot(log_data_nbh['Step'], log_data_nbh['v_nNa'], color=cm.jet(i/11), label=f'mu = {round(np.arange(mu_1, mu_2+step, step)[i],2)}')#, marker='*')

ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')
#ax1.tick_params(axis='y', labelcolor=color)
#ax1.set_xlim(0,1e6)
ax1.legend()
plt.grid()

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
temp=450
mu_1, mu_2, step = -3, -2, 0.1
for i in range(11):
    file = f'./test_training_set/phase_diagram/{temp}_phase_{i+1}.log'
    log_data_nbh = parse_lammps_log(file)
    ax1.plot(log_data_nbh['Step'], log_data_nbh['v_nNa'], color=cm.jet(i/11), label=f'mu = {round(np.arange(mu_1, mu_2+step, step)[i],2)}')#, marker='*')

ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')
#ax1.tick_params(axis='y', labelcolor=color)
#ax1.set_xlim(0,1e6)
ax1.legend()
plt.grid()

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
temp=500
mu_1, mu_2, step = -3, -2, 0.1
for i in range(11):
    file = f'./test_training_set/phase_diagram/{temp}_phase_{i+1}.log'
    log_data_nbh = parse_lammps_log(file)
    ax1.plot(log_data_nbh['Step'], log_data_nbh['v_nNa'], color=cm.jet(i/11), label=f'mu = {round(np.arange(mu_1, mu_2+step, step)[i],2)}')#, marker='*')

ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Number of Na+')
plt.title(f'{temp} K  mu = {mu_1} ~ {mu_2}')
#ax1.tick_params(axis='y', labelcolor=color)
#ax1.set_xlim(0,1e6)
ax1.legend()
plt.grid()

In [ ]:
for i in range(len(data)-1):
    delta_r_square_1.append(abs(data['c_msd_Na[4]'].iloc[i+1]-data['c_msd_Na[4]'].iloc[i]))
for i in range(0, len(data)-1, 2):
    delta_r_square_2.append(abs(data['c_msd_Na[4]'].iloc[i+1]-data['c_msd_Na[4]'].iloc[i]))
for i in range(0, len(data)-1, 100):
    delta_r_square_5.append(abs(data['c_msd_Na[4]'].iloc[i+1]-data['c_msd_Na[4]'].iloc[i]))

In [ ]:
for i in range(8,16):
    file = f'./test_training_set/in_diffuse_{i+1}.log'
    plot_log_diff(file)

In [ ]:
num_Na_ls/512

In [ ]:
num_Na_ls = np.array([101.0, 143.0, 194.0, 243.0, 284.0, 410.0, 449.0, 494.0])*8
p1, p2 = 4,8
for i in range(p1, p2):
    filename = f'600K_msd_Na_{i+1}.dat'
    path = './test_training_set/'
    data = pd.read_csv(path + filename, delim_whitespace=True, comment='#', header=None)
    data.columns = ['TimeStep', 'c_msd_Na[1]', 'c_msd_Na[2]', 'c_msd_Na[3]', 'c_msd_Na[4]']
    plt.figure(figsize=(8,5))
    plt.plot(data['TimeStep'], data['c_msd_Na[1]']*num_Na_ls[i-p1], label='MSD(x)')
    plt.plot(data['TimeStep'], data['c_msd_Na[2]']*num_Na_ls[i-p1], label='MSD(y)')
    plt.plot(data['TimeStep'], data['c_msd_Na[3]']*num_Na_ls[i-p1], label='MSD(z)')
    plt.plot(data['TimeStep'], data['c_msd_Na[4]']*num_Na_ls[i-p1], label='TMSD')
    plt.legend()
    plt.xlabel('Timestep (1 fs/step)')
    plt.ylabel('MSD (A^2)')
    plt.title(filename+ f'  {int(num_Na_ls[i-p1])} Na+')


In [ ]:
# Path to your LAMMPS log file
i=4
log_file_path = f'./test_almtp_lv20/gcmc_{i}.log'
#log_file_path = 'gcmc_3.log'
log_data = parse_lammps_log(log_file_path)

#E_Na = -.13023100E+01
#V_ave = -(log_data['TotEng'][20]-log_data['TotEng'][-1]-(log_data['v_nNa'][20]-log_data['v_nNa'][-1])*E_Na)/(log_data['v_nNa'][20]-log_data['v_nNa'][-1])

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.4 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 80)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
#plt.title(log_file_path+f'    Average V = {round(V_ave,3)}')
#plt.xlim(10000,30000)
#ax1.set_ylim(-31500, -31000)
#ax2.set_ylim(54000,59000)


In [ ]:
i=5
file = f'./test_almtp_lv20/gcmc_{i}.log'
plot_log(file)

In [ ]:
for i in range(4):
    file = f'./test_almtp_stress/gcmc_{i+1}.log'
    plot_log(file)

In [ ]:
plot_log('gcmc_3.log')

In [ ]:
plot_log('gcmc_2.log')

In [ ]:
plot_log('gcmc_2.log')

In [ ]:
plot_log('gcmc_2.log')

In [ ]:
# Path to your LAMMPS log file
log_file_path = './pot_save/gcmc_6_497.log'
log_data = parse_lammps_log(log_file_path)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.1 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 60)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
plt.title(log_file_path)
#plt.xlim(10000,30000)
#ax1.set_ylim(-31500, -31000)
#ax2.set_ylim(54000,59000)
E_Na = -.13023100E+01
V_ave = -(log_data['TotEng'][20]-log_data['TotEng'][-1]-(log_data['v_nNa'][20]-log_data['v_nNa'][-1])*E_Na)/(log_data['v_nNa'][20]-log_data['v_nNa'][-1])
V_ave

In [ ]:
log_data['TotEng'][-1],log_data['v_nNa'][-1],log_data['Volume'][-1]

In [ ]:
# Path to your LAMMPS log file
log_file_path = './pot_save/gcmc_3_488.log'
log_data = parse_lammps_log(log_file_path)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.1 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 60)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
plt.title(log_file_path)
#plt.xlim(10000,30000)
#ax1.set_ylim(-31500, -31000)
#ax2.set_ylim(54000,59000)
E_Na = -.13023100E+01
V_ave = -(log_data['TotEng'][20]-log_data['TotEng'][-1]-(log_data['v_nNa'][20]-log_data['v_nNa'][-1])*E_Na)/(log_data['v_nNa'][20]-log_data['v_nNa'][-1])
V_ave

In [ ]:
# Path to your LAMMPS log file
log_file_path = './pot_save/gcmc_2_466.log'
log_data = parse_lammps_log(log_file_path)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.1 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 60)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
plt.title(log_file_path)
#plt.xlim(10000,30000)
#ax1.set_ylim(-31500, -31000)
#ax2.set_ylim(54000,59000)
E_Na = -.13023100E+01
V_ave = -(log_data['TotEng'][20]-log_data['TotEng'][-1]-(log_data['v_nNa'][20]-log_data['v_nNa'][-1])*E_Na)/(log_data['v_nNa'][20]-log_data['v_nNa'][-1])
V_ave

In [ ]:
# Path to your LAMMPS log file
log_file_path = './pot_save/gcmc_4_468.log'
log_data = parse_lammps_log(log_file_path)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.1 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 60)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
plt.title(log_file_path)
#plt.xlim(10000,30000)
#ax1.set_ylim(-31500, -31000)
#ax2.set_ylim(54000,59000)


In [ ]:
# Path to your LAMMPS log file
log_file_path = './pot_save/gcmc_235.log'
log_data = parse_lammps_log(log_file_path)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.1 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 60)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
plt.title(log_file_path)
#plt.xlim(10000,30000)
#ax1.set_ylim(-31500, -31000)
#ax2.set_ylim(54000,59000)


In [ ]:
# Path to your LAMMPS log file
log_file_path = './pot_save/gcmc_257.log'
log_data = parse_lammps_log(log_file_path)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.1 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 60)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
plt.title(log_file_path)
#plt.xlim(10000,30000)
#ax1.set_ylim(-31500, -31000)
#ax2.set_ylim(54000,59000)


In [ ]:
# Path to your LAMMPS log file
log_file_path = './pot_save/gcmc_274.log'
log_data = parse_lammps_log(log_file_path)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.1 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 60)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
plt.title(log_file_path)
#plt.xlim(10000,30000)
#ax1.set_ylim(-31500, -31000)
#ax2.set_ylim(54000,59000)


In [ ]:
# Path to your LAMMPS log file
log_file_path = './pot_save/gcmc_279.log'
log_data = parse_lammps_log(log_file_path)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.1 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 60)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
plt.title(log_file_path)
#plt.xlim(10000,30000)
#ax1.set_ylim(-31500, -31000)
#ax2.set_ylim(54000,59000)


In [ ]:
# Path to your LAMMPS log file
log_file_path = './pot_save/gcmc_288.log'
log_data = parse_lammps_log(log_file_path)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.1 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 60)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
plt.title(log_file_path)
#plt.xlim(10000,30000)
#ax1.set_ylim(-31500, -31000)
#ax2.set_ylim(54000,59000)


In [ ]:
# Path to your LAMMPS log file
log_file_path = './pot_save/gcmc_321.log'
log_data = parse_lammps_log(log_file_path)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.1 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 60)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
plt.title(log_file_path)
#plt.xlim(10000,30000)
#ax1.set_ylim(-31500, -31000)
#ax2.set_ylim(54000,59000)


In [ ]:
# Path to your LAMMPS log file
log_file_path = './pot_save/gcmc_331.log'
log_data = parse_lammps_log(log_file_path)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.1 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 60)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
plt.title(log_file_path)
#plt.xlim(10000,30000)
#ax1.set_ylim(-31500, -31000)
#ax2.set_ylim(54000,59000)


In [ ]:
# Path to your LAMMPS log file
log_file_path = './pot_save/gcmc_332.log'
log_data = parse_lammps_log(log_file_path)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.1 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 60)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
plt.title(log_file_path)
#plt.xlim(10000,30000)
#ax1.set_ylim(-31500, -31000)
#ax2.set_ylim(54000,59000)


In [ ]:
# Path to your LAMMPS log file
log_file_path = './pot_save/gcmc_340.log'
log_data = parse_lammps_log(log_file_path)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.1 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 60)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
plt.title(log_file_path)


E_Na = -.13023100E+01
V_ave = -(log_data['TotEng'][20]-log_data['TotEng'][-1]-(log_data['v_nNa'][20]-log_data['v_nNa'][-1])*E_Na)/(log_data['v_nNa'][20]-log_data['v_nNa'][-1])
V_ave

In [ ]:
# Path to your LAMMPS log file
log_file_path = './pot_save/gcmc_356.log'
log_data = parse_lammps_log(log_file_path)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.1 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 60)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
plt.title(log_file_path)
#plt.xlim(10000,30000)
#ax1.set_ylim(-31500, -31000)
#ax2.set_ylim(54000,59000)

E_Na = -.13023100E+01
V_ave = -(log_data['TotEng'][20]-log_data['TotEng'][-1]-(log_data['v_nNa'][20]-log_data['v_nNa'][-1])*E_Na)/(log_data['v_nNa'][20]-log_data['v_nNa'][-1])
V_ave

In [ ]:
# Path to your LAMMPS log file
log_file_path = './pot_save/gcmc_419.log'
log_data = parse_lammps_log(log_file_path)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot RMS(E/atom) on the first y-axis
color = 'tab:blue'
ax1.set_xlabel('Timestep (0.1 fs/step)')
ax1.set_ylabel('Total E (eV)', color=color)
ax1.plot(log_data['Step'], log_data['TotEng'], color=color, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color)

# Create the second y-axis and plot RMS(f)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Volume (A^3)', color=color)
ax2.plot(log_data['Step'][10:], log_data['Volume'][10:], color=color, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color)

ax3 = ax1.twinx()
color = 'tab:green'
ax3.spines['right'].set_position(('outward', 60)) 
ax3.set_ylabel('Number of Na+', color=color)
ax3.plot(log_data['Step'], log_data['v_nNa'], color=color, marker='*')
ax3.tick_params(axis='y', labelcolor=color)
plt.grid()
plt.title(log_file_path)
#plt.xlim(10000,30000)
#ax1.set_ylim(-31500, -31000)
#ax2.set_ylim(54000,59000)

E_Na = -.13023100E+01
V_ave = -(log_data['TotEng'][20]-log_data['TotEng'][-1]-(log_data['v_nNa'][20]-log_data['v_nNa'][-1])*E_Na)/(log_data['v_nNa'][20]-log_data['v_nNa'][-1])
V_ave



In [ ]:
E_Na = -0.14023
V_ave = -(log_data['TotEng'][20]-log_data['TotEng'][-1]-(log_data['v_nNa'][20]-log_data['v_nNa'][-1])*E_Na)/(log_data['v_nNa'][20]-log_data['v_nNa'][-1])
V_ave

In [ ]:
E_Na = -0.14023
V_ave = -(log_data['TotEng'][20]-log_data['TotEng'][-1]-(log_data['v_nNa'][20]-log_data['v_nNa'][-1])*E_Na)/(log_data['v_nNa'][20]-log_data['v_nNa'][-1])
V_ave

In [ ]:
E_Na = -0.14023
V_ave = -(log_data['TotEng'][20]-log_data['TotEng'][-1]-(log_data['v_nNa'][20]-log_data['v_nNa'][-1])*E_Na)/(log_data['v_nNa'][20]-log_data['v_nNa'][-1])
V_ave

In [ ]:
log_data['v_nNa'][-1]

In [ ]:
x = int((log_data['Step'][-1]-5000)/4000)
a = range(0,x*20,20)

In [ ]:
len(a)

In [ ]:
i=1
log_data['Step'][a[i]-1], log_data['Step'][a[i+1]-1]

In [ ]:
i
log_data['v_nNa'][a[i]] - log_data['v_nNa'][a[i+1]]

In [ ]:
x = int((log_data['Step'][-1]-5000)/4000)
a = range(0,x*20,20)
V = []
E_Na = -.13023100E+01

for i in range(len(a)-1):
    delta_Na = log_data['v_nNa'][a[i]] - log_data['v_nNa'][a[i+1]]
    delta_E =  log_data['TotEng'][a[i]-1] - log_data['TotEng'][a[i+1]-1]
    V.append(-(delta_E-delta_Na*E_Na)/delta_Na)

In [ ]:
V

In [ ]:
fig = plt.figure(figsize=(10, 6))

plt.plot(range(len(V)-1), V[1:])
#plt.xlim(40,300)
plt.ylabel('Voltage')
plt.xlabel('Number of Na (512 Na originally)')
plt.title('')
plt.grid()
#plt.text(50,5, f'Average V = {round(np.mean(V[5:50]),3)}')
#plt.xlim(512,480)
#plt.savefig('desodiation_V.png', dpi=300, bbox_inches='tight')

In [ ]:
np.mean(V[1:])

In [ ]:
log_data['TotEng'][20],log_data['TotEng'][-1],log_data['v_nNa'][20],log_data['v_nNa'][-1]